# Raijin: an MXINT8 attention accelerator for edge LLM inference

**Course:** AI Hardware Architectures, IIT Bombay — final project
**Deliverables in this notebook:** the complete RTL (Verilog-2005), the cocotb testbench, the bit-exact Python golden model the testbench checks against, and the test run with its output. The OpenLane (sky130) layout results are summarised at the end.

## 1. Project definition

Raijin is a hardware accelerator for the **attention layer** of a transformer, the operation

$$O = \mathrm{softmax}(QK^\top)\,V$$

for one attention head with sequence length **N = 16** and head dimension **d = 16**. It extends two of the course's suggested projects, a **matrix multiplier** (the QKᵀ and P·V products) and a **softmax accelerator**, into one complete attention engine, and adds the features a small language model needs to generate text on the chip (KV cache, decode steps, RoPE, eviction).

**Number format.** MXINT8: every 16-element row stores INT8 elements and one shared 8-bit power-of-two exponent (E8M0), as in the OCP Microscaling spec. Exponents turn into shift amounts in hardware, so no per-tensor calibration is needed. The KV cache can also hold MXINT4 or MXFP4 (E2M1) rows, which doubles it to 32 tokens.

**Building blocks** (each in its own Verilog file, each tested on its own):

| Block | File | Function |
| --- | --- | --- |
| Dot-product unit (DPU) | `raijin_dpu.v` | 16 INT8×INT8 MACs and a 2-stage adder tree, one q·k per cycle |
| Requantiser | `raijin_requant.v` | Score → Q7.4 fixed point, shift by e_q + e_k |
| exp2 unit | `raijin_exp2.v` | 2^(s−m): 16-entry LUT (exact) or one/two powers of two (multiplier-free) |
| Softmax lane | `raijin_softmax_lane.v` | Base-2 **online softmax**: running max m, running sum l (14-bit), restoring divider for 1/l |
| Accumulate array | `raijin_acc_array.v` | P·V as an outer product: rescale by 2^−Δ and add p·v (shift-add in multiplier-free modes) |
| Output packer | `raijin_out.v` | Multiply by 1/l, leading-one detect, round back to MXINT8 |
| RoPE rotator | `raijin_rope.v` | Rotary position embedding applied as Q/K rows are written |
| Heavy-hitter tracker | `raijin_hh.v` | Per-token attention statistics and an on-chip argmin: which KV entry to evict |
| Core | `raijin_core.v` | Controller, register map, SRAM ports, the three dataflows |
| SPI / QSPI port | `raijin_spi.v` | Host interface on 4–6 pins |
| Chip top | `raijin_chip.v` | SPI port, idle clock gating, reset synchroniser (the OpenLane top level) |

**Dataflows** (same hardware, chosen by the CTRL register):
1. **Unfused:** S = QKᵀ into a score buffer, softmax per row, then O = P·V.
2. **Fused (FlashAttention-style):** keys stream past P = 4 query rows with online softmax; the N×N score matrix is never stored. 111 cycles per full pass vs 411 unfused. A causal flag makes it a prompt-prefill pass.
3. **Decode:** one KV-cache step of autoregressive generation, with multi-query/speculative variants, a paged cache, attention sinks, hardware key skipping and a *max-first* mode that reads V only for keys whose weight is nonzero.

**Verification approach.** Every RTL block and every pass of the core is compared **bit for bit** with the Python golden model (`model/raijin_model.py` and its vectorised twin `model/raijin_vec.py`), and against FP32 attention by cosine similarity.


## 2. How to run this notebook

Requirements: Python 3.10+, `numpy`, `cocotb` 2.x and **Icarus Verilog** 12 (`iverilog`). On Google Colab or Ubuntu:

```
!apt-get -qq install iverilog
!pip -q install cocotb numpy
```

Every source file below is a `%%writefile` cell, so running the notebook top to bottom recreates the project tree (`rtl/`, `model/`, `tb/`, `results/`) next to the notebook and then runs the tests. The outputs saved in this notebook are from a full run.

In [1]:
import os, shutil, subprocess, sys
for d in ("rtl", "model", "tb", "results"):
    os.makedirs(d, exist_ok=True)
print("python", sys.version.split()[0])
print(subprocess.run(["iverilog", "-V"], capture_output=True, text=True).stdout.splitlines()[0])
import cocotb, numpy; print("cocotb", cocotb.__version__, "| numpy", numpy.__version__)

python 3.13.18
Icarus Verilog version 12.0 (stable) ()


cocotb 2.1.0 | numpy 2.5.3


## 3. RTL source (`rtl/`)

Thirteen Verilog-2005 files, about 2,600 lines. The same RTL goes through OpenLane 2 to a sky130 GDS.

### `rtl/raijin_dpu.v`
Dot-product unit: 16 INT8 products, 2-stage adder tree.

In [2]:
%%writefile rtl/raijin_dpu.v
// Raijin dot-product unit (DPU)
// D signed INT8 multipliers and an adder tree, registered in two stages.
// Stage 1 registers D/4 partial sums of four products each; stage 2 registers
// the 20-bit dot product. A tag and the summed row exponent ride alongside.
`default_nettype none
module raijin_dpu #(
    parameter D  = 16,
    parameter TW = 8          // tag width
) (
    input  wire              clk,
    input  wire              rst_n,
    input  wire              in_valid,
    input  wire [TW-1:0]     in_tag,
    input  wire [D*8-1:0]    a,        // row of signed INT8, element c at [c*8 +: 8]
    input  wire [D*8-1:0]    b,
    input  wire [7:0]        ea,       // E8M0 row exponents
    input  wire [7:0]        eb,
    output reg               out_valid,
    output reg  [TW-1:0]     out_tag,
    output reg  signed [19:0] dot,
    output reg  [8:0]        esum
);
    localparam G = D / 4;

    // ---- stage 1: products and groups of four ----
    reg signed [17:0] part_r [0:G-1];
    reg               v1;
    reg [TW-1:0]      tag1;
    reg [8:0]         es1;

    integer gi, ci;
    reg signed [17:0] acc4;
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            v1 <= 1'b0;
        end else begin
            v1 <= in_valid;
        end
    end

    always @(posedge clk) begin
        tag1 <= in_tag;
        es1  <= {1'b0, ea} + {1'b0, eb};
        for (gi = 0; gi < G; gi = gi + 1) begin
            acc4 = 18'sd0;
            for (ci = 0; ci < 4; ci = ci + 1)
                acc4 = acc4 + $signed(a[(gi*4+ci)*8 +: 8]) * $signed(b[(gi*4+ci)*8 +: 8]);
            part_r[gi] <= acc4;
        end
    end

    // ---- stage 2: final sum ----
    integer si;
    reg signed [19:0] tot;
    always @(*) begin
        tot = 20'sd0;
        for (si = 0; si < G; si = si + 1)
            tot = tot + part_r[si];
    end

    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) out_valid <= 1'b0;
        else        out_valid <= v1;
    end

    always @(posedge clk) begin
        out_tag <= tag1;
        esum    <= es1;
        dot     <= tot;
    end
endmodule
`default_nettype wire


Writing rtl/raijin_dpu.v


### `rtl/raijin_requant.v`
Score requantiser to Q7.4 with exponent shift and saturation.

In [3]:
%%writefile rtl/raijin_requant.v
// Raijin score requantiser (combinational)
// s = sat12(dot << sh) or sat12(dot >>> -sh), with sh = e_q + e_k - 262 + offset.
// The left shift is clamped to 15 and the right shift to 31.
`default_nettype none
module raijin_requant (
    input  wire signed [19:0] dot,
    input  wire        [8:0]  esum,
    input  wire signed [7:0]  offset,
    output reg  signed [11:0] s
);
    wire signed [11:0] sh = $signed({3'b000, esum}) - 12'sd262 + offset;
    reg  signed [35:0] wide;
    reg  [4:0]         amt;
    always @(*) begin
        if (sh >= 0) begin
            amt  = (sh > 12'sd15) ? 5'd15 : sh[4:0];
            wide = $signed({{16{dot[19]}}, dot}) <<< amt;
        end else begin
            amt  = (-sh > 12'sd31) ? 5'd31 : (-sh);
            wide = $signed({{16{dot[19]}}, dot}) >>> amt;
        end
        if (wide > 36'sd2047)       s = 12'sd2047;
        else if (wide < -36'sd2048) s = -12'sd2048;
        else                        s = wide[11:0];
    end
endmodule
`default_nettype wire


Writing rtl/raijin_requant.v


### `rtl/raijin_exp2.v`
Base-2 exponential: exact LUT mode and the two multiplier-free modes (MF1, MF2).

In [4]:
%%writefile rtl/raijin_exp2.v
// Raijin base-2 exponent unit (combinational)
// t = 16*m - s (>= 0), k = t >> 4, f = t & 15.
//   exact mode : p = LUT[f] >> k            (p = 0 when k >= 9)
//   MF1 mode   : p = 2^-(k + s1(f))          (one power of two)
//   MF2 mode   : p = 2^-(k + s1(f)) + 2^-(k + s2(f))
// p is unsigned Q1.8. sa/sb are the shift codes (15 = no term) that the
// multiplier-free AV path uses instead of p.
`default_nettype none
module raijin_exp2 (
    input  wire signed [11:0] s,
    input  wire signed [8:0]  m,
    input  wire        [1:0]  mode,     // 0 exact, 1 MF1, 2 MF2
    output reg         [8:0]  p,
    output reg         [3:0]  sa,
    output reg         [3:0]  sb
);
    wire signed [14:0] t = ($signed(m) <<< 4) - $signed(s);  // 0 .. 6144
    wire [10:0] k = t[14:4];
    wire [3:0]  f = t[3:0];

    reg [8:0] lut;
    reg [3:0] s1, s2;
    always @(*) begin
        case (f)
            4'd0:  lut = 9'd256; 4'd1:  lut = 9'd245; 4'd2:  lut = 9'd235; 4'd3:  lut = 9'd225;
            4'd4:  lut = 9'd215; 4'd5:  lut = 9'd206; 4'd6:  lut = 9'd197; 4'd7:  lut = 9'd189;
            4'd8:  lut = 9'd181; 4'd9:  lut = 9'd173; 4'd10: lut = 9'd166; 4'd11: lut = 9'd159;
            4'd12: lut = 9'd152; 4'd13: lut = 9'd146; 4'd14: lut = 9'd140; default: lut = 9'd134;
        endcase
        if (mode == 2'd1) begin
            s1 = (f >= 4'd7) ? 4'd1 : 4'd0;
            s2 = 4'd15;
        end else begin
            s1 = (f >= 4'd4) ? 4'd1 : 4'd0;
            case (f)
                4'd0, 4'd1, 4'd2, 4'd3:           s2 = 4'd15;
                4'd4, 4'd5, 4'd6, 4'd7, 4'd8:     s2 = 4'd2;
                4'd9, 4'd10, 4'd11, 4'd12:        s2 = 4'd3;
                4'd13:                            s2 = 4'd4;
                4'd14:                            s2 = 4'd5;
                default:                          s2 = 4'd6;
            endcase
        end
    end

    function [8:0] term;
        input [3:0] sh;
        begin
            term = (sh > 4'd8) ? 9'd0 : (9'd256 >> sh);
        end
    endfunction

    wire [11:0] ka = {1'b0, k} + s1;
    wire [11:0] kb = {1'b0, k} + s2;
    always @(*) begin
        if (mode == 2'd0) begin
            p  = (k >= 11'd9) ? 9'd0 : (lut >> k[3:0]);
            sa = 4'd15;
            sb = 4'd15;
        end else begin
            sa = (ka > 12'd15) ? 4'd15 : ka[3:0];
            sb = (s2 == 4'd15 || kb > 12'd15) ? 4'd15 : kb[3:0];
            p  = term(sa) + term(sb);
        end
    end
endmodule
`default_nettype wire


Writing rtl/raijin_exp2.v


### `rtl/raijin_softmax_lane.v`
One online-softmax lane: running max, rescaled running sum l (14-bit Q6.8) and the 1/l restoring divider.

In [5]:
%%writefile rtl/raijin_softmax_lane.v
// Raijin softmax lane: integer running max, base-2 exponent, denominator,
// and a sequential reciprocal. One lane serves one query row.
//
//   Stage A (registered): m' = m_update ? max(m, ceil(s)) : m,  delta = m' - m
//   Stage B (registered): p = exp2(s - m'),  l <= (l >> delta) + p
//
// Batch mode, pass 1 : s_valid, m_update, !emit   (max only)
// Batch mode, pass 2 : s_valid, !m_update, emit   (exp and sum, delta = 0)
// Online mode        : s_valid, m_update, emit    (max, exp, rescale every key)
//
// recip_start launches floor(2^19 / l), done 13 cycles later (r_valid).
//
// Group pipelining (fused dataflow): first marks the first key of a new query
// group and restarts m and l without a clear cycle, so the next group can
// stream right behind the previous one. last marks a group's final key: the
// lane latches the final l and starts the reciprocal on its own, so the next
// group's keys can update l while the division runs. Both are 0 elsewhere.
// mask (causal pass) passes a key through without touching m or l and with
// p = 0, so the accumulators skip it; out_mask tells the core not to count it
// as an underflow skip.
// thr (key skip, 0 = off): a key whose score lies thr or more powers of two
// below the running max gets p = 0 (and no MF terms), like an underflow.
`default_nettype none
module raijin_softmax_lane #(
    parameter TW = 8
) (
    input  wire              clk,
    input  wire              rst_n,
    input  wire              clear,
    input  wire [1:0]        mode,
    input  wire              s_valid,
    input  wire signed [11:0] s,
    input  wire [TW-1:0]     tag,
    input  wire              m_update,
    input  wire              emit,
    input  wire              first,      // first key of a group (restart m, l)
    input  wire              last,       // last key of a group (auto reciprocal)
    input  wire              mask,       // key masked for this row (causal)
    input  wire [3:0]        thr,        // key-skip threshold (0 = off)
    // stage B outputs
    output reg               out_valid,
    output reg  [TW-1:0]     out_tag,
    output reg  [8:0]        out_p,
    output reg  [3:0]        out_sa,
    output reg  [3:0]        out_sb,
    output wire [4:0]        out_delta,
    output reg               out_first,
    output reg               out_last,
    output reg               out_mask,
    // state
    output reg  signed [8:0] m,
    output reg  [13:0]       l,          // Q6.8: sums up to 63.99 (32 keys of 1.0)
    // reciprocal
    input  wire              recip_start,
    output reg               r_valid,
    output reg  [12:0]       r
);
    // ---------------- stage A ----------------
    wire signed [12:0] s_p15 = $signed(s) + 13'sd15;
    wire signed [8:0]  mc    = s_p15 >>> 4;
    wire signed [8:0]  m_b   = first ? -9'sd256 : m;    // m before this key
    wire signed [8:0]  mn    = (m_update && !mask && (mc > m_b)) ? mc : m_b;
    wire        [9:0]  dlt   = $signed({mn[8], mn}) - $signed({m_b[8], m_b});

    reg               a_valid;
    reg  [TW-1:0]     a_tag;
    reg signed [11:0] a_s;
    reg signed [8:0]  a_m;
    reg        [9:0]  a_delta;
    reg               a_first, a_last, a_mask;

    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            m       <= -9'sd256;
            a_valid <= 1'b0;
        end else if (clear) begin
            m       <= -9'sd256;
            a_valid <= 1'b0;
        end else begin
            a_valid <= s_valid & emit;
            if (s_valid) m <= mn;
        end
    end

    always @(posedge clk) begin
        a_tag   <= tag;
        a_s     <= s;
        a_m     <= mn;
        a_delta <= dlt;
        a_first <= first;
        a_last  <= last;
        a_mask  <= mask;
    end

    // ---------------- stage B ----------------
    wire [8:0] p;
    wire [3:0] sa, sb;
    raijin_exp2 u_exp (.s(a_s), .m(a_m), .mode(mode), .p(p), .sa(sa), .sb(sb));

    wire [4:0]  dsh = (a_delta > 10'd31) ? 5'd31 : a_delta[4:0];
    wire [13:0] l_b  = a_first ? 14'd0 : l;              // l before this key
    wire [13:0] l_sh = l_b >> dsh;
    wire signed [14:0] t_k  = ($signed(a_m) <<< 4) - $signed(a_s);
    wire        prune = (thr != 4'd0) && (t_k[14:4] >= {7'd0, thr});
    wire        zero  = a_mask || prune;
    wire [8:0]  p_e  = zero ? 9'd0 : p;                   // masked / pruned: nothing
    wire [13:0] l_nx = l_sh + {5'd0, p_e};
    reg         auto_go;                                  // reciprocal of l_fin
    reg  [13:0] l_fin;
    reg  [4:0]  b_delta;
    assign out_delta = b_delta;

    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            out_valid <= 1'b0;
            l         <= 14'd0;
            auto_go   <= 1'b0;
        end else if (clear) begin
            out_valid <= 1'b0;
            l         <= 14'd0;
            auto_go   <= 1'b0;
        end else begin
            out_valid <= a_valid;
            auto_go   <= a_valid && a_last;
            if (a_valid) l <= l_nx;
        end
    end
    always @(posedge clk) begin
        if (a_valid && a_last) l_fin <= l_nx;
    end

    always @(posedge clk) begin
        out_tag <= a_tag;
        out_p   <= p_e;
        out_sa  <= zero ? 4'd15 : sa;
        out_sb  <= zero ? 4'd15 : sb;
        b_delta <= dsh;
        out_first <= a_valid && a_first;
        out_last  <= a_valid && a_last;
        out_mask  <= a_valid && a_mask;
    end

    // ---------------- reciprocal: floor(2^19 / l), l >= 128 ----------------
    reg [13:0] rem;
    reg [3:0]  cnt;
    reg        busy;
    reg [13:0] l_div;                 // divisor, latched at the start
    wire [14:0] rem2 = {rem, 1'b0};
    wire        ge   = rem2 >= {1'b0, l_div};
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            busy    <= 1'b0;
            r_valid <= 1'b0;
            r       <= 13'd0;
            rem     <= 14'd0;
            cnt     <= 4'd0;
        end else if (recip_start || auto_go) begin
            l_div   <= auto_go ? l_fin : l;
            busy    <= 1'b1;
            r_valid <= 1'b0;
            rem     <= 14'd64;
            cnt     <= 4'd12;
            r       <= 13'd0;
        end else if (busy) begin
            rem <= ge ? (rem2[13:0] - l_div) : rem2[13:0];
            r   <= {r[11:0], ge};
            if (cnt == 4'd0) begin
                busy    <= 1'b0;
                r_valid <= 1'b1;
            end
            cnt <= cnt - 4'd1;
        end
    end
endmodule
`default_nettype wire


Writing rtl/raijin_softmax_lane.v


### `rtl/raijin_acc_array.v`
Accumulate array for P·V (outer-product form).

In [6]:
%%writefile rtl/raijin_acc_array.v
// Raijin accumulate array: R rows x D lanes of 24-bit accumulators in
// outer-product form. All rows share one V row; each row has its own p.
//
//   acc[r][c] <= (acc[r][c] >>> delta[r]) + ((p[r] * v[c]) >>> dv)
//
// Exact mode uses a 9x8 multiplier per lane. Multiplier-free mode (mf = 1)
// replaces it with two shifts and an add driven by the codes sa/sb, which give
// the same value because p = 2^(8-sa) + 2^(8-sb) there.
// A row whose en is low keeps its value (underflow skip, delta is 0 then).
// The product is registered, so an update lands two edges after its inputs.
// first (group pipelining) restarts every row at this update instead of
// accumulating into the previous group's values, so no clear cycle is needed.
`default_nettype none
module raijin_acc_array #(
    parameter R = 4,
    parameter D = 16
) (
    input  wire              clk,
    input  wire              rst_n,
    input  wire              clear,
    input  wire              mf,
    input  wire [D*8-1:0]    v,
    input  wire [4:0]        dv,
    input  wire [R-1:0]      en,
    input  wire              first,
    input  wire [R*9-1:0]    p,
    input  wire [R*4-1:0]    sa,
    input  wire [R*4-1:0]    sb,
    input  wire [R*5-1:0]    delta,
    output wire [R*D*24-1:0] acc
);
    genvar gr, gc;
    generate
        for (gr = 0; gr < R; gr = gr + 1) begin : g_row
            wire [8:0] pr  = p[gr*9 +: 9];
            wire [3:0] sar = sa[gr*4 +: 4];
            wire [3:0] sbr = sb[gr*4 +: 4];
            wire [4:0] dr  = delta[gr*5 +: 5];
            reg  [4:0] dr_r;
            reg        en_r, first_r;
            always @(posedge clk) dr_r <= dr;
            always @(posedge clk or negedge rst_n)
                if (!rst_n) begin en_r <= 1'b0; first_r <= 1'b0; end
                else begin
                    en_r    <= en[gr] && !clear;
                    first_r <= first && !clear;
                end
            for (gc = 0; gc < D; gc = gc + 1) begin : g_lane
                wire signed [7:0]  vc = v[gc*8 +: 8];
                // exact: 9x8 multiply
                wire signed [17:0] prod_x = $signed({1'b0, pr}) * vc;
                // multiplier-free: v << (8 - sa) + v << (8 - sb)
                wire signed [17:0] vx = {{10{vc[7]}}, vc};
                wire signed [17:0] ta = (sar > 4'd8) ? 18'sd0 : (vx <<< (4'd8 - sar));
                wire signed [17:0] tb = (sbr > 4'd8) ? 18'sd0 : (vx <<< (4'd8 - sbr));
                wire signed [17:0] prod_m = ta + tb;
                wire signed [17:0] prod = mf ? prod_m : prod_x;
                wire signed [17:0] t_sh = prod >>> dv;
                reg  signed [17:0] t_r;
                reg  signed [23:0] a;
                always @(posedge clk) t_r <= t_sh;
                always @(posedge clk or negedge rst_n) begin
                    if (!rst_n)                a <= 24'sd0;
                    else if (clear)            a <= 24'sd0;
                    else if (first_r)          a <= en_r ? $signed({{6{t_r[17]}}, t_r}) : 24'sd0;
                    else if (en_r)             a <= (a >>> dr_r) + $signed({{6{t_r[17]}}, t_r});
                end
                assign acc[(gr*D + gc)*24 +: 24] = a;
            end
        end
    endgenerate
endmodule
`default_nettype wire


Writing rtl/raijin_acc_array.v


### `rtl/raijin_dupreg.v`
Registered duplicate copies of high-fanout signals (timing closure).

In [7]:
%%writefile rtl/raijin_dupreg.v
// Raijin: C identical copies of a W-bit register.
//
// Used for signals that fan out to lanes spread across the die (1/l and the
// output shift amount). Each copy drives a local net instead of one die-wide
// buffer tree. Yosys merges identical flip-flops even when they carry
// (* keep *), so the sky130 build (RAIJIN_SKY130_CELLS) instantiates library
// flops directly: opt_merge never merges library cell instances.
`default_nettype none
module raijin_dupreg #(
    parameter W = 8,
    parameter C = 4
) (
    input  wire           clk,
    input  wire [W-1:0]   d,
    output wire [C*W-1:0] q
);
    genvar i, b;
    generate
        for (i = 0; i < C; i = i + 1) begin : g_copy
`ifdef RAIJIN_SKY130_CELLS
            for (b = 0; b < W; b = b + 1) begin : g_bit
                (* keep *) sky130_fd_sc_hd__dfxtp_1 u_ff (.CLK(clk), .D(d[b]), .Q(q[i*W + b]));
            end
`else
            reg [W-1:0] r;
            always @(posedge clk) r <= d;
            assign q[i*W +: W] = r;
`endif
        end
    endgenerate
endmodule
`default_nettype wire


Writing rtl/raijin_dupreg.v


### `rtl/raijin_out.v`
Output stage: × 1/l, leading-one detect, rounding to MXINT8.

In [8]:
%%writefile rtl/raijin_out.v
// Raijin output stage: per-row requant by 1/l and MXINT8 packing.
//   y[c]  = acc[c] * r                         (stage 1)
//   sh    = max(0, bitlen(OR_c |y[c]|') - 7)   (stage 2, leading-one detect)
//   o[c]  = sat8((y[c] + 2^(sh-1)) >>> sh)     (stage 3, as (y >>> sh) + y[sh-1])
//   e_o   = clamp(e_vmax + sh - 19, 0, 255)
//
// Timing: 1/l (r) and the shift amount (sh) each feed all D lanes, and the
// lanes spread across the die. Both are therefore held in RG copies, one per
// group of D/RG lanes, so each copy drives a short local net (raijin_dupreg).
`default_nettype none
module raijin_out #(
    parameter D  = 16,
    parameter TW = 8,
    parameter RG = 4             // copies of r and sh; must divide D
) (
    input  wire              clk,
    input  wire              rst_n,
    input  wire              in_valid,
    input  wire [TW-1:0]     in_tag,
    input  wire [D*24-1:0]   acc,
    input  wire [RG*13-1:0]  r,         // RG identical copies of 1/l
    input  wire [7:0]        emax,
    output reg               out_valid,
    output reg  [TW-1:0]     out_tag,
    output reg  [D*8-1:0]    o,
    output reg  [7:0]        eo
);
    // ---------------- stage 1 ----------------
    reg signed [37:0] y1 [0:D-1];
    reg               v1;
    reg [TW-1:0]      tag1;
    reg [7:0]         em1, em2;
    // one loop variable per always block, so synthesis never infers a register for it
    integer c1, c2, c3, c4;
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) v1 <= 1'b0;
        else        v1 <= in_valid;
    end
    always @(posedge clk) begin
        tag1 <= in_tag;
        em1  <= emax;
        for (c1 = 0; c1 < D; c1 = c1 + 1)
            y1[c1] <= $signed(acc[c1*24 +: 24]) * $signed({1'b0, r[(c1/(D/RG))*13 +: 13]});
    end

    // ---------------- stage 2 ----------------
    reg [37:0] orv;
    reg [5:0]  nb;
    integer b;
    always @(*) begin
        orv = 38'd0;
        for (c2 = 0; c2 < D; c2 = c2 + 1)
            orv = orv | (y1[c2] ^ {38{y1[c2][37]}});
        nb = 6'd0;
        for (b = 0; b < 38; b = b + 1)
            if (orv[b]) nb = b + 1;
    end

    reg signed [37:0] y2 [0:D-1];
    reg               v2;
    reg [TW-1:0]      tag2;
    wire [RG*6-1:0]   sh2;               // RG copies of the shift amount
    wire [5:0]        sh_n = (nb > 6'd7) ? (nb - 6'd7) : 6'd0;
    raijin_dupreg #(.W(6), .C(RG)) u_sh2 (.clk(clk), .d(sh_n), .q(sh2));
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) v2 <= 1'b0;
        else        v2 <= v1;
    end
    always @(posedge clk) begin
        tag2 <= tag1;
        em2  <= em1;
        for (c3 = 0; c3 < D; c3 = c3 + 1) y2[c3] <= y1[c3];
    end

    // ---------------- stage 3 ----------------
    reg signed [38:0] rnd;
    reg [D*8-1:0]     o_n;
    reg [5:0]         sh_l;
    wire [9:0]        eo_w = {2'b00, em2} + {4'd0, sh2[5:0]};
    always @(*) begin
        for (c4 = 0; c4 < D; c4 = c4 + 1) begin
            sh_l = sh2[(c4/(D/RG))*6 +: 6];
            // round half up: (y + 2^(sh-1)) >>> sh == (y >>> sh) + y[sh-1], so the
            // 39-bit carry chain becomes a shift plus a one-bit increment
            if (sh_l == 6'd0) rnd = {y2[c4][37], y2[c4]};
            else              rnd = ($signed({y2[c4][37], y2[c4]}) >>> sh_l) + $signed({38'd0, y2[c4][sh_l - 6'd1]});
            if (rnd > 39'sd127)       o_n[c4*8 +: 8] = 8'sd127;
            else if (rnd < -39'sd128) o_n[c4*8 +: 8] = -8'sd128;
            else                      o_n[c4*8 +: 8] = rnd[7:0];
        end
    end
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) out_valid <= 1'b0;
        else        out_valid <= v2;
    end
    always @(posedge clk) begin
        out_tag <= tag2;
        o       <= o_n;
        eo      <= (eo_w < 10'd19) ? 8'd0 : ((eo_w - 10'd19) > 10'd255 ? 8'd255 : (eo_w - 10'd19));
    end
endmodule
`default_nettype wire


Writing rtl/raijin_out.v


### `rtl/raijin_rope.v`
RoPE-on-write rotator (Q1.8 cos/sin table, phase multiplier).

In [9]:
%%writefile rtl/raijin_rope.v
// Raijin RoPE-on-write rotator (feature 9).
//
// With ROPE on, the host writes Q and K rows as usual, one byte at a time,
// and each element pair (2i, 2i+1) is turned by pos * theta_i on its way
// into SRAM, theta_i = 10000^(-2i/16). Real models cache K after RoPE, so
// rotating on write is the natural place, and it keeps the rotator off the
// pass datapath: no cycles, no timing cost on the SRAM-to-DPU path.
//
//   even byte (x, column 2i): latched, with cos/sin for (pos, i) from the
//                             phase multiplier and a quarter-wave table
//   odd byte  (y, column 2i+1): x' = sat8((x c - y s + 128) >> 8),
//                               y' = sat8((x s + y c + 128) >> 8), written as one
//                               two-byte SRAM write in place of the host's byte
//
// cos/sin are Q1.8. The result is rounded and saturated to INT8, and the row
// keeps its exponent (a rotation preserves the pair's length, so saturation
// only clips a pair whose length is above 127).
// Phase is in turns, Q0.20: phase = pos * W_i mod 2^20, rounded to 10 bits.
// Only D = 16 (eight frequencies) is tabulated.
`default_nettype none
module raijin_rope (
    input  wire              clk,
    input  wire              we_even,   // host writes x of pair i
    input  wire [2:0]        pair,      // i
    input  wire [7:0]        pos,
    input  wire [7:0]        wdata,     // x (even) or y (odd)
    output wire [7:0]        xo,        // x'
    output wire [7:0]        yo         // y'
);
    function [17:0] wfreq;
        input [2:0] i;
        case (i)
            3'd0: wfreq = 18'd166886;
            3'd1: wfreq = 18'd52774;
            3'd2: wfreq = 18'd16689;
            3'd3: wfreq = 18'd5277;
            3'd4: wfreq = 18'd1669;
            3'd5: wfreq = 18'd528;
            3'd6: wfreq = 18'd167;
            3'd7: wfreq = 18'd53;
            default: wfreq = 18'd0;
        endcase
    endfunction

    // quarter-wave sine, Q1.8; entry 256 (= 256) is handled below
    function [8:0] qsin;
        input [7:0] k;
        case (k)
            8'd0: qsin = 9'd0;
            8'd1: qsin = 9'd2;
            8'd2: qsin = 9'd3;
            8'd3: qsin = 9'd5;
            8'd4: qsin = 9'd6;
            8'd5: qsin = 9'd8;
            8'd6: qsin = 9'd9;
            8'd7: qsin = 9'd11;
            8'd8: qsin = 9'd13;
            8'd9: qsin = 9'd14;
            8'd10: qsin = 9'd16;
            8'd11: qsin = 9'd17;
            8'd12: qsin = 9'd19;
            8'd13: qsin = 9'd20;
            8'd14: qsin = 9'd22;
            8'd15: qsin = 9'd24;
            8'd16: qsin = 9'd25;
            8'd17: qsin = 9'd27;
            8'd18: qsin = 9'd28;
            8'd19: qsin = 9'd30;
            8'd20: qsin = 9'd31;
            8'd21: qsin = 9'd33;
            8'd22: qsin = 9'd34;
            8'd23: qsin = 9'd36;
            8'd24: qsin = 9'd38;
            8'd25: qsin = 9'd39;
            8'd26: qsin = 9'd41;
            8'd27: qsin = 9'd42;
            8'd28: qsin = 9'd44;
            8'd29: qsin = 9'd45;
            8'd30: qsin = 9'd47;
            8'd31: qsin = 9'd48;
            8'd32: qsin = 9'd50;
            8'd33: qsin = 9'd51;
            8'd34: qsin = 9'd53;
            8'd35: qsin = 9'd55;
            8'd36: qsin = 9'd56;
            8'd37: qsin = 9'd58;
            8'd38: qsin = 9'd59;
            8'd39: qsin = 9'd61;
            8'd40: qsin = 9'd62;
            8'd41: qsin = 9'd64;
            8'd42: qsin = 9'd65;
            8'd43: qsin = 9'd67;
            8'd44: qsin = 9'd68;
            8'd45: qsin = 9'd70;
            8'd46: qsin = 9'd71;
            8'd47: qsin = 9'd73;
            8'd48: qsin = 9'd74;
            8'd49: qsin = 9'd76;
            8'd50: qsin = 9'd77;
            8'd51: qsin = 9'd79;
            8'd52: qsin = 9'd80;
            8'd53: qsin = 9'd82;
            8'd54: qsin = 9'd83;
            8'd55: qsin = 9'd85;
            8'd56: qsin = 9'd86;
            8'd57: qsin = 9'd88;
            8'd58: qsin = 9'd89;
            8'd59: qsin = 9'd91;
            8'd60: qsin = 9'd92;
            8'd61: qsin = 9'd94;
            8'd62: qsin = 9'd95;
            8'd63: qsin = 9'd97;
            8'd64: qsin = 9'd98;
            8'd65: qsin = 9'd99;
            8'd66: qsin = 9'd101;
            8'd67: qsin = 9'd102;
            8'd68: qsin = 9'd104;
            8'd69: qsin = 9'd105;
            8'd70: qsin = 9'd107;
            8'd71: qsin = 9'd108;
            8'd72: qsin = 9'd109;
            8'd73: qsin = 9'd111;
            8'd74: qsin = 9'd112;
            8'd75: qsin = 9'd114;
            8'd76: qsin = 9'd115;
            8'd77: qsin = 9'd117;
            8'd78: qsin = 9'd118;
            8'd79: qsin = 9'd119;
            8'd80: qsin = 9'd121;
            8'd81: qsin = 9'd122;
            8'd82: qsin = 9'd123;
            8'd83: qsin = 9'd125;
            8'd84: qsin = 9'd126;
            8'd85: qsin = 9'd128;
            8'd86: qsin = 9'd129;
            8'd87: qsin = 9'd130;
            8'd88: qsin = 9'd132;
            8'd89: qsin = 9'd133;
            8'd90: qsin = 9'd134;
            8'd91: qsin = 9'd136;
            8'd92: qsin = 9'd137;
            8'd93: qsin = 9'd138;
            8'd94: qsin = 9'd140;
            8'd95: qsin = 9'd141;
            8'd96: qsin = 9'd142;
            8'd97: qsin = 9'd144;
            8'd98: qsin = 9'd145;
            8'd99: qsin = 9'd146;
            8'd100: qsin = 9'd147;
            8'd101: qsin = 9'd149;
            8'd102: qsin = 9'd150;
            8'd103: qsin = 9'd151;
            8'd104: qsin = 9'd152;
            8'd105: qsin = 9'd154;
            8'd106: qsin = 9'd155;
            8'd107: qsin = 9'd156;
            8'd108: qsin = 9'd157;
            8'd109: qsin = 9'd159;
            8'd110: qsin = 9'd160;
            8'd111: qsin = 9'd161;
            8'd112: qsin = 9'd162;
            8'd113: qsin = 9'd164;
            8'd114: qsin = 9'd165;
            8'd115: qsin = 9'd166;
            8'd116: qsin = 9'd167;
            8'd117: qsin = 9'd168;
            8'd118: qsin = 9'd170;
            8'd119: qsin = 9'd171;
            8'd120: qsin = 9'd172;
            8'd121: qsin = 9'd173;
            8'd122: qsin = 9'd174;
            8'd123: qsin = 9'd175;
            8'd124: qsin = 9'd177;
            8'd125: qsin = 9'd178;
            8'd126: qsin = 9'd179;
            8'd127: qsin = 9'd180;
            8'd128: qsin = 9'd181;
            8'd129: qsin = 9'd182;
            8'd130: qsin = 9'd183;
            8'd131: qsin = 9'd184;
            8'd132: qsin = 9'd185;
            8'd133: qsin = 9'd186;
            8'd134: qsin = 9'd188;
            8'd135: qsin = 9'd189;
            8'd136: qsin = 9'd190;
            8'd137: qsin = 9'd191;
            8'd138: qsin = 9'd192;
            8'd139: qsin = 9'd193;
            8'd140: qsin = 9'd194;
            8'd141: qsin = 9'd195;
            8'd142: qsin = 9'd196;
            8'd143: qsin = 9'd197;
            8'd144: qsin = 9'd198;
            8'd145: qsin = 9'd199;
            8'd146: qsin = 9'd200;
            8'd147: qsin = 9'd201;
            8'd148: qsin = 9'd202;
            8'd149: qsin = 9'd203;
            8'd150: qsin = 9'd204;
            8'd151: qsin = 9'd205;
            8'd152: qsin = 9'd206;
            8'd153: qsin = 9'd207;
            8'd154: qsin = 9'd207;
            8'd155: qsin = 9'd208;
            8'd156: qsin = 9'd209;
            8'd157: qsin = 9'd210;
            8'd158: qsin = 9'd211;
            8'd159: qsin = 9'd212;
            8'd160: qsin = 9'd213;
            8'd161: qsin = 9'd214;
            8'd162: qsin = 9'd215;
            8'd163: qsin = 9'd215;
            8'd164: qsin = 9'd216;
            8'd165: qsin = 9'd217;
            8'd166: qsin = 9'd218;
            8'd167: qsin = 9'd219;
            8'd168: qsin = 9'd220;
            8'd169: qsin = 9'd220;
            8'd170: qsin = 9'd221;
            8'd171: qsin = 9'd222;
            8'd172: qsin = 9'd223;
            8'd173: qsin = 9'd224;
            8'd174: qsin = 9'd224;
            8'd175: qsin = 9'd225;
            8'd176: qsin = 9'd226;
            8'd177: qsin = 9'd227;
            8'd178: qsin = 9'd227;
            8'd179: qsin = 9'd228;
            8'd180: qsin = 9'd229;
            8'd181: qsin = 9'd229;
            8'd182: qsin = 9'd230;
            8'd183: qsin = 9'd231;
            8'd184: qsin = 9'd231;
            8'd185: qsin = 9'd232;
            8'd186: qsin = 9'd233;
            8'd187: qsin = 9'd233;
            8'd188: qsin = 9'd234;
            8'd189: qsin = 9'd235;
            8'd190: qsin = 9'd235;
            8'd191: qsin = 9'd236;
            8'd192: qsin = 9'd237;
            8'd193: qsin = 9'd237;
            8'd194: qsin = 9'd238;
            8'd195: qsin = 9'd238;
            8'd196: qsin = 9'd239;
            8'd197: qsin = 9'd239;
            8'd198: qsin = 9'd240;
            8'd199: qsin = 9'd241;
            8'd200: qsin = 9'd241;
            8'd201: qsin = 9'd242;
            8'd202: qsin = 9'd242;
            8'd203: qsin = 9'd243;
            8'd204: qsin = 9'd243;
            8'd205: qsin = 9'd244;
            8'd206: qsin = 9'd244;
            8'd207: qsin = 9'd245;
            8'd208: qsin = 9'd245;
            8'd209: qsin = 9'd245;
            8'd210: qsin = 9'd246;
            8'd211: qsin = 9'd246;
            8'd212: qsin = 9'd247;
            8'd213: qsin = 9'd247;
            8'd214: qsin = 9'd248;
            8'd215: qsin = 9'd248;
            8'd216: qsin = 9'd248;
            8'd217: qsin = 9'd249;
            8'd218: qsin = 9'd249;
            8'd219: qsin = 9'd249;
            8'd220: qsin = 9'd250;
            8'd221: qsin = 9'd250;
            8'd222: qsin = 9'd250;
            8'd223: qsin = 9'd251;
            8'd224: qsin = 9'd251;
            8'd225: qsin = 9'd251;
            8'd226: qsin = 9'd252;
            8'd227: qsin = 9'd252;
            8'd228: qsin = 9'd252;
            8'd229: qsin = 9'd252;
            8'd230: qsin = 9'd253;
            8'd231: qsin = 9'd253;
            8'd232: qsin = 9'd253;
            8'd233: qsin = 9'd253;
            8'd234: qsin = 9'd254;
            8'd235: qsin = 9'd254;
            8'd236: qsin = 9'd254;
            8'd237: qsin = 9'd254;
            8'd238: qsin = 9'd254;
            8'd239: qsin = 9'd255;
            8'd240: qsin = 9'd255;
            8'd241: qsin = 9'd255;
            8'd242: qsin = 9'd255;
            8'd243: qsin = 9'd255;
            8'd244: qsin = 9'd255;
            8'd245: qsin = 9'd255;
            8'd246: qsin = 9'd256;
            8'd247: qsin = 9'd256;
            8'd248: qsin = 9'd256;
            8'd249: qsin = 9'd256;
            8'd250: qsin = 9'd256;
            8'd251: qsin = 9'd256;
            8'd252: qsin = 9'd256;
            8'd253: qsin = 9'd256;
            8'd254: qsin = 9'd256;
            8'd255: qsin = 9'd256;
            default: qsin = 9'd0;
        endcase
    endfunction

    wire [25:0] ph_full = pos * wfreq(pair);
    wire [19:0] ph      = ph_full[19:0];
    wire [20:0] ph_r    = {1'b0, ph} + 21'd512;
    wire [9:0]  a       = ph_r[19:10];
    wire [1:0]  quad    = a[9:8];
    wire [7:0]  k       = a[7:0];
    wire [8:0]  s0      = qsin(k);
    wire [8:0]  c0      = (k == 8'd0) ? 9'd256 : qsin(8'd0 - k);   // sin(256 - k)
    reg  signed [9:0] c_n, s_n;
    always @(*) begin
        case (quad)
            2'd0: begin c_n =  $signed({1'b0, c0}); s_n =  $signed({1'b0, s0}); end
            2'd1: begin c_n = -$signed({1'b0, s0}); s_n =  $signed({1'b0, c0}); end
            2'd2: begin c_n = -$signed({1'b0, c0}); s_n = -$signed({1'b0, s0}); end
            default: begin c_n = $signed({1'b0, s0}); s_n = -$signed({1'b0, c0}); end
        endcase
    end

    reg signed [7:0] x_r;
    reg signed [9:0] c_r, s_r;
    always @(posedge clk)
        if (we_even) begin
            x_r <= wdata;
            c_r <= c_n;
            s_r <= s_n;
        end

    wire signed [7:0]  y  = wdata;
    wire signed [18:0] v1 = x_r * c_r - y * s_r + 19'sd128;
    wire signed [18:0] v2 = x_r * s_r + y * c_r + 19'sd128;
    wire signed [10:0] t1 = v1[18:8];
    wire signed [10:0] t2 = v2[18:8];
    assign xo = (t1 > 11'sd127) ? 8'h7F : (t1 < -11'sd128) ? 8'h80 : t1[7:0];
    assign yo = (t2 > 11'sd127) ? 8'h7F : (t2 < -11'sd128) ? 8'h80 : t2[7:0];
endmodule
`default_nettype wire


Writing rtl/raijin_rope.v


### `rtl/raijin_hh.v`
Heavy-hitter (H2O) tracker: accumulators and a 3-stage pipelined argmin walk.

In [10]:
%%writefile rtl/raijin_hh.v
// Raijin heavy-hitter tracker (feature 15): per-slot attention-weight
// accumulators for KV-cache eviction (H2O-style), with an on-chip argmin.
//
// During a single-row decode step the core passes every key's score s (Q.4
// fixed point, base-2 units) and its physical cache slot. When the step's keys
// have all streamed, the final running maximum m of that row is known and the
// tracker walks slots 0..NK-1, one per cycle, adding each captured key's
// max-normalised weight
//     w = 2^((s - 16 m) / 16)        in Q0.8, i.e. 256 for the top key,
// approximated as (frac >= 1/2 ? 181 : 256) >> floor((16 m - s) / 16)
// (181 = 256 / sqrt 2), zero once it falls below 2^-8. Accumulators saturate
// at 16 bits. The same walk finds the slot with the smallest accumulator among
// the slots that took part, skipping the first SKIP slots (attention sinks):
// that is the eviction candidate the host reads back.
//
// Bus (region 7, offsets):
//   0x000 + 2k, +1  ACC[k] low / high byte (read); any write clears ACC[k]
//   0x080  HHCTL   bit0 enable (kept), bit1 clear all, bit2 halve all (decay)
//   0x081  HHMIN   slot with the smallest ACC after the last step (read)
//   0x082  HHSKIP  slots 0..HHSKIP-1 never chosen by HHMIN (read/write)
//   0x083  HHRECENT keys among the newest HHRECENT positions of the step are
//                  never chosen either (H2O keeps a recent window) (read/write)
`default_nettype none
module raijin_hh #(
    parameter NK = 32,
    parameter LK = 5
) (
    input  wire              clk,
    input  wire              rst_n,
    // capture (core, during the decode stream)
    input  wire              cap_valid,
    input  wire [LK-1:0]     cap_slot,
    input  wire signed [11:0] cap_s,
    input  wire              cap_recent,     // key is in the step's recent window
    input  wire              cap_clear,      // start of a decode step
    // walk
    input  wire              go,             // all keys of the step have passed stage A
    input  wire signed [8:0] m_fin,
    output wire              busy,
    // bus
    input  wire              bus_we,
    input  wire [7:0]        bus_off,
    input  wire [7:0]        bus_wdata,
    output reg  [7:0]        bus_rdata,      // combinational from bus_off
    output wire              en,
    output wire [LK-1:0]     recent
);
    reg              en_r;
    reg  [LK-1:0]    skip;
    reg  [LK-1:0]    recent_r;
    reg  [NK-1:0]    rec;
    reg  [LK-1:0]    minslot;
    reg  [15:0]      acc [0:NK-1];
    reg signed [11:0] sc [0:NK-1];
    reg  [NK-1:0]    used;
    reg              walking;
    reg  [LK:0]      k;
    reg  [16:0]      best;
    reg  [LK-1:0]    best_k;
    reg signed [8:0] m_l;
    assign en   = en_r;
    assign recent = recent_r;
    assign busy = walking;

    // The walk is a 3-stage pipeline (timing: the 32:1 slot read, the weight
    // and the add-compare were one path from k to best):
    //   R: read slot kk          -> r_*   (sc, acc, flags)
    //   W: weight from r_sc, m_l -> w_*   (weight, acc, flags)
    //   A: saturating add, write acc back, argmin compare
    // Each slot is visited once, so the write-back in A never races a read in R.
    wire [LK-1:0] kk     = k[LK-1:0];
    reg               r_v, w_v;
    reg  [LK-1:0]     r_k, w_k;
    reg signed [11:0] r_sc;
    reg  [15:0]       r_acc, w_acc;
    reg               r_used, r_cand, w_used, w_cand;
    reg  [8:0]        w_w;
    reg               w_last, r_last;
    wire signed [13:0] d = ($signed({m_l[8], m_l}) <<< 4) - $signed({{2{r_sc[11]}}, r_sc});
    wire [9:0]    di     = d[13] ? 10'd0 : d[13:4];
    wire [8:0]    w0     = (d[13] || !d[3]) ? 9'd256 : 9'd181;
    wire [8:0]    w      = (di > 10'd8) ? 9'd0 : (w0 >> di[3:0]);
    wire [16:0]   sum    = {1'b0, w_acc} + {8'd0, w_w};
    wire [15:0]   nacc   = sum[16] ? 16'hFFFF : sum[15:0];
    wire          better = w_used && w_cand && {1'b0, nacc} < best;

    wire          ctl_w  = bus_we && bus_off == 8'h80;
    wire          acc_w  = bus_we && !bus_off[7];
    integer i;
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            en_r    <= 1'b0;
            skip    <= {LK{1'b0}};
            recent_r <= {LK{1'b0}};
            minslot <= {LK{1'b0}};
            walking <= 1'b0;
            used    <= {NK{1'b0}};
            k       <= 0;
            r_v     <= 1'b0;
            w_v     <= 1'b0;
            for (i = 0; i < NK; i = i + 1) acc[i] <= 16'd0;
        end else begin
            if (ctl_w) begin
                en_r <= bus_wdata[0];
                for (i = 0; i < NK; i = i + 1) begin
                    if (bus_wdata[1])      acc[i] <= 16'd0;
                    else if (bus_wdata[2]) acc[i] <= acc[i] >> 1;
                end
            end
            if (bus_we && bus_off == 8'h82) skip <= bus_wdata[LK-1:0];
            if (bus_we && bus_off == 8'h83) recent_r <= bus_wdata[LK-1:0];
            if (acc_w) acc[bus_off[LK:1]] <= 16'd0;
            if (cap_clear) used <= {NK{1'b0}};
            if (en_r && cap_valid) begin
                used[cap_slot] <= 1'b1;
                rec[cap_slot]  <= cap_recent;
            end
            if (go && en_r && !walking) begin
                walking <= 1'b1;
                k       <= 0;
                m_l     <= m_fin;
                best    <= 17'h1FFFF;
                best_k  <= {LK{1'b0}};
            end else if (walking) begin
                // stage R
                r_v    <= !k[LK];
                r_k    <= kk;
                r_sc   <= sc[kk];
                r_acc  <= acc[kk];
                r_used <= used[kk];
                r_cand <= kk >= skip && !rec[kk];
                r_last <= k == NK - 1;
                if (!k[LK]) k <= k + 1'b1;
                // stage W
                w_v    <= r_v;
                w_k    <= r_k;
                w_w    <= w;
                w_acc  <= r_acc;
                w_used <= r_used;
                w_cand <= r_cand;
                w_last <= r_v && r_last;
                // stage A
                if (w_v) begin
                    if (w_used) acc[w_k] <= nacc;
                    if (better) begin
                        best   <= {1'b0, nacc};
                        best_k <= w_k;
                    end
                    if (w_last) begin
                        walking <= 1'b0;
                        minslot <= better ? w_k : best_k;
                    end
                end
            end else begin
                r_v <= 1'b0;
                w_v <= 1'b0;
            end
        end
    end
    always @(posedge clk) begin
        if (en_r && cap_valid) sc[cap_slot] <= cap_s;
    end

    wire [15:0] acc_rd = acc[bus_off[LK:1]];
    always @(*) begin
        if (!bus_off[7])            bus_rdata = bus_off[0] ? acc_rd[15:8] : acc_rd[7:0];
        else if (bus_off == 8'h80)  bus_rdata = {7'd0, en_r};
        else if (bus_off == 8'h81)  bus_rdata = {{(8-LK){1'b0}}, minslot};
        else if (bus_off == 8'h82)  bus_rdata = {{(8-LK){1'b0}}, skip};
        else if (bus_off == 8'h83)  bus_rdata = {{(8-LK){1'b0}}, recent_r};
        else                        bus_rdata = 8'd0;
    end
endmodule
`default_nettype wire


Writing rtl/raijin_hh.v


### `rtl/raijin_mem.v`
Behavioural model of the sky130 1 KB OpenRAM macro used for Q, K, V, O and the score buffer.

In [11]:
%%writefile rtl/raijin_mem.v
// Cycle-accurate behavioural model of sky130_sram_1kbyte_1rw1r_32x256_8
// (OpenRAM, 256 words x 32 bits, byte write mask, one RW port and one R port).
//
// Port semantics match the macro at the clock-edge level: inputs are sampled
// on the rising edge; read data is valid before the next rising edge.
// The core synthesises with RAIJIN_SKY130_SRAM defined, which swaps this model
// for the real macro; the FPGA build keeps this model and infers block RAM.
// Never read and write the same address in the same cycle (the macro forbids it).
`default_nettype none
module raijin_mem (
    input  wire        clk0,
    input  wire        csb0,      // active-low select, port 0 (RW)
    input  wire        web0,      // active-low write enable
    input  wire [3:0]  wmask0,
    input  wire [7:0]  addr0,
    input  wire [31:0] din0,
    output reg  [31:0] dout0,
    input  wire        clk1,
    input  wire        csb1,      // active-low select, port 1 (R)
    input  wire [7:0]  addr1,
    output reg  [31:0] dout1
);
    reg [31:0] mem [0:255];
    integer bi;
    always @(posedge clk0) begin
        if (!csb0) begin
            if (!web0) begin
                for (bi = 0; bi < 4; bi = bi + 1)
                    if (wmask0[bi]) mem[addr0][bi*8 +: 8] <= din0[bi*8 +: 8];
            end else begin
                dout0 <= mem[addr0];
            end
        end
    end
    always @(posedge clk1) begin
        if (!csb1) dout1 <= mem[addr1];
    end
`ifndef SYNTHESIS
    always @(posedge clk0)
        if (!csb0 && !web0 && !csb1 && addr0 == addr1)
            $display("%t %m ERROR: write and read of address %0d in the same cycle", $time, addr0);
`endif
endmodule
`default_nettype wire


Writing rtl/raijin_mem.v


### `rtl/raijin_core.v`
The core: register map, controller FSM, SRAM ports and all dataflows.

In [12]:
%%writefile rtl/raijin_core.v
// Raijin attention core
// Single-head MXINT8 attention with three dataflows on the same blocks:
//   unfused : S = Q K^T into the score buffer, batch softmax per row, then O = P V
//   fused   : keys stream past P query rows with base-2 online softmax; S is never stored
//   decode  : KV-cache step. One query row attends to keys 0..LEN-1 already
//             resident in SRAM (fused dataflow, only that row's group runs)
//
// Group pipelining (fused, non-decode): query group g+1 loads and streams its
// keys right behind group g. The softmax lanes and accumulators restart on the
// first key of a group (no clear cycle), the final accumulator rows are copied
// to acc_snap, and the reciprocal, output normalise and O write of group g run
// while group g+1 streams. O rows queue in a small FIFO and are written on the
// port-0 cycles that carry no V read.
//
// Causal pass (CTRL bit 5, fused only): row i attends to keys 0..i. Group g
// streams only keys 0..gP+P-1, so the pass does about half the work, and the
// lanes mask the keys above their own row within the group's last P keys.
// A group may issue its last key only L_MIN cycles after the previous group's
// last key, so short causal groups never overrun the reciprocal and output
// stages of the group before them.
//
// Multi-query decode (CTRL bit 6 with decode): all P lanes of QROW's group
// attend to the LEN cached keys in one step, sharing every K and V read. The P
// query rows can be P heads of one grouped-query-attention group (GQA/MQA),
// or P sequences that share a prompt. With bit 7 (speculative verify), lane r
// sees keys 0..LEN-P+r: the P rows are P draft tokens whose K/V rows are the
// last P cached rows, so one step scores a whole speculative draft.
//
// Storage: Q, K, V, O and the score buffer live in four 1 KB SRAM macros
// (sky130_sram_1kbyte_1rw1r_32x256_8, one RW and one R port each). Bank b holds
// bytes 4b..4b+3 of every row, so the four banks together return one full
// 128-bit row per port per cycle: a K row on port 1 and a V row on port 0.
//   word  0 ..  N-1   Q rows          word 2N .. 3N-1  V rows
//   word  N .. 2N-1   K rows          word 3N .. 4N-1  O rows
//   word 4N + {g, j}  score buffer, lanes 2b and 2b+1 in bank b (12 bits each)
// Q rows for the current group are latched in P row registers before the keys
// stream. SRAM read data is registered once before use, so a read issued in
// cycle t is consumed in cycle t + 2.
//
// Bus: byte-wide, 16-bit address, all four inputs registered on entry.
// A write takes effect two rising edges after it is presented. A read returns
// rdata after the third rising edge (RD_LATENCY = 3).
// The accumulators and the output stage each carry one extra pipeline register
// for timing; the controller's drain counts cover them. Q/K/V/O and the exponent
// registers are accessible only while the core is idle.
//   0x0rcc  Q[r][c]   (write)          0x4000+r  e_q[r]  (r/w)
//   0x1rcc  K[r][c]   (write)          0x4100+r  e_k[r]  (r/w)
//   0x2rcc  V[r][c]   (write)          0x4200+r  e_v[r]  (r/w)
//   0x3rcc  O[r][c]   (read)           0x4300+r  e_o[r]  (read)
//   (offset = r * D + c)
//   0x5000  CTRL   w: bit0 start, bit1 fused, bits3:2 mode (0 exact, 1 MF1, 2 MF2),
//                     bit4 decode (implies the fused dataflow),
//                     bit5 causal (fused, non-decode: row i sees keys 0..i),
//                     bit6 multi-query decode (all P rows of QROW's group),
//                     bit7 speculative verify (multi, lane r sees keys 0..LEN-P+r)
//   0x5001  OFFSET signed score requant offset
//   0x5002  STATUS bit0 busy, bit1 done
//   0x5003  RD_LATENCY (3)
//   0x5004..7 CYCLES (little endian)   0x5008..B SKIPS
//   0x500C N, 0x500D D, 0x500E P, 0x500F version
//   0x5010  LEN    decode: number of cached keys, 1..N (1..2N with KVFMT INT4)
//   0x5011  QROW   decode: query row (its output lands in O[QROW])
//   0x5012  KVFMT  bit0: K is MXINT4, bit1: V is MXINT4. An INT4 region packs
//                  two tokens per SRAM word (token t's 16 nibbles at byte offset
//                  8t of its region, element 2b in the low nibble of byte b), so
//                  with both bits set the cache holds 2N tokens. Each nibble
//                  enters the INT8 datapath as x4 << 4.
//                  bit2: 4-bit K nibbles are MXFP4 (E2M1: s e1 e0 m) instead of
//                  INT4, bit3: the same for V. Both apply to every 4-bit token,
//                  in KVFMT and mixed (PFMT) modes. An E2M1 nibble enters the
//                  datapath as +-(2|x|) << 3 = 16 x, 2|x| in {0,1,2,3,4,6,8,12}.
//   0x5013  SKIPTHR bits3:0 key-skip threshold (0 = off): keys scoring SKIPTHR
//                  or more powers of two below the running max are dropped (p = 0)
//                  bit7: max-first decode (feature 17, single-row decode only).
//                  Pass A streams the K rows and finds the row's final maximum
//                  while the score of each key is latched; pass B then visits
//                  only the keys whose weight against that maximum is nonzero,
//                  reading their V rows. Keys below 2^-9 of the maximum (or
//                  SKIPTHR below it) cost neither a V read nor a cycle. The
//                  result equals decode with the maximum known in advance.
//   0x5014..7 VSKIPS (little endian) V-row reads saved this pass: no lane
//                  needed the V row, so it was never read
//   0x5018  ROPE   bit0: rotate Q and K rows on write (RoPE, interleaved pairs,
//                  theta_i = 10000^(-2i/16)); bit1: position = ROPEPOS instead
//                  of the row index. Write each pair's even byte before its odd
//                  byte. Rows keep their exponent. Not applied to an MXINT4 K
//                  region.
//   0x5019  ROPEPOS position used when ROPE bit1 is set
//   0x501A  PAGED  bit0: decode reads the KV cache through the page table;
//                  bit1: mixed precision, each page INT8 or INT4 per PFMT
//   0x501B  PFMT   bit p: logical page p is MXINT4 (mixed mode; KVFMT ignored)
//   0x501C..F PTAB page table, 2N/4 entries of 4 tokens each, entry 2b in the
//                  low nibble of byte b: logical page -> physical page. Logical
//                  token j lives in physical token PTAB[j/4]*4 + j%4 (its K/V
//                  row and its e_k / e_v). Resets to the identity map.
//   0x6000.. token window (decode): see the bus decode below
//   e_k / e_v exponents: 0x41tt / 0x42tt for tokens t = 0..2N-1
`default_nettype none
`ifdef RAIJIN_SKY130_SRAM
`define RAIJIN_RAM sky130_sram_1kbyte_1rw1r_32x256_8
`else
`define RAIJIN_RAM raijin_mem
`endif
module raijin_core #(
    parameter N = 16,       // sequence length (multiple of P, power of two)
    parameter D = 16,       // head dimension (four SRAM banks of 4 bytes)
    parameter P = 4         // DPUs / softmax lanes / accumulator rows
) (
    input  wire        clk,
    input  wire        rst_n,
    input  wire [15:0] addr,
    input  wire [7:0]  wdata,
    input  wire        we,
    input  wire        re,
    output reg  [7:0]  rdata,
    output wire        busy,
    output wire        done
);
    localparam NG  = N / P;
    localparam LN  = (N  > 1) ? $clog2(N)  : 1;
    localparam LD  = (D  > 1) ? $clog2(D)  : 1;
    localparam LG  = (NG > 1) ? $clog2(NG) : 1;
    localparam LP  = (P  > 1) ? $clog2(P)  : 1;
    localparam LK  = LN + 1;          // key index: up to 2N cached tokens (INT4)
    localparam NK  = 2 * N;
    localparam TW  = LG + LK;
    localparam VERSION = 8'h02;
    localparam NB  = 4;               // SRAM banks
    localparam NSB = (P + 1) / 2;     // banks holding the score buffer
    localparam [7:0] AQ = 0, AK = N, AV = 2 * N, AO = 3 * N, AS = 4 * N;

`ifndef SYNTHESIS
    initial begin
        if (D != 16 || 4 * N + NG * N > 256 || P > 8) begin
            $display("raijin_core: unsupported N=%0d D=%0d P=%0d for the 4 x 1 KB SRAM map", N, D, P);
            $finish;
        end
    end
`endif

    // ------------------------------------------------------------------
    // Registered bus inputs
    // ------------------------------------------------------------------
    reg [15:0] addr_r;
    reg [7:0]  wdata_r;
    reg        we_r, re_r;
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            we_r <= 1'b0;
            re_r <= 1'b0;
        end else begin
            we_r <= we;
            re_r <= re;
        end
    end
    always @(posedge clk) begin
        addr_r  <= addr;
        wdata_r <= wdata;
    end

    // ------------------------------------------------------------------
    // Small storage that stays in flip-flops: row exponents and 1/l
    // ------------------------------------------------------------------
    reg [7:0]     eqm [0:N-1];
    reg [7:0]     ekm [0:NK-1];
    reg [7:0]     evm [0:NK-1];
    reg [7:0]     eom [0:N-1];
    reg [12:0]    rinv [0:N-1];
    reg [D*8-1:0] qreg [0:P-1];       // Q rows of the current group

    // ------------------------------------------------------------------
    // Control registers
    // ------------------------------------------------------------------
    reg               busy_r, done_r;
    reg               fused, decode, causal, multi, spec;
    reg [1:0]         mode;
    reg signed [7:0]  offset;
    reg [LK:0]        len;
    reg               k4, v4;         // KVFMT: MXINT4 K (bit 0), V (bit 1)
    reg               kfp, vfp;       // KVFMT: 4-bit K (bit 2), V (bit 3) is MXFP4
    reg  [3:0]        thr;            // SKIPTHR
    reg               mfirst;         // SKIPTHR bit 7: max-first decode
    reg  [31:0]       vskips;         // V reads saved
    reg               rope_en, rope_abs;  // ROPE
    reg  [7:0]        ropepos;        // ROPEPOS
    localparam        LPG = 2;                    // 4 tokens per page
    localparam        NPG = NK >> LPG;            // pages in the cache
    reg               paged;                      // PAGED bit 0
    reg               mixed;                      // PAGED bit 1: per-page precision
    reg  [NPG-1:0]    pfmt;                       // PFMT: page is MXINT4
    reg  [LK-LPG-1:0] ptab [0:NPG-1];             // PTAB
    wire              pg_on = paged && decode;
    wire              mx_on = pg_on && mixed;
    // Mixed precision (feature 14): the cache is addressed in INT4 slots, two
    // per SRAM word. An INT4 page puts token i at slot 4*PTAB + i, an INT8 page
    // at slot 4*PTAB + 2i (it fills two slot pages; PTAB must be even). The
    // word is slot >> 1, the nibble half slot & 1, and e_k / e_v sit at the slot.
    // NOTE: all lookups below are written as plain expressions (not functions
    // reading module state), so event-driven simulation re-evaluates them
    // whenever PTAB, PFMT, PAGED or KVFMT change.
    reg [LN-1:0]      qrow;
    reg [31:0]        cycles, skips;
    assign busy = busy_r;
    assign done = done_r;
    wire mf = (mode != 2'd0);
    wire [LP-1:0] qlane = qrow % P;
    // one-hot decode lane, registered in three copies (qrow only changes while
    // the core is idle): the lane muxes below are AND-OR trees on a local
    // copy instead of a die-wide qlane broadcast
    wire [P-1:0]   qsel_d = {{(P-1){1'b0}}, 1'b1} << qlane;
    wire [3*P-1:0] qsel_bus;
    raijin_dupreg #(.W(P), .C(3)) u_qsel (.clk(clk), .d(qsel_d), .q(qsel_bus));
    wire [P-1:0]   qsel_s = qsel_bus[0 +: P];     // score capture
    wire [P-1:0]   qsel_m = qsel_bus[P +: P];     // final m
    wire [P-1:0]   qsel_a = qsel_bus[2*P +: P];   // accumulate lane enable
    reg  signed [11:0] score_q;                   // score of the query lane
    reg  signed [8:0]  m_q;                       // running m of the query lane
    wire          pipe  = fused && !decode;   // group-pipelined fused pass
    localparam    L_MIN = 14 + P;             // min cycles between two groups' last keys
    reg  [4:0]    since_last;                 // cycles since the last group's last key

    // ------------------------------------------------------------------
    // Bus decode
    // ------------------------------------------------------------------
    // Token window (region 6, feature 13): one burst per decode token.
    //   write 0x6000 + b:  b 0..15 Q[QROW], 16..31 K[tok], 32..47 V[tok],
    //                      48 e_q[QROW], 49 e_k[tok], 50 e_v[tok],
    //                      51 CTRL: LEN <= LEN + 1, then start with this value
    //   read  0x6000 + b:  b 0..15 O[QROW], 16 e_o[QROW], 17 HHMIN (eviction slot)
    // tok is the next cache slot, LEN (through PTAB when PAGED is set).
    // INT8 K/V only. Each byte is translated to its ordinary address, so RoPE,
    // paging and the busy rules apply unchanged.
    wire          twin    = (addr_r[15:12] == 4'h6);
    wire [5:0]    tb      = addr_r[5:0];
    wire [LK-1:0] ttok_l  = len[LK-1:0];
    wire [LK-1:0] ttok    = paged ? {ptab[ttok_l[LK-1:LPG]], ttok_l[LPG-1:0]} : ttok_l;
    wire [LN-1:0] ttrow   = ttok[LN-1:0];
    reg  [15:0]   baddr;
    always @(*) begin
        baddr = addr_r;
        if (twin) begin
            if (re_r && !we_r)
                baddr = (tb < 6'd16)  ? {4'h3, {(12-LN-LD){1'b0}}, qrow, tb[LD-1:0]} :
                        (tb == 6'd17) ? 16'h7081 :
                                        {8'h43, {(8-LN){1'b0}}, qrow};
            else if (tb < 6'd16) baddr = {4'h0, {(12-LN-LD){1'b0}}, qrow,  tb[LD-1:0]};
            else if (tb < 6'd32) baddr = {4'h1, {(12-LN-LD){1'b0}}, ttrow, tb[LD-1:0]};
            else if (tb < 6'd48) baddr = {4'h2, {(12-LN-LD){1'b0}}, ttrow, tb[LD-1:0]};
            else if (tb == 6'd48) baddr = {8'h40, {(8-LN){1'b0}}, qrow};
            else if (tb == 6'd49) baddr = {8'h41, {(8-LK){1'b0}}, ttok};
            else if (tb == 6'd50) baddr = {8'h42, {(8-LK){1'b0}}, ttok};
            else                  baddr = 16'h5000;
        end
    end
    wire          tw_go  = we_r && twin && tb == 6'd51 && !busy_r;   // LEN + 1 and start
    wire [3:0]    region = baddr[15:12];
    wire [11:0]   boff   = baddr[11:0];
    wire [LN-1:0] brow   = boff[LD +: LN];
    wire [LD-1:0] bcol   = boff[LD-1:0];
    wire [1:0]    bbank  = bcol[3:2];
    wire [1:0]    bbyte  = bcol[1:0];
    wire [LK-1:0] erow   = boff[LK-1:0];
    wire [1:0]    esel   = boff[9:8];
    wire          creg   = (region == 4'h5);
    wire [4:0]    cidx   = boff[4:0];
    wire          start_w    = we_r && creg && cidx == 5'h00 && wdata_r[0] && !busy_r;
    wire          bus_mem_we = we_r && !busy_r && region < 4'h3;
    wire          bus_mem_re = re_r && !busy_r && region == 4'h3;
    // RoPE on write: pairs of Q / K bytes are rotated before they reach SRAM
    wire          rope_hit   = bus_mem_we && rope_en && (region == 4'h0 || (region == 4'h1 && !k4));
    wire [7:0]    rope_xo, rope_yo;
    // Token-window K writes take their row from LEN (through PTAB). For timing,
    // RoPE sees a registered copy of that row: LEN and PTAB only change on
    // earlier bus writes or at a step's start, never in the cycle before a
    // K byte, so the copy always holds the current row.
    reg  [LN-1:0] ttrow_r;
    always @(posedge clk) ttrow_r <= ttrow;
    wire [LN-1:0] rope_row = !twin ? addr_r[LD +: LN] : (tb < 6'd16) ? qrow : ttrow_r;
    raijin_rope u_rope (.clk(clk), .we_even(rope_hit && !bcol[0]), .pair(bcol[3:1]),
        .pos(rope_abs ? ropepos : {{(8-LN){1'b0}}, rope_row}), .wdata(wdata_r),
        .xo(rope_xo), .yo(rope_yo));

    always @(posedge clk) begin
        if (we_r && !busy_r && region == 4'h4) begin
            case (esel)
                2'd0: if (erow < N) eqm[erow[LN-1:0]] <= wdata_r;
                2'd1: ekm[erow] <= wdata_r;
                2'd2: evm[erow] <= wdata_r;
                default: ;
            endcase
        end
    end

    // ------------------------------------------------------------------
    // Controller state (declared early, the SRAM ports depend on it)
    // ------------------------------------------------------------------
    localparam [4:0]
        S_IDLE     = 5'd0,
        Q_LOAD     = 5'd1,
        U_QK       = 5'd2,
        U_QK_DRAIN = 5'd3,
        U_SM_CLR   = 5'd4,
        U_SM_P1    = 5'd5,
        U_SM_P2    = 5'd6,
        U_SM_DRAIN = 5'd7,
        RCP_START  = 5'd8,
        RCP_WAIT   = 5'd9,
        U_PV_CLR   = 5'd10,
        U_PV       = 5'd11,
        U_PV_DRAIN = 5'd12,
        OUT_ISSUE  = 5'd13,
        OUT_DRAIN  = 5'd14,
        F_CLR      = 5'd15,
        F_STREAM   = 5'd16,
        F_DRAIN    = 5'd17,
        S_DONE     = 5'd18,
        OUT_WAIT   = 5'd19,
        P_WAIT     = 5'd20,   // pipelined fused: wait for the last O writes
        MF_WAIT    = 5'd21,   // max-first decode: pass A drains, m is final
        MF_MASK    = 5'd22,   //   survivor mask from the latched scores
        MF_B       = 5'd23;   //   pass B: one surviving key per cycle

    reg [4:0]    state;
    reg [LG-1:0] g;
    reg [LK-1:0] j;
    reg [LP-1:0] orow;
    reg [LP-1:0] qcnt;
    reg [3:0]    dcnt;
    wire [TW-1:0] issue_tag = {g, j};

    // ------------------------------------------------------------------
    // SRAM banks
    // ------------------------------------------------------------------
    reg  [NB-1:0]    p0_csb, p0_web, p1_csb;
    reg  [NB*4-1:0]  p0_mask;
    reg  [7:0]       p0_addr, p1_addr;
    reg  [NB*32-1:0] p0_din;
    wire [NB*32-1:0] p0_dout, p1_dout;
    reg  [NB*32-1:0] d0_r, d1_r;          // registered read data

    `RAIJIN_RAM sram0 (.clk0(clk), .csb0(p0_csb[0]), .web0(p0_web[0]), .wmask0(p0_mask[3:0]),
        .addr0(p0_addr), .din0(p0_din[31:0]), .dout0(p0_dout[31:0]),
        .clk1(clk), .csb1(p1_csb[0]), .addr1(p1_addr), .dout1(p1_dout[31:0]));
    `RAIJIN_RAM sram1 (.clk0(clk), .csb0(p0_csb[1]), .web0(p0_web[1]), .wmask0(p0_mask[7:4]),
        .addr0(p0_addr), .din0(p0_din[63:32]), .dout0(p0_dout[63:32]),
        .clk1(clk), .csb1(p1_csb[1]), .addr1(p1_addr), .dout1(p1_dout[63:32]));
    `RAIJIN_RAM sram2 (.clk0(clk), .csb0(p0_csb[2]), .web0(p0_web[2]), .wmask0(p0_mask[11:8]),
        .addr0(p0_addr), .din0(p0_din[95:64]), .dout0(p0_dout[95:64]),
        .clk1(clk), .csb1(p1_csb[2]), .addr1(p1_addr), .dout1(p1_dout[95:64]));
    `RAIJIN_RAM sram3 (.clk0(clk), .csb0(p0_csb[3]), .web0(p0_web[3]), .wmask0(p0_mask[15:12]),
        .addr0(p0_addr), .din0(p0_din[127:96]), .dout0(p0_dout[127:96]),
        .clk1(clk), .csb1(p1_csb[3]), .addr1(p1_addr), .dout1(p1_dout[127:96]));

    always @(posedge clk) begin
        d0_r <= p0_dout;
        d1_r <= p1_dout;
    end

    // ---- port 1 (read only): K rows, Q rows, score buffer reads ----
    wire fstall;
    wire k_rd = (state == U_QK) || ((state == F_STREAM) && !fstall);
    wire q_rd = (state == Q_LOAD);
    wire s_rd = (state == U_SM_P1) || (state == U_SM_P2) || (state == U_PV);
    // logical -> physical slot of the key being read (feature 10, 14)
    wire          j_is4 = pfmt[j[LK-1:LPG]];
    wire [LK-1:0] pj    = mx_on ? ({ptab[j[LK-1:LPG]], {LPG{1'b0}}} + (j_is4 ? j[LPG-1:0] : {j[LPG-1:0], 1'b0}))
                        : pg_on ? {ptab[j[LK-1:LPG]], j[LPG-1:0]} : j;
    wire          k4_j  = mx_on ? j_is4 : k4;
    integer pb;
    always @(*) begin
        p1_csb  = {NB{1'b1}};
        p1_addr = 8'd0;
        if (k_rd) begin
            p1_csb  = {NB{1'b0}};
            p1_addr = AK + ((k4_j || mx_on) ? (pj >> 1) : pj);
        end else if (q_rd) begin
            p1_csb  = {NB{1'b0}};
            p1_addr = AQ + g * P + qcnt;
        end else if (s_rd) begin
            for (pb = 0; pb < NSB; pb = pb + 1) p1_csb[pb] = 1'b0;
            p1_addr = AS + {g, j[LN-1:0]};
        end
    end

    // read-side pipeline: a port-1 read in cycle t is used in cycle t + 2
    reg          k_d1, k_d2, q_d1, q_d2, b_d1, b_d2, pv_d1, pv_d2;
    reg          mu_d1, mu_d2, em_d1, em_d2;
    reg [TW-1:0] tag_d1, tag_d2;
    reg [LP-1:0] qc_d1, qc_d2;
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            {k_d1, k_d2, q_d1, q_d2, b_d1, b_d2, pv_d1, pv_d2} <= 8'd0;
        end else begin
            k_d1  <= k_rd;                    k_d2  <= k_d1;
            q_d1  <= q_rd;                    q_d2  <= q_d1;
            b_d1  <= (state == U_SM_P1) || (state == U_SM_P2);
            b_d2  <= b_d1;
            pv_d1 <= (state == U_PV);         pv_d2 <= pv_d1;
        end
    end
    always @(posedge clk) begin
        tag_d1 <= issue_tag;           tag_d2 <= tag_d1;
        qc_d1  <= qcnt;                qc_d2  <= qc_d1;
        mu_d1  <= (state == U_SM_P1);  mu_d2  <= mu_d1;
        em_d1  <= (state == U_SM_P2);  em_d2  <= em_d1;
    end
    wire [LG-1:0] g_d2 = tag_d2[LK +: LG];
    wire [LK-1:0] j_d2 = tag_d2[LK-1:0];
    wire          jd2_is4 = pfmt[j_d2[LK-1:LPG]];
    wire [LK-1:0] pj_d2 = mx_on ? ({ptab[j_d2[LK-1:LPG]], {LPG{1'b0}}} + (jd2_is4 ? j_d2[LPG-1:0] : {j_d2[LPG-1:0], 1'b0}))
                        : pg_on ? {ptab[j_d2[LK-1:LPG]], j_d2[LPG-1:0]} : j_d2;   // K row in d1_r
    wire          k4_d2 = mx_on ? jd2_is4 : k4;

    // MXINT4 K/V: pick token t's half of the SRAM word and widen each nibble
    // to INT8 as x4 << 4 (the host's e_k / e_v account for the 4-bit shift).
    // MXFP4 (fp = 1): the nibble is E2M1 and widens to +-(2|x|) << 3, the same
    // 16 x scale, so the rest of the datapath is shared.
    function [7:0] fp4to8;
        input [3:0] n;
        reg   [7:0] mag;
        begin
            case (n[2:0])
                3'd0: mag = 8'd0;   3'd1: mag = 8'd8;
                3'd2: mag = 8'd16;  3'd3: mag = 8'd24;
                3'd4: mag = 8'd32;  3'd5: mag = 8'd48;
                3'd6: mag = 8'd64;  default: mag = 8'd96;
            endcase
            fp4to8 = n[3] ? (~mag + 8'd1) : mag;
        end
    endfunction
    function [D*8-1:0] unpack4;
        input [NB*32-1:0] w;
        input             hi;
        input             fp;
        integer c;
        begin
            for (c = 0; c < D; c = c + 1)
                unpack4[c*8 +: 8] = fp ? fp4to8(w[(hi ? 64 : 0) + c*4 +: 4])
                                       : {w[(hi ? 64 : 0) + c*4 +: 4], 4'd0};
        end
    endfunction
    wire [D*8-1:0] k_row = k4_d2 ? unpack4(d1_r, pj_d2[0], kfp) : d1_r;

    integer qi;
    always @(posedge clk) begin
        if (q_d2)
            for (qi = 0; qi < P; qi = qi + 1)
                if (qc_d2 == qi) qreg[qi] <= d1_r;
    end

    // score buffer read data, lane r
    wire [P*12-1:0] sb_rd;
    genvar gr;
    generate
        for (gr = 0; gr < P; gr = gr + 1) begin : g_sbrd
            assign sb_rd[gr*12 +: 12] = d1_r[(gr/2)*32 + (gr%2)*12 +: 12];
        end
    endgenerate

    // ------------------------------------------------------------------
    // V exponent handling: e_vmax and per-row shift dv = e_vmax - e_v[j].
    // In decode mode only the LEN cached rows count towards the maximum.
    // ------------------------------------------------------------------
    reg [7:0] emax_c, emax;
    reg [4:0] dvr [0:NK-1];
    integer ei, ew, di;
    reg [NK*8-1:0] mt;    // pairwise max tree over every cache row
    reg [NK-1:0]   pvalid;  // paged decode: physical tokens holding logical j < LEN
    integer pl, pt;
    always @(*) begin
        pvalid = {NK{1'b0}};
        for (pl = 0; pl < NPG; pl = pl + 1)
            for (pt = 0; pt < (1 << LPG); pt = pt + 1)
                if (pl * (1 << LPG) + pt < len)
                    pvalid[ptab[pl] * (1 << LPG) + ((mixed && !pfmt[pl]) ? 2 * pt : pt)] = 1'b1;
        for (ei = 0; ei < NK; ei = ei + 1)
            mt[ei*8 +: 8] = (decode ? (pg_on ? pvalid[ei] : (ei < len)) : (ei < N)) ? evm[ei] : 8'd0;
        for (ew = NK / 2; ew >= 1; ew = ew / 2)
            for (ei = 0; ei < ew; ei = ei + 1)
                mt[ei*8 +: 8] = (mt[(2*ei)*8 +: 8] > mt[(2*ei+1)*8 +: 8]) ?
                                mt[(2*ei)*8 +: 8] : mt[(2*ei+1)*8 +: 8];
        emax_c = mt[7:0];
    end
    reg [7:0] dtmp;
    always @(posedge clk) begin
        emax <= emax_c;
        for (di = 0; di < NK; di = di + 1) begin
            dtmp    = emax - evm[di];
            dvr[di] <= (dtmp > 8'd31) ? 5'd31 : dtmp[4:0];
        end
    end

    // ------------------------------------------------------------------
    // DPU array: P DPUs, Q rows from qreg, K row straight from the banks
    // ------------------------------------------------------------------
    wire [P-1:0]        dpu_valid;
    wire [P*TW-1:0]     dpu_tag;
    wire [P*20-1:0]     dpu_dot;
    wire [P*9-1:0]      dpu_esum;
    wire [P*12-1:0]     score;

    generate
        for (gr = 0; gr < P; gr = gr + 1) begin : g_dpu
            wire [LN-1:0] qr = g_d2 * P + gr;
            raijin_dpu #(.D(D), .TW(TW)) u_dpu (
                .clk(clk), .rst_n(rst_n),
                .in_valid(k_d2), .in_tag(tag_d2),
                .a(qreg[gr]), .b(k_row), .ea(eqm[qr]), .eb(ekm[pj_d2]),
                .out_valid(dpu_valid[gr]), .out_tag(dpu_tag[gr*TW +: TW]),
                .dot(dpu_dot[gr*20 +: 20]), .esum(dpu_esum[gr*9 +: 9])
            );
            raijin_requant u_rq (
                .dot(dpu_dot[gr*20 +: 20]), .esum(dpu_esum[gr*9 +: 9]),
                .offset(offset), .s(score[gr*12 +: 12])
            );
        end
    endgenerate

    // ------------------------------------------------------------------
    // Softmax unit: P lanes
    // ------------------------------------------------------------------
    wire sm_clear   = (state == U_SM_CLR) || (state == F_CLR);
    // max-first decode: pass A = batch pass 1 (max only) on the DPU scores,
    // pass B = batch pass 2 (exp and sum against the final m) on latched scores
    reg               mf_a;                  // pass A of a max-first step
    reg               mb_v;                  // pass B issue (registered)
    reg  [LK-1:0]     mb_j;
    reg signed [11:0] mb_s;
    wire mfd        = decode && !multi && mfirst;
    wire sm_valid   = fused ? (dpu_valid[0] || mb_v) : b_d2;
    wire sm_mupd    = fused ? !mb_v : mu_d2;
    wire sm_emit    = fused ? !(mfd && mf_a) : em_d2;
    wire [TW-1:0] sm_tag_in = !fused ? tag_d2 : mb_v ? {g, mb_j} : dpu_tag[0 +: TW];
    wire rcp_go     = (state == RCP_START);

    wire [P-1:0]    sm_ovalid;
    wire [P*TW-1:0] sm_otag;
    wire [P*9-1:0]  sm_p;
    wire [P*4-1:0]  sm_sa, sm_sb;
    wire [P*5-1:0]  sm_delta;
    wire [P-1:0]    sm_rvalid;
    wire [P*13-1:0] sm_r;
    wire [P-1:0]    sm_ofirst, sm_olast;
    wire            sm_first = pipe && (sm_tag_in[LK-1:0] == {LK{1'b0}});
    wire [LG-1:0]   sm_g     = sm_tag_in[LK +: LG];
    wire [LK-1:0]   sm_j     = sm_tag_in[LK-1:0];
    wire            sm_last  = pipe && (causal ? (sm_j == sm_g * P + P - 1) : (sm_j == N - 1));
    wire [P-1:0]    sm_omask;
    wire [P*9-1:0]  sm_m;

    generate
        for (gr = 0; gr < P; gr = gr + 1) begin : g_sm
            wire signed [11:0] s_in = !fused ? sb_rd[gr*12 +: 12] : mb_v ? mb_s : score[gr*12 +: 12];
            wire [13:0]       l_unused;
            raijin_softmax_lane #(.TW(TW)) u_lane (
                .clk(clk), .rst_n(rst_n), .clear(sm_clear), .mode(mode),
                .s_valid(sm_valid), .s(s_in), .tag(sm_tag_in),
                .m_update(sm_mupd), .emit(sm_emit),
                .first(sm_first), .last(sm_last),
                .mask((causal && (sm_j > sm_g * P + gr)) ||
                      (spec && ({1'b0, sm_j} + P > len + gr))),
                .thr(thr),
                .out_valid(sm_ovalid[gr]), .out_tag(sm_otag[gr*TW +: TW]),
                .out_p(sm_p[gr*9 +: 9]), .out_sa(sm_sa[gr*4 +: 4]), .out_sb(sm_sb[gr*4 +: 4]),
                .out_delta(sm_delta[gr*5 +: 5]),
                .out_first(sm_ofirst[gr]), .out_last(sm_olast[gr]), .out_mask(sm_omask[gr]),
                .m(sm_m[gr*9 +: 9]), .l(l_unused),
                .recip_start(rcp_go), .r_valid(sm_rvalid[gr]), .r(sm_r[gr*13 +: 13])
            );
        end
    endgenerate

    integer qi2;
    always @(*) begin
        score_q = 12'sd0;
        m_q     = 9'sd0;
        for (qi2 = 0; qi2 < P; qi2 = qi2 + 1) begin
            score_q = score_q | ({12{qsel_s[qi2]}} & score[qi2*12 +: 12]);
            m_q     = m_q     | ({9{qsel_m[qi2]}} & sm_m[qi2*9 +: 9]);
        end
    end

    // ---- heavy-hitter tracker (feature 15, region 7) ----
    // physical slot of the key now entering the lanes (same mapping as pj)
    wire          sj_is4 = pfmt[sm_j[LK-1:LPG]];
    wire [LK-1:0] pj_s   = mx_on ? ({ptab[sm_j[LK-1:LPG]], {LPG{1'b0}}} + (sj_is4 ? sm_j[LPG-1:0] : {sm_j[LPG-1:0], 1'b0}))
                         : pg_on ? {ptab[sm_j[LK-1:LPG]], sm_j[LPG-1:0]} : sm_j;
    wire          hh_busy, hh_en;
    wire [LK-1:0] hh_recent;
    wire [7:0]    hh_rd;
    raijin_hh #(.NK(NK), .LK(LK)) u_hh (
        .clk(clk), .rst_n(rst_n),
        .cap_valid(decode && !multi && dpu_valid[0]), .cap_slot(pj_s),
        .cap_s(score_q),
        .cap_recent({1'b0, sm_j} + hh_recent >= len), .cap_clear(state == F_CLR && decode),
        .go(state == RCP_START && decode && !multi), .m_fin(m_q),
        .busy(hh_busy),
        .bus_we(we_r && !busy_r && region == 4'h7), .bus_off(boff[7:0]),
        .bus_wdata(wdata_r), .bus_rdata(hh_rd), .en(hh_en), .recent(hh_recent));

    // ---- max-first decode (feature 17): scores of pass A by logical key ----
    reg  signed [11:0] mf_sc [0:NK-1];
    reg  [NK-1:0]      mf_mask;               // keys pass B still has to visit
    always @(posedge clk) begin
        if (mfd && mf_a && dpu_valid[0]) mf_sc[dpu_tag[LK-1:0]] <= score_q;
    end
    // a key survives if its weight against the final m can be nonzero:
    // fewer than 9 (or SKIPTHR) powers of two below m (the lanes' own rule)
    reg  signed [8:0] mf_m;                  // final m, registered for the mask
    always @(posedge clk) mf_m <= m_q;
    wire        [3:0] mf_lim = (thr != 4'd0 && thr < 4'd9) ? thr : 4'd9;
    reg  [NK-1:0]      mf_keep;
    reg  [LK:0]        mf_drop;               // keys dropped by the latched mask
    reg                mf_cnt;                // first MF_B cycle: mask is complete
    reg                mf_cnt_d;              // the counters add mf_drop_r one cycle later
    reg  [LK:0]        mf_drop_r;
    reg  signed [14:0] mf_t;
    reg  [LK-1:0]      mf_first;              // lowest set bit of mf_mask
    integer mi;
    always @(*) begin
        mf_drop = 0;
        for (mi = 0; mi < NK; mi = mi + 1) begin
            mf_t = ($signed({{2{mf_m[8]}}, mf_m}) <<< 4) - $signed({{3{mf_sc[mi][11]}}, mf_sc[mi]});
            mf_keep[mi] = (mi < len) && (mf_t[14:4] < {7'd0, mf_lim});
            if (mi < len && !mf_mask[mi]) mf_drop = mf_drop + 1'b1;
        end
        mf_first = {LK{1'b0}};
        for (mi = NK - 1; mi >= 0; mi = mi - 1)
            if (mf_mask[mi]) mf_first = mi;
    end
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            mb_v    <= 1'b0;
            mf_cnt  <= 1'b0;
            mf_cnt_d <= 1'b0;
            mf_mask <= {NK{1'b0}};
        end else begin
            mb_v   <= (state == MF_B) && (|mf_mask);
            mf_cnt <= (state == MF_MASK);
            mf_cnt_d <= mf_cnt;
            if (state == MF_MASK) mf_mask <= mf_keep;
            else if (state == MF_B && (|mf_mask)) mf_mask[mf_first] <= 1'b0;
        end
    end
    always @(posedge clk) begin
        if (mf_cnt) mf_drop_r <= mf_drop;
        mb_j <= mf_first;
        mb_s <= mf_sc[mf_first];
    end

    // score buffer writes (unfused): scores from the DPUs, then p codes
    wire            sb_we   = !fused && (dpu_valid[0] || sm_ovalid[0]);
    wire [TW-1:0]   sb_wtag_f = dpu_valid[0] ? dpu_tag[0 +: TW] : sm_otag[0 +: TW];
    wire [LG+LN-1:0] sb_wtag = {sb_wtag_f[LK +: LG], sb_wtag_f[LN-1:0]};   // {g, j}
    reg  [NB*32-1:0] sb_wdata;
    integer wi;
    always @(*) begin
        sb_wdata = {NB*32{1'b0}};
        for (wi = 0; wi < P; wi = wi + 1)
            sb_wdata[(wi/2)*32 + (wi%2)*12 +: 12] = dpu_valid[0] ? score[wi*12 +: 12] :
                (mf ? {4'd0, sm_sb[wi*4 +: 4], sm_sa[wi*4 +: 4]} : {3'd0, sm_p[wi*9 +: 9]});
    end

    function [8:0] term;
        input [3:0] sh;
        begin
            term = (sh > 4'd8) ? 9'd0 : (9'd256 >> sh);
        end
    endfunction

    // ------------------------------------------------------------------
    // Accumulate inputs. Stage c picks the source and issues the V read;
    // two register stages later the V row arrives and the array updates.
    // ------------------------------------------------------------------
    reg            acc_valid;
    reg [LK-1:0]   acc_j;
    reg [P*9-1:0]  acc_p;
    reg [P*4-1:0]  acc_sa, acc_sb;
    reg [P*5-1:0]  acc_delta;
    reg [P-1:0]    acc_en, acc_lane;
    integer ai;
    always @(*) begin
        if (fused) begin
            acc_valid = sm_ovalid[0];
            acc_j     = sm_otag[0 +: LK];
            acc_p     = sm_p;
            acc_sa    = sm_sa;
            acc_sb    = sm_sb;
            acc_delta = sm_delta;
        end else begin
            acc_valid = pv_d2;
            acc_j     = j_d2;
            acc_delta = {P*5{1'b0}};
            for (ai = 0; ai < P; ai = ai + 1) begin
                if (mf) begin
                    acc_sa[ai*4 +: 4] = sb_rd[ai*12 +: 4];
                    acc_sb[ai*4 +: 4] = sb_rd[ai*12 + 4 +: 4];
                    acc_p[ai*9 +: 9]  = term(sb_rd[ai*12 +: 4]) + term(sb_rd[ai*12 + 4 +: 4]);
                end else begin
                    acc_sa[ai*4 +: 4] = 4'd15;
                    acc_sb[ai*4 +: 4] = 4'd15;
                    acc_p[ai*9 +: 9]  = sb_rd[ai*12 +: 9];
                end
            end
        end
        for (ai = 0; ai < P; ai = ai + 1) begin
            acc_lane[ai] = (!decode || multi || qsel_a[ai]) && !(fused && sm_omask[ai]);
            acc_en[ai]   = acc_valid && acc_lane[ai] && (acc_p[ai*9 +: 9] != 9'd0);
        end
    end
    // a V row is read only if some lane will use it (key skip saves the read)
    wire v_rd    = acc_valid && (|acc_en);
    // physical slot and format of the V row being accumulated
    wire          acc_is4 = pfmt[acc_j[LK-1:LPG]];
    wire [LK-1:0] pacc    = mx_on ? ({ptab[acc_j[LK-1:LPG]], {LPG{1'b0}}} + (acc_is4 ? acc_j[LPG-1:0] : {acc_j[LPG-1:0], 1'b0}))
                          : pg_on ? {ptab[acc_j[LK-1:LPG]], acc_j[LPG-1:0]} : acc_j;
    wire          v4_acc  = mx_on ? acc_is4 : v4;
    wire v_saved = acc_valid && !(|acc_en);

    reg            av1, av2;
    reg [LK-1:0]   aj1, aj2;
    reg            av41, av42;
    reg [P*9-1:0]  ap1, ap2;
    reg [P*4-1:0]  asa1, asa2, asb1, asb2;
    reg [P*5-1:0]  ad1, ad2;
    reg [P-1:0]    aen1, aen2, aln1, aln2;
    // group pipelining: first / last key of a group, and its group index,
    // carried alongside the accumulate pipeline
    reg            af1, af2, al1, al2, al3, al4;
    reg [LG-1:0]   ag1, ag2, ag3, ag4;
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            av1 <= 1'b0; av2 <= 1'b0; aen1 <= {P{1'b0}}; aen2 <= {P{1'b0}};
            {af1, af2, al1, al2, al3, al4} <= 6'd0;
        end else begin
            av1 <= acc_valid; av2 <= av1;
            aen1 <= acc_en;   aen2 <= aen1;
            af1 <= pipe && sm_ofirst[0];  af2 <= af1;
            al1 <= pipe && sm_olast[0];   al2 <= al1;  al3 <= al2;  al4 <= al3;
        end
    end
    always @(posedge clk) begin
        ag1 <= sm_otag[LK +: LG];  ag2 <= ag1;  ag3 <= ag2;  ag4 <= ag3;
    end
    always @(posedge clk) begin
        aj1 <= pacc;       aj2 <= aj1;   // physical token: V nibble half, dv
        av41 <= v4_acc;    av42 <= av41;
        ap1 <= acc_p;      ap2 <= ap1;
        asa1 <= acc_sa;    asa2 <= asa1;
        asb1 <= acc_sb;    asb2 <= asb1;
        ad1 <= acc_delta;  ad2 <= ad1;
        aln1 <= acc_lane;  aln2 <= aln1;
    end

    wire acc_clear = (state == U_PV_CLR) || (state == F_CLR);
    wire [P*D*24-1:0] acc;
    raijin_acc_array #(.R(P), .D(D)) u_acc (
        .clk(clk), .rst_n(rst_n), .clear(acc_clear), .mf(mf),
        .v(av42 ? unpack4(d0_r, aj2[0], vfp) : d0_r), .dv(dvr[aj2]),
        .en(aen2), .first(af2), .p(ap2), .sa(asa2), .sb(asb2), .delta(ad2),
        .acc(acc)
    );

    // number of rows skipped this cycle
    reg [LP:0] nskip;
    integer ki;
    always @(*) begin
        nskip = 0;
        for (ki = 0; ki < P; ki = ki + 1)
            if (av2 && aln2[ki] && !aen2[ki]) nskip = nskip + 1'b1;
    end

    // ------------------------------------------------------------------
    // Output stage
    // ------------------------------------------------------------------
    wire              out_issue = (state == OUT_ISSUE);
    wire [LN-1:0]     out_row_i = g * P + orow;

    // pipelined fused pass: the finished group's accumulator rows are copied
    // the cycle after its last update lands (al4), then issued to the output
    // stage once that group's reciprocals are ready
    reg  [P*D*24-1:0] acc_snap;
    reg  [LG-1:0]     snap_g;
    reg               pissue;
    reg  [LP-1:0]     pcnt;
    reg               rv_q;              // r_valid is a level: use its rising edge
    always @(posedge clk or negedge rst_n)
        if (!rst_n) rv_q <= 1'b0;
        else        rv_q <= sm_rvalid[0];
    wire              rcp_done = sm_rvalid[0] && !rv_q;
    always @(posedge clk) begin
        if (al4) begin
            acc_snap <= acc;
            snap_g   <= ag4;
        end
    end
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            pissue <= 1'b0;
            pcnt   <= {LP{1'b0}};
        end else if (pipe && rcp_done) begin
            pissue <= 1'b1;
            pcnt   <= {LP{1'b0}};
        end else if (pissue) begin
            pcnt <= pcnt + 1'b1;
            if (pcnt == P - 1) pissue <= 1'b0;
        end
    end
    wire [LN-1:0] pout_row = snap_g * P + pcnt;
    wire              o_valid;
    wire [LN-1:0]     o_tag;
    wire [D*8-1:0]    o_row;
    wire [7:0]        o_e;
    // stage 0: register the selected accumulator row and its 1/l, so the row
    // select does not sit in front of the 24 x 13 multipliers
    reg              o0_v;
    reg [LN-1:0]     o0_tag;
    reg [D*24-1:0]   o0_acc;
    // 1/l is held in OUT_RG copies, one per group of output lanes (see raijin_out)
    localparam OUT_RG = 4;
    wire [OUT_RG*13-1:0]  o0_r_bus;
    wire [12:0] o0_r_d = pissue ? sm_r[pcnt*13 +: 13] : rinv[out_row_i];
    raijin_dupreg #(.W(13), .C(OUT_RG)) u_o0r (.clk(clk), .d(o0_r_d), .q(o0_r_bus));
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) o0_v <= 1'b0;
        else        o0_v <= out_issue || pissue;
    end
    always @(posedge clk) begin
        o0_tag <= pissue ? pout_row : out_row_i;
        o0_acc <= pissue ? acc_snap[pcnt*D*24 +: D*24] : acc[orow*D*24 +: D*24];
    end
    raijin_out #(.D(D), .TW(LN), .RG(OUT_RG)) u_out (
        .clk(clk), .rst_n(rst_n),
        .in_valid(o0_v), .in_tag(o0_tag),
        .acc(o0_acc), .r(o0_r_bus), .emax(emax),
        .out_valid(o_valid), .out_tag(o_tag), .o(o_row), .eo(o_e)
    );
    always @(posedge clk) begin
        if (o_valid) eom[o_tag] <= o_e;
    end

    // O write queue (pipelined pass): rows wait for a port-0 cycle with no V read
    localparam OQ = 2 * P;
    localparam LQ = $clog2(OQ);
    reg  [D*8-1:0] oq_row [0:OQ-1];
    reg  [LN-1:0]  oq_tag [0:OQ-1];
    reg  [LQ-1:0]  oq_wp, oq_rp;
    reg  [LQ:0]    oq_n;
    reg  [LN:0]    owr_cnt;          // O rows written this pass
    // a row goes straight to the SRAM when the queue is empty and port 0 is free
    wire           oq_free = pipe && !v_rd;
    wire           oq_byp  = oq_free && o_valid && (oq_n == 0);
    wire           oq_deq  = oq_free && (oq_n != 0);
    wire           oq_pop  = oq_byp || oq_deq;          // an O write this cycle
    wire           oq_push = pipe && o_valid && !oq_byp;
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            oq_wp <= 0; oq_rp <= 0; oq_n <= 0;
        end else begin
            if (oq_push) oq_wp <= oq_wp + 1'b1;
            if (oq_deq)  oq_rp <= oq_rp + 1'b1;
            oq_n <= oq_n + oq_push - oq_deq;
        end
    end
    always @(posedge clk) begin
        if (oq_push) begin
            oq_row[oq_wp] <= o_row;
            oq_tag[oq_wp] <= o_tag;
        end
    end
`ifdef RAIJIN_TRACE
    always @(posedge clk)
        if (pipe && busy_r)
            $display("TR c=%0d st=%0d g=%0d j=%0d smv=%b smtag=%h olast=%b al4=%b rv=%b pis=%b pc=%0d o0v=%b ov=%b otag=%0d vrd=%b oqn=%0d pop=%b owr=%0d",
                cycles, state, g, j, sm_valid, sm_tag_in, sm_olast[0], al4, sm_rvalid[0], pissue, pcnt, o0_v, o_valid, o_tag, v_rd, oq_n, oq_pop, owr_cnt);
`endif
`ifndef SYNTHESIS
    always @(posedge clk)
        if (rst_n && oq_push && !oq_deq && oq_n == OQ)
            $display("%t raijin_core ERROR: O write queue overflow", $time);
`endif

    // ---- port 0 (read/write): O writes, V reads, score writes, bus ----
    integer mb;
    always @(*) begin
        p0_csb  = {NB{1'b1}};
        p0_web  = {NB{1'b1}};
        p0_mask = {NB*4{1'b0}};
        p0_addr = 8'd0;
        p0_din  = {NB*32{1'b0}};
        if (pipe ? oq_pop : o_valid) begin
            p0_csb  = {NB{1'b0}};
            p0_web  = {NB{1'b0}};
            p0_mask = {NB*4{1'b1}};
            p0_addr = AO + ((pipe && !oq_byp) ? oq_tag[oq_rp] : o_tag);
            p0_din  = (pipe && !oq_byp) ? oq_row[oq_rp] : o_row;
        end else if (v_rd) begin
            p0_csb  = {NB{1'b0}};
            p0_addr = AV + ((v4_acc || mx_on) ? (pacc >> 1) : pacc);
        end else if (sb_we) begin
            for (mb = 0; mb < NSB; mb = mb + 1) begin
                p0_csb[mb] = 1'b0;
                p0_web[mb] = 1'b0;
                p0_mask[mb*4 +: 4] = 4'hF;
            end
            p0_addr = AS + sb_wtag;
            p0_din  = sb_wdata;
        end else if (rope_hit) begin
            // the even byte only primes the rotator; the odd byte writes both
            if (bcol[0]) begin
                p0_csb[bbank] = 1'b0;
                p0_web        = {NB{1'b0}};
                p0_mask[bbank*4 + bbyte]       = 1'b1;
                p0_mask[bbank*4 + bbyte - 1'b1] = 1'b1;
            end
            p0_addr = region[1:0] * N + brow;
            p0_din  = {NB*2{rope_yo, rope_xo}};
        end else if (bus_mem_we) begin
            p0_csb[bbank] = 1'b0;
            p0_web        = {NB{1'b0}};
            p0_mask[bbank*4 + bbyte] = 1'b1;
            p0_addr = region[1:0] * N + brow;
            p0_din  = {NB*4{wdata_r}};
        end else if (bus_mem_re) begin
            p0_csb[bbank] = 1'b0;
            p0_addr = AO + brow;
        end
    end

`ifndef SYNTHESIS
    always @(posedge clk)
        if (rst_n && ((pipe ? oq_pop : o_valid) + v_rd + sb_we) > 1)
            $display("%t raijin_core ERROR: SRAM port 0 conflict o=%b v=%b sb=%b", $time, pipe ? oq_pop : o_valid, v_rd, sb_we);
`endif

    // ------------------------------------------------------------------
    // Bus read path: stage 1 picks a register value, stage 2 muxes in SRAM data
    // ------------------------------------------------------------------
    reg       rd_v, rd_mem;
    reg [1:0] rd_bank, rd_byte;
    reg [7:0] rd_reg;
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            rd_v   <= 1'b0;
            rd_mem <= 1'b0;
            rdata  <= 8'd0;
        end else begin
            rd_v    <= re_r;
            rd_mem  <= (region == 4'h3);
            rd_bank <= bbank;
            rd_byte <= bbyte;
            if (re_r) begin
                case (region)
                    4'h4: case (esel)
                              2'd0: rd_reg <= eqm[erow[LN-1:0]];
                              2'd1: rd_reg <= ekm[erow];
                              2'd2: rd_reg <= evm[erow];
                              default: rd_reg <= eom[erow[LN-1:0]];
                          endcase
                    4'h5: case (cidx)
                              5'h00: rd_reg <= {3'd0, decode, mode, fused, 1'b0};
                              5'h01: rd_reg <= offset;
                              5'h02: rd_reg <= {6'd0, done_r, busy_r};
                              5'h03: rd_reg <= 8'd3;
                              5'h04: rd_reg <= cycles[7:0];
                              5'h05: rd_reg <= cycles[15:8];
                              5'h06: rd_reg <= cycles[23:16];
                              5'h07: rd_reg <= cycles[31:24];
                              5'h08: rd_reg <= skips[7:0];
                              5'h09: rd_reg <= skips[15:8];
                              5'h0A: rd_reg <= skips[23:16];
                              5'h0B: rd_reg <= skips[31:24];
                              5'h0C: rd_reg <= N;
                              5'h0D: rd_reg <= D;
                              5'h0E: rd_reg <= P;
                              5'h0F: rd_reg <= VERSION;
                              5'h10: rd_reg <= len;
                              5'h11: rd_reg <= qrow;
                              5'h12: rd_reg <= {4'd0, vfp, kfp, v4, k4};
                              5'h13: rd_reg <= {mfirst, 3'd0, thr};
                              5'h14: rd_reg <= vskips[7:0];
                              5'h15: rd_reg <= vskips[15:8];
                              5'h16: rd_reg <= vskips[23:16];
                              5'h17: rd_reg <= vskips[31:24];
                              5'h18: rd_reg <= {6'd0, rope_abs, rope_en};
                              5'h19: rd_reg <= ropepos;
                              5'h1A: rd_reg <= {6'd0, mixed, paged};
                              5'h1B: rd_reg <= pfmt;
                              5'h1C, 5'h1D, 5'h1E, 5'h1F:
                                  rd_reg <= {{(4-(LK-LPG)){1'b0}}, ptab[{cidx[1:0], 1'b1}],
                                             {(4-(LK-LPG)){1'b0}}, ptab[{cidx[1:0], 1'b0}]};
                              default: rd_reg <= 8'd0;
                          endcase
                    4'h7: rd_reg <= hh_rd;
                    default: rd_reg <= 8'd0;
                endcase
            end
            if (rd_v)
                rdata <= rd_mem ? p0_dout[rd_bank*32 + rd_byte*8 +: 8] : rd_reg;
        end
    end

    // reciprocal results
    integer ri, pi;
    always @(posedge clk) begin
        if (state == RCP_WAIT && sm_rvalid[0])
            for (ri = 0; ri < P; ri = ri + 1)
                rinv[g * P + ri] <= sm_r[ri*13 +: 13];
    end

    // ------------------------------------------------------------------
    // Controller FSM
    // ------------------------------------------------------------------
    wire last_j  = (j == N - 1);
    wire last_jf = decode ? ({1'b0, j} == len - 1'b1) :
                   causal ? (j == g * P + P - 1) : last_j;
    assign fstall = pipe && last_jf && (since_last < L_MIN);
    wire last_g  = (g == NG - 1);

    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            state  <= S_IDLE;
            busy_r <= 1'b0;
            done_r <= 1'b0;
            fused  <= 1'b0;
            decode <= 1'b0;
            causal <= 1'b0;
            multi  <= 1'b0;
            spec   <= 1'b0;
            since_last <= 5'd31;
            mode   <= 2'd0;
            offset <= 8'sd0;
            len    <= N;
            k4     <= 1'b0;
            v4     <= 1'b0;
            kfp    <= 1'b0;
            vfp    <= 1'b0;
            thr    <= 4'd0;
            mfirst <= 1'b0;
            mf_a   <= 1'b0;
            vskips <= 32'd0;
            rope_en  <= 1'b0;
            rope_abs <= 1'b0;
            ropepos  <= 8'd0;
            paged    <= 1'b0;
            mixed    <= 1'b0;
            pfmt     <= {NPG{1'b0}};
            for (pi = 0; pi < NPG; pi = pi + 1) ptab[pi] <= pi;
            qrow   <= {LN{1'b0}};
            cycles <= 32'd0;
            skips  <= 32'd0;
            g      <= {LG{1'b0}};
            j      <= {LK{1'b0}};
            orow   <= {LP{1'b0}};
            qcnt   <= {LP{1'b0}};
            dcnt   <= 4'd0;
            owr_cnt <= 0;
        end else begin
            if (oq_pop) owr_cnt <= owr_cnt + 1'b1;
            if (state == F_STREAM && pipe && last_jf && !fstall) since_last <= 5'd0;
            else if (since_last != 5'd31)                         since_last <= since_last + 1'b1;
            if (we_r && creg && !busy_r) begin
                if (cidx == 5'h01) offset <= wdata_r;
                if (cidx == 5'h10 && !twin) len <= wdata_r[LK:0];
                if (tw_go)                  len <= len + 1'b1;
                if (cidx == 5'h13) begin
                    thr    <= wdata_r[3:0];
                    mfirst <= wdata_r[7];
                end
                if (cidx == 5'h18) begin
                    rope_en  <= wdata_r[0];
                    rope_abs <= wdata_r[1];
                end
                if (cidx == 5'h19) ropepos <= wdata_r;
                if (cidx == 5'h1A) begin
                    paged <= wdata_r[0];
                    mixed <= wdata_r[1];
                end
                if (cidx == 5'h1B) pfmt <= wdata_r[NPG-1:0];
                if (cidx[4:2] == 3'b111) begin
                    ptab[{cidx[1:0], 1'b0}] <= wdata_r[LK-LPG-1:0];
                    ptab[{cidx[1:0], 1'b1}] <= wdata_r[4 +: LK-LPG];
                end
                if (cidx == 5'h12) begin
                    k4  <= wdata_r[0];
                    v4  <= wdata_r[1];
                    kfp <= wdata_r[2];
                    vfp <= wdata_r[3];
                end
                if (cidx == 5'h11) qrow   <= wdata_r[LN-1:0];
            end
            if (busy_r) begin
                cycles <= cycles + 32'd1;
                // keys pass B never visits are counted one cycle after the
                // mask is complete (registered popcount, for timing)
                skips  <= skips + nskip + (mf_cnt_d ? mf_drop_r : {(LK+1){1'b0}});
                vskips <= vskips + {31'd0, v_saved && !mf_cnt} + (mf_cnt_d ? mf_drop_r : {(LK+1){1'b0}});
            end
            case (state)
                S_IDLE: begin
                    if (start_w) begin
                        fused  <= wdata_r[1] | wdata_r[4];
                        decode <= wdata_r[4];
                        causal <= wdata_r[5] && wdata_r[1] && !wdata_r[4];
                        multi  <= wdata_r[4] && (wdata_r[6] || wdata_r[7]);
                        spec   <= wdata_r[4] && wdata_r[7];
                        since_last <= 5'd31;
                        mode   <= wdata_r[3:2];
                        busy_r <= 1'b1;
                        done_r <= 1'b0;
                        cycles <= 32'd0;
                        skips  <= 32'd0;
                        vskips <= 32'd0;
                        owr_cnt <= 0;
                        g      <= wdata_r[4] ? qrow / P : {LG{1'b0}};
                        j      <= 0;
                        qcnt   <= 0;
                        state  <= (wdata_r[1] | wdata_r[4]) ? F_CLR : Q_LOAD;
                    end
                end
                // load the P query rows of group g (both dataflows)
                Q_LOAD: begin
                    qcnt <= qcnt + 1'b1;
                    if (qcnt == P - 1) begin
                        j     <= 0;
                        state <= fused ? F_STREAM : U_QK;
                    end
                end
                // ---------------- unfused ----------------
                U_QK: begin
                    j <= last_j ? {LK{1'b0}} : j + 1'b1;   // wrap at N (j is LK bits)
                    if (last_j) begin
                        g <= g + 1'b1;
                        if (last_g) begin
                            dcnt  <= 4'd5;
                            state <= U_QK_DRAIN;
                        end else begin
                            qcnt  <= 0;
                            state <= Q_LOAD;
                        end
                    end
                end
                U_QK_DRAIN: begin
                    dcnt <= dcnt - 1'b1;
                    if (dcnt == 0) begin
                        g <= 0; j <= 0;
                        state <= U_SM_CLR;
                    end
                end
                U_SM_CLR: begin
                    j <= 0;
                    state <= U_SM_P1;
                end
                U_SM_P1: begin
                    j <= last_j ? {LK{1'b0}} : j + 1'b1;   // wrap at N (j is LK bits)
                    if (last_j) state <= U_SM_P2;
                end
                U_SM_P2: begin
                    j <= last_j ? {LK{1'b0}} : j + 1'b1;   // wrap at N (j is LK bits)
                    if (last_j) begin
                        dcnt  <= 4'd4;
                        state <= U_SM_DRAIN;
                    end
                end
                U_SM_DRAIN: begin
                    dcnt <= dcnt - 1'b1;
                    if (dcnt == 0) state <= RCP_START;
                end
                RCP_START: state <= RCP_WAIT;
                RCP_WAIT: begin
                    if (sm_rvalid[0]) begin
                        if (fused) begin
                            orow  <= (decode && !multi) ? qlane : {LP{1'b0}};
                            state <= OUT_ISSUE;
                        end else if (last_g) begin
                            g <= 0;
                            state <= U_PV_CLR;
                        end else begin
                            g <= g + 1'b1;
                            state <= U_SM_CLR;
                        end
                    end
                end
                U_PV_CLR: begin
                    j <= 0;
                    state <= U_PV;
                end
                U_PV: begin
                    j <= last_j ? {LK{1'b0}} : j + 1'b1;   // wrap at N (j is LK bits)
                    if (last_j) begin
                        dcnt  <= 4'd4;
                        state <= U_PV_DRAIN;
                    end
                end
                U_PV_DRAIN: begin
                    dcnt <= dcnt - 1'b1;
                    if (dcnt == 0) begin
                        orow  <= 0;
                        state <= OUT_ISSUE;
                    end
                end
                OUT_ISSUE: begin
                    orow <= orow + 1'b1;
                    if ((decode && !multi) || orow == P - 1) begin
                        if (decode || last_g) begin
                            dcnt  <= 4'd4;
                            state <= OUT_DRAIN;
                        end else begin
                            g     <= g + 1'b1;
                            state <= OUT_WAIT;
                        end
                    end
                end
                // one cycle so the last O write (port 0) lands before the next V read
                OUT_WAIT: state <= fused ? F_CLR : U_PV_CLR;
                OUT_DRAIN: begin
                    dcnt <= dcnt - 1'b1;
                    if (dcnt == 0) state <= S_DONE;
                end
                // ---------------- fused / decode ----------------
                F_CLR: begin
                    qcnt  <= 0;
                    mf_a  <= mfd;
                    state <= Q_LOAD;
                end
                F_STREAM: if (!fstall) begin
                    j <= j + 1'b1;
                    if (last_jf) begin
                        if (pipe && !last_g) begin
                            // next group streams right behind this one
                            g     <= g + 1'b1;
                            qcnt  <= 0;
                            state <= Q_LOAD;
                        end else if (pipe) begin
                            state <= P_WAIT;
                        end else begin
                            // max-first: the last score reaches the lanes 4 cycles on, m is registered once more
                            dcnt  <= mf_a ? 4'd4 : 4'd6;
                            state <= mf_a ? MF_WAIT : F_DRAIN;
                        end
                    end
                end
                MF_WAIT: begin
                    dcnt <= dcnt - 1'b1;
                    if (dcnt == 0) state <= MF_MASK;
                end
                MF_MASK: begin
                    mf_a  <= 1'b0;
                    state <= MF_B;
                end
                MF_B: begin
                    if (!(|mf_mask)) begin
                        dcnt  <= 4'd4;
                        state <= F_DRAIN;
                    end
                end
                P_WAIT: begin
                    if (owr_cnt == N) state <= S_DONE;
                end
                F_DRAIN: begin
                    dcnt <= dcnt - 1'b1;
                    if (dcnt == 0) state <= RCP_START;
                end
                S_DONE: if (!hh_busy) begin    // the tracker walk finishes first
                    busy_r <= 1'b0;
                    done_r <= 1'b1;
                    state  <= S_IDLE;
                end
                default: state <= S_IDLE;
            endcase
        end
    end
endmodule
`default_nettype wire


Writing rtl/raijin_core.v


### `rtl/raijin_spi.v`
SPI / QSPI host port.

In [13]:
%%writefile rtl/raijin_spi.v
// Raijin SPI / QSPI target: the chip's host port for tapeout (features 11, 12).
//
// SPI mode 0 (CPOL = 0, CPHA = 0), MSB first, byte commands in the style of
// SPI SRAMs and flash, with an auto-incrementing 16-bit address. io[0] is
// MOSI and io[1] is MISO in single-lane mode; quad commands move data on
// io[3:0], high nibble first:
//   WRITE       0x02, AH, AL, data...             (all single lane)
//   READ        0x0B, AH, AL, dummy, data...      (data on io[1])
//   QUAD WRITE  0x32, AH, AL, data...             (data on io[3:0], 2 clocks per byte)
//   QUAD READ   0x6B, AH, AL, dummy, data...      (data driven on io[3:0])
// The command, address and dummy byte are always single lane, so a plain
// SPI host can use the chip and a QSPI host gets 4x the data bandwidth.
// Every data byte becomes one core bus write or read, so the whole core
// address map is reachable.
//
// SCK, CS_N and io are sampled in the core clock domain through two-flop
// synchronizers, so SCK must be at most f_clk / 8 (6.25 MHz at 50 MHz). The
// read for each data byte is issued one byte ahead, so the core's 3-cycle
// read latency is hidden.
`default_nettype none
module raijin_spi (
    input  wire        clk,
    input  wire        rst_n,
    input  wire        sck,
    input  wire        cs_n,
    input  wire [3:0]  io_in,
    output wire [3:0]  io_out,
    output wire [3:0]  io_oe,
    output wire        active,        // a transaction is in progress
    // core bus
    output reg  [15:0] addr,
    output reg  [7:0]  wdata,
    output reg         we,
    output reg         re,
    input  wire [7:0]  rdata
);
    localparam [7:0] CMD_WRITE = 8'h02, CMD_READ = 8'h0B, CMD_QWRITE = 8'h32, CMD_QREAD = 8'h6B;
    localparam [1:0] PH_CMD = 2'd0, PH_AH = 2'd1, PH_AL = 2'd2, PH_DATA = 2'd3;

    reg [2:0] sck_s;
    reg [1:0] cs_s;
    reg [3:0] io_s0, io_s1;
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            sck_s <= 3'b000;
            cs_s  <= 2'b11;
            io_s0 <= 4'd0;
            io_s1 <= 4'd0;
        end else begin
            sck_s <= {sck_s[1:0], sck};
            cs_s  <= {cs_s[0], cs_n};
            io_s0 <= io_in;
            io_s1 <= io_s0;
        end
    end
    wire sel  = !cs_s[1];
    wire rise = sel && sck_s[1] && !sck_s[2];
    wire fall = sel && !sck_s[1] && sck_s[2];
    assign active = sel;

    reg  [1:0] phase;
    reg  [2:0] bitcnt;
    reg  [6:0] sh;
    reg  [7:0] cmd;
    reg        dummy;          // read: the dummy byte is still to come
    reg        load;           // next falling edge starts a MISO byte
    reg  [7:0] tx, rbuf;
    reg  [2:0] rv;             // read in flight (core latency 3)
    wire       rd_cmd  = (cmd == CMD_READ) || (cmd == CMD_QREAD);
    wire       wr_cmd  = (cmd == CMD_WRITE) || (cmd == CMD_QWRITE);
    wire       quad    = (cmd == CMD_QREAD) || (cmd == CMD_QWRITE);
    // quad data phase (after the dummy byte for reads): 4 bits per clock
    wire       qdata   = quad && (phase == PH_DATA) && !dummy;
    wire [7:0] byte_in = {sh, io_s1[0]};
    wire [7:0] qbyte   = {sh[3:0], io_s1};
    wire       byte_end = qdata ? (bitcnt[0] == 1'b1) : (bitcnt == 3'd7);

    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            phase  <= PH_CMD;
            bitcnt <= 3'd0;
            sh     <= 7'd0;
            cmd    <= 8'd0;
            dummy  <= 1'b0;
            load   <= 1'b0;
            tx     <= 8'd0;
            rbuf   <= 8'd0;
            rv     <= 3'd0;
            addr   <= 16'd0;
            wdata  <= 8'd0;
            we     <= 1'b0;
            re     <= 1'b0;
        end else begin
            we <= 1'b0;
            re <= 1'b0;
            rv <= {rv[1:0], re};
            if (rv[2]) rbuf <= rdata;
            if (!sel) begin
                phase  <= PH_CMD;
                bitcnt <= 3'd0;
                load   <= 1'b0;
                dummy  <= 1'b0;
                tx     <= 8'd0;
                cmd    <= 8'd0;
            end else begin
                if (rise) begin
                    sh     <= qdata ? {3'd0, io_s1} : byte_in[6:0];
                    bitcnt <= byte_end ? 3'd0 : bitcnt + 3'd1;
                    if (byte_end) begin
                        case (phase)
                            PH_CMD: begin
                                cmd   <= byte_in;
                                phase <= PH_AH;
                            end
                            PH_AH: begin
                                addr[15:8] <= byte_in;
                                phase      <= PH_AL;
                            end
                            PH_AL: begin
                                addr[7:0] <= byte_in;
                                phase     <= PH_DATA;
                                dummy     <= rd_cmd;
                                if (rd_cmd) re <= 1'b1;      // prefetch the first byte
                            end
                            default: begin
                                if (rd_cmd) begin
                                    dummy <= 1'b0;
                                    load  <= 1'b1;
                                end else if (wr_cmd) begin
                                    wdata <= qdata ? qbyte : byte_in;
                                    we    <= 1'b1;
                                end
                            end
                        endcase
                    end
                end
                // a write byte advances the address once its write is issued
                if (we) addr <= addr + 16'd1;
                if (fall) begin
                    if (load) begin
                        tx   <= rbuf;
                        load <= 1'b0;
                        addr <= addr + 16'd1;
                        re   <= 1'b1;               // prefetch the byte after
                    end else begin
                        tx <= quad ? {tx[3:0], 4'd0} : {tx[6:0], 1'b0};
                    end
                end
            end
        end
    end
    // single lane: MISO on io[1]; quad read data phase: tx[7:4] on io[3:0]
    wire qout = (cmd == CMD_QREAD) && (phase == PH_DATA) && !dummy;
    assign io_out = qout ? tx[7:4] : {2'b00, tx[7], 1'b0};
    assign io_oe  = !sel ? 4'b0000 : (qout ? 4'b1111 : 4'b0010);
endmodule
`default_nettype wire


Writing rtl/raijin_spi.v


### `rtl/raijin_chip.v`
Chip top level used for OpenLane: SPI port, clock gating, reset synchroniser.

In [14]:
%%writefile rtl/raijin_chip.v
// Raijin chip top (feature 11): SPI host port, idle clock gating, and the core.
//
// Pins: clk, rst_n, sck, cs_n, io[3:0] (bidirectional: io_in / io_out / io_oe
// to the pads; io[0] MOSI and io[1] MISO in single-lane SPI, all four in QSPI),
// done (IRQ), busy. The SPI target runs on the free-running clock; the core runs on a
// gated clock that is on only while a pass runs and for HOLD cycles after each
// bus access the SPI target makes, or after the pass ends. HOLD covers the
// core's two-cycle write and three-cycle read latency, and lets done settle.
// An SPI byte takes 64 clocks at SCK = f_clk / 8, so even during host traffic
// the core clock runs about 8 of every 64 cycles, and between tokens it stops.
// This removes the clock-tree and flop power of the whole datapath for that
// time. The gate is a sky130 integrated clock gate under
// RAIJIN_SKY130_CELLS, and a behavioural latch-and-AND otherwise.
// GATE_EN = 0 ties the gate on (for A/B comparisons).
`default_nettype none
module raijin_chip #(
    parameter N = 16,
    parameter D = 16,
    parameter P = 4,
    parameter GATE_EN = 1,
    parameter HOLD = 7
) (
    input  wire clk,
    input  wire rst_n,
    input  wire sck,
    input  wire cs_n,
    input  wire [3:0] io_in,        // io[0] MOSI, io[1] MISO, io[3:0] quad data
    output wire [3:0] io_out,
    output wire [3:0] io_oe,
    output wire done,
    output wire busy,
    output reg  [31:0] gated_cycles     // free-running count of cycles the core clock was off
);
    wire [15:0] addr;
    wire [7:0]  wdata, rdata;
    wire        we, re, active;

    // ---- reset synchroniser: rst_n asserts asynchronously and is released
    // two clk edges later, in step with clk, so every flop leaves reset in the
    // same cycle and the pin needs no timing (raijin_chip.sdc false-paths it) ----
    reg [1:0] rst_s;
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) rst_s <= 2'b00;
        else        rst_s <= {rst_s[0], 1'b1};
    end
    wire srst_n = rst_s[1];

    raijin_spi u_spi (
        .clk(clk), .rst_n(srst_n), .sck(sck), .cs_n(cs_n),
        .io_in(io_in), .io_out(io_out), .io_oe(io_oe), .active(active),
        .addr(addr), .wdata(wdata), .we(we), .re(re), .rdata(rdata));

    // ---- clock gate enable ----
    reg [4:0] hold;
    always @(posedge clk or negedge srst_n) begin
        if (!srst_n)                hold <= HOLD;
        else if (we || re || busy)  hold <= HOLD;
        else if (hold != 5'd0)      hold <= hold - 5'd1;
    end
    wire cg_en = (GATE_EN == 0) || we || re || busy || (hold != 5'd0);

    always @(posedge clk or negedge srst_n) begin
        if (!srst_n)     gated_cycles <= 32'd0;
        else if (!cg_en) gated_cycles <= gated_cycles + 32'd1;
    end

    wire gclk;
`ifdef RAIJIN_SKY130_CELLS
    (* keep *) sky130_fd_sc_hd__dlclkp_1 u_icg (.CLK(clk), .GATE(cg_en), .GCLK(gclk));
`else
    reg cg_l;
    always @(clk or cg_en)
        if (!clk) cg_l = cg_en;
    assign gclk = clk & cg_l;
`endif

    // core reset: released again in step with gclk, so its recovery/removal
    // checks are gclk-to-gclk and see no clk/gclk insertion skew. The gate is
    // open throughout (hold reloads to HOLD in reset and counts 7 cycles after).
    reg [1:0] crst_s;
    always @(posedge gclk or negedge srst_n) begin
        if (!srst_n) crst_s <= 2'b00;
        else         crst_s <= {crst_s[0], 1'b1};
    end

    raijin_core #(.N(N), .D(D), .P(P)) u_core (
        .clk(gclk), .rst_n(crst_s[1]), .addr(addr), .wdata(wdata), .we(we), .re(re),
        .rdata(rdata), .busy(busy), .done(done));
endmodule
`default_nettype wire


Writing rtl/raijin_chip.v


## 4. Golden model (`model/`)

`raijin_model.py` is the bit-exact reference: MX quantisation, every arithmetic step of the hardware in integers, and an FP32 reference. `raijin_vec.py` is a vectorised version of the decode path. The testbench imports both.

In [15]:
%%writefile model/raijin_model.py
"""Bit-exact golden model of the Raijin attention accelerator.

Every function here mirrors one RTL block. The integer formats are described in
docs/numerics.md. Run this file directly for a quick self-check.

Formats (summary):
  Q, K, V elements : signed INT8, 6 fraction bits; one E8M0 exponent per row
                     value = x * 2^-6 * 2^(E - 127)
  dot q.k          : signed 20-bit integer
  score s          : signed 12-bit, Q7.4 (log2 domain; host folds log2(e)/sqrt(d) into Q)
  running max m    : signed 9-bit integer, initial value -256
  p                : unsigned 9-bit, Q1.8
  l                : unsigned 14-bit, Q6.8 (sums up to 63.99, 32 keys of 1.0)
  acc              : signed 24-bit, sum of (p * v) >> dv
  1/l              : unsigned 13-bit, floor(2^19 / l)
  output           : MXINT8 row (INT8 elements + E8M0 exponent)
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import List, Sequence, Tuple

import numpy as np

# ---------------------------------------------------------------------------
# Constants shared with the RTL (rtl/raijin_exp2.v)
# ---------------------------------------------------------------------------
EXP_LUT = [256, 245, 235, 225, 215, 206, 197, 189, 181, 173, 166, 159, 152, 146, 140, 134]
# Multiplier-free, one power of two: 2^-f/16 ~= 2^-MF1_S1[f]
MF1_S1 = [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1]
# Multiplier-free, two powers of two: 2^-f/16 ~= 2^-s1 + 2^-s2  (s2 = 15 means no second term)
MF2_S1 = [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
MF2_S2 = [15, 15, 15, 15, 2, 2, 2, 2, 2, 3, 3, 3, 3, 4, 5, 6]

MODE_EXACT, MODE_MF1, MODE_MF2 = 0, 1, 2
M_INIT = -256          # running-max reset value (below any reachable ceil(s))
NONE_CODE = 15         # shift code meaning "no term"
EXP_BIAS = 127
FRAC = 6               # fraction bits of MXINT8 elements


def _sat(x: int, bits: int) -> int:
    lo, hi = -(1 << (bits - 1)), (1 << (bits - 1)) - 1
    return lo if x < lo else hi if x > hi else x


def _sra(x: int, n: int) -> int:
    """Arithmetic shift right, Python >> is already floor for negatives."""
    return x >> n


# ---------------------------------------------------------------------------
# Block-level functions
# ---------------------------------------------------------------------------
def dpu(q: Sequence[int], k: Sequence[int]) -> int:
    """Dot product of two INT8 rows, signed 20-bit result."""
    d = int(sum(int(a) * int(b) for a, b in zip(q, k)))
    assert -(1 << 19) <= d < (1 << 19)
    return d


def requant(dot: int, esum: int, offset: int) -> int:
    """Move a raw dot product into Q7.4 using the row exponents.

    sh = e_q + e_k - 262 + offset.  s = sat12(dot << sh) or sat12(dot >> -sh).
    The shift amount is clamped to [-31, 15] in hardware.
    """
    sh = esum - 262 + offset
    if sh >= 0:
        sh = min(sh, 15)
        return _sat(dot << sh, 12)
    sh = min(-sh, 31)
    return _sat(_sra(dot, sh), 12)


def ceil_int(s: int) -> int:
    """ceil(s / 16) for Q7.4 s, a 9-bit signed integer."""
    return (s + 15) >> 4


def term(sh: int) -> int:
    """256 >> sh, or 0 when the shift code means 'no term' (sh > 8)."""
    return 0 if sh > 8 else 256 >> sh


def exp2(s: int, m: int, mode: int, thr: int = 0) -> Tuple[int, int, int]:
    """p = 2^(s/16 - m) in Q1.8 plus the two shift codes used by the AV path.

    Returns (p, sa, sb). In exact mode sa/sb are unused (returned as NONE_CODE).
    thr (key-skip threshold, 0 = off): a key whose score is thr or more
    powers of two below the running max gets p = 0 and is skipped.
    """
    t = m * 16 - s
    assert t >= 0
    k, f = t >> 4, t & 15
    if thr and k >= thr:
        return 0, NONE_CODE, NONE_CODE
    if mode == MODE_EXACT:
        p = 0 if k >= 9 else EXP_LUT[f] >> k
        return p, NONE_CODE, NONE_CODE
    if mode == MODE_MF1:
        s1, s2 = MF1_S1[f], NONE_CODE
    else:
        s1, s2 = MF2_S1[f], MF2_S2[f]
    sa = min(k + s1, NONE_CODE)
    sb = NONE_CODE if s2 == NONE_CODE else min(k + s2, NONE_CODE)
    return term(sa) + term(sb), sa, sb


def pv_term(p: int, v: int, dv: int) -> int:
    """(p * v) >> dv. In multiplier-free mode p has at most two bits set, so the
    RTL computes the same value with shifts and one add."""
    return _sra(p * int(v), min(dv, 31))


def recip(l: int) -> int:
    """floor(2^19 / l), 13 bits. l >= 128 is guaranteed by construction."""
    assert l >= 128, l
    return (1 << 19) // l


def _bitlen_signed(y: int) -> int:
    """Number of magnitude bits of a signed value (excluding sign)."""
    return (y ^ (y >> 63)).bit_length() if y < 0 else y.bit_length()


def out_row(acc: Sequence[int], r: int, emax_v: int) -> Tuple[List[int], int]:
    """Normalise one accumulator row by 1/l and pack it as MXINT8."""
    y = [int(a) * r for a in acc]
    nb = max(_bitlen_signed(v) for v in y)
    sh = max(0, nb - 7)
    out = []
    for v in y:
        if sh:
            v = (v + (1 << (sh - 1))) >> sh
        out.append(_sat(v, 8))
    eo = min(max(emax_v + sh - 19, 0), 255)
    return out, eo


# ---------------------------------------------------------------------------
# Full attention pass
# ---------------------------------------------------------------------------
@dataclass
class PassResult:
    o: np.ndarray                 # (N, D) int8 elements
    eo: np.ndarray                # (N,) output row exponents
    skips: int = 0
    l: List[int] = field(default_factory=list)
    scores: np.ndarray | None = None
    used: np.ndarray | None = None   # (rows, keys) bool: key j added to row i


def v_reads_saved(used: np.ndarray, p: int, causal: bool = False) -> int:
    """V-row reads the RTL saves: one per (group, key) beat in which no row of
    the P-row group uses the key. A causal pass streams group g only up to
    key g*P+P-1."""
    n_rows, n_keys = used.shape
    saved = 0
    for g in range(n_rows // p):
        last = g * p + p - 1 if causal else n_keys - 1
        for j in range(last + 1):
            if not used[g * p:(g + 1) * p, j].any():
                saved += 1
    return saved


def attention(q, eq, k, ek, v, ev, *, fused: bool, mode: int = MODE_EXACT,
              offset: int = 0, causal: bool = False, thr: int = 0) -> PassResult:
    """Full attention pass. causal (fused only): row i sees keys 0..i; masked
    keys are not processed at all (and not counted as skips). e_vmax is still
    taken over all N V rows, as in the RTL."""
    assert fused or not causal
    q, k, v = (np.asarray(a, dtype=np.int64) for a in (q, k, v))
    eq, ek, ev = (np.asarray(a, dtype=np.int64) for a in (eq, ek, ev))
    n, d = q.shape
    emax = int(ev.max())
    dvs = [emax - int(e) for e in ev]
    o = np.zeros((n, d), dtype=np.int64)
    eo = np.zeros(n, dtype=np.int64)
    skips = 0
    ls = []
    scores = np.zeros((n, n), dtype=np.int64)
    used = np.zeros((n, n), dtype=bool)
    for i in range(n):
        s = [requant(dpu(q[i], k[j]), int(eq[i]) + int(ek[j]), offset) for j in range(n)]
        scores[i] = s
        acc = [0] * d
        lsum = 0
        if fused:
            m = M_INIT
            for j in range(i + 1 if causal else n):
                mn = max(m, ceil_int(s[j]))
                delta = mn - m
                m = mn
                p, _, _ = exp2(s[j], m, mode, thr)
                if p == 0:
                    skips += 1          # delta is always 0 here
                    continue
                used[i, j] = True
                dsh = min(delta, 31)
                lsum = (lsum >> dsh) + p
                for c in range(d):
                    acc[c] = _sra(acc[c], dsh) + pv_term(p, v[j][c], dvs[j])
        else:
            m = M_INIT
            for j in range(n):
                m = max(m, ceil_int(s[j]))
            for j in range(n):
                p, _, _ = exp2(s[j], m, mode, thr)
                if p == 0:
                    skips += 1
                    continue
                used[i, j] = True
                lsum += p
                for c in range(d):
                    acc[c] += pv_term(p, v[j][c], dvs[j])
        for c in range(d):
            assert -(1 << 23) <= acc[c] < (1 << 23)
        assert lsum < (1 << 14)
        ls.append(lsum)
        row, e = out_row(acc, recip(lsum), emax)
        o[i] = row
        eo[i] = e
    return PassResult(o=o, eo=eo, skips=skips, l=ls, scores=scores, used=used)


def attention_decode(q, eq, k, ek, v, ev, *, qrow: int, length: int,
                     mode: int = MODE_EXACT, offset: int = 0,
                     emax_len: int | None = None, thr: int = 0,
                     final_max: bool = False) -> PassResult:
    """One KV-cache decode step: query row `qrow` attends to keys 0..length-1.

    final_max (max-first decode, feature 17): the running maximum starts at the
    row's final maximum, as when a score pass finds it before any weight is
    formed. No rescale then happens, and every key whose weight underflows
    against the true maximum is skipped.

    This is the fused (online softmax) pass for one query over a key prefix.
    e_vmax is taken over the cached V rows only, as in the RTL. The result
    holds a single output row.
    """
    q, k, v = (np.asarray(a, dtype=np.int64) for a in (q, k, v))
    eq, ek, ev = (np.asarray(a, dtype=np.int64) for a in (eq, ek, ev))
    d = q.shape[1]
    emax = int(ev[:length if emax_len is None else emax_len].max())
    s = [requant(dpu(q[qrow], k[j]), int(eq[qrow]) + int(ek[j]), offset) for j in range(length)]
    acc = [0] * d
    lsum, m, skips = 0, M_INIT, 0
    if final_max:
        m = max(M_INIT, max(ceil_int(x) for x in s))
    for j in range(length):
        mn = max(m, ceil_int(s[j]))
        delta = mn - m
        m = mn
        p, _, _ = exp2(s[j], m, mode, thr)
        if p == 0:
            skips += 1
            continue
        dsh = min(delta, 31)
        lsum = (lsum >> dsh) + p
        dv = emax - int(ev[j])
        for c in range(d):
            acc[c] = _sra(acc[c], dsh) + pv_term(p, v[j][c], dv)
    assert lsum < (1 << 14), lsum          # the 14-bit l register
    row, e = out_row(acc, recip(lsum), emax)
    return PassResult(o=np.array([row], dtype=np.int64), eo=np.array([e], dtype=np.int64),
                      skips=skips, l=[lsum])


def decode_scores(q, eq, k, ek, *, qrow: int, length: int, offset: int = 0):
    """The Q7.4 scores of one decode step (what the softmax lane sees)."""
    q, k = np.asarray(q, dtype=np.int64), np.asarray(k, dtype=np.int64)
    return [requant(dpu(q[qrow], k[j]), int(eq[qrow]) + int(ek[j]), offset) for j in range(length)]


HH_NK = 32


def hh_step(acc, scores, slots, skip=0, recent=0):
    """Heavy-hitter tracker (rtl/raijin_hh.v) after one single-row decode step.

    acc: list of HH_NK 16-bit accumulators (updated in place); scores[i] is the
    score of the key held in physical slot slots[i]. Each key adds its
    max-normalised weight 2^((s - 16 m) / 16) in Q0.8, as
    (frac >= 1/2 ? 181 : 256) >> floor((16 m - s) / 16), 0 below 2^-8, where m
    is the row's final running maximum. Returns the eviction candidate: the
    slot >= skip with the smallest accumulator among this step's slots (lowest
    index on ties). Keys among the step's newest `recent` positions are never
    chosen (H2O keeps a recent window)."""
    m = M_INIT
    for sj in scores:
        m = max(m, ceil_int(sj))
    used, rec = {}, set()
    n = len(scores)
    for j, (sj, sl) in enumerate(zip(scores, slots)):
        used[sl] = sj
        if j + recent >= n:
            rec.add(sl)
    best, best_k = 1 << 17, 0
    for sl in range(HH_NK):
        if sl not in used:
            continue
        d = 16 * m - used[sl]
        if d < 0:
            w = 256
        else:
            di = d >> 4
            w = 0 if di > 8 else ((181 if d & 8 else 256) >> di)
        acc[sl] = min(acc[sl] + w, 0xFFFF)
        if sl >= skip and sl not in rec and acc[sl] < best:
            best, best_k = acc[sl], sl
    return best_k


def attention_decode_multi(q, eq, k, ek, v, ev, *, group: int, p: int, length: int,
                           spec: bool = False, mode: int = MODE_EXACT,
                           offset: int = 0, thr: int = 0) -> PassResult:
    """Multi-query decode step: the p query rows of `group` (rows group*p ..
    group*p+p-1) attend to the cached keys in one step, as the RTL does.

    GQA / MQA (spec=False): every row sees keys 0..length-1.
    Speculative verify (spec=True): the rows are p draft tokens whose K/V are
    the last p cached rows; row r sees keys 0..length-p+r.
    e_vmax is over the length cached V rows for every row."""
    rows, eos, ls, skips = [], [], [], 0
    for r in range(p):
        n_keys = length - p + r + 1 if spec else length
        res = attention_decode(q, eq, k, ek, v, ev, qrow=group * p + r, length=n_keys,
                               mode=mode, offset=offset, emax_len=length, thr=thr)
        rows.append(res.o[0])
        eos.append(res.eo[0])
        ls += res.l
        skips += res.skips
    return PassResult(o=np.array(rows, dtype=np.int64), eo=np.array(eos, dtype=np.int64),
                      skips=skips, l=ls)


def _check_decode_matches_fused(rng) -> None:
    """With length = N, decode of row i must equal row i of the fused pass."""
    qf, kf, vf = random_case(rng, scale=1.5)
    args = prepare(qf, kf, vf)
    full = attention(*args, fused=True)
    for i in range(qf.shape[0]):
        r = attention_decode(*args, qrow=i, length=qf.shape[0])
        assert np.array_equal(r.o[0], full.o[i]) and r.eo[0] == full.eo[i], i


def streaming_slot(t: int, n: int, sinks: int) -> int:
    """KV-cache row for token t in a StreamingLLM-style cache of n rows: the
    first `sinks` tokens stay pinned (attention sinks), and the other n - sinks
    rows form a ring that always holds the most recent tokens."""
    if t < n:
        return t
    return sinks + (t - sinks) % (n - sinks)


def reference_streaming(qf, kf, vf, n: int, sinks: int):
    """FP32 attention for every token t over the keys a streaming cache of n
    rows holds at step t: tokens 0..sinks-1 plus the last n - sinks tokens."""
    out = []
    for t in range(qf.shape[0]):
        keys = list(range(t + 1)) if t < n else \
            list(range(sinks)) + list(range(t - (n - sinks) + 1, t + 1))
        s = qf[t] @ kf[keys].T / np.sqrt(qf.shape[1])
        p = np.exp(s - s.max())
        out.append((p / p.sum()) @ vf[keys])
    return np.array(out)


# ---------------------------------------------------------------------------
# Quantisation helpers (host side)
# ---------------------------------------------------------------------------
def mx_quantize(x: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Quantise each row of a float matrix to MXINT8 (INT8 with 6 fraction bits
    and one shared E8M0 exponent per row)."""
    x = np.asarray(x, dtype=np.float64)
    q = np.zeros(x.shape, dtype=np.int64)
    e = np.zeros(x.shape[0], dtype=np.int64)
    for i, row in enumerate(x):
        amax = float(np.max(np.abs(row)))
        if amax == 0:
            ex = 0
        else:
            # want |row| / 2^ex * 64 <= 127  ->  ex >= log2(amax * 64 / 127)
            ex = int(np.ceil(np.log2(amax * 64 / 127)))
        ex = min(max(ex, -127), 128)
        q[i] = np.clip(np.round(row / 2.0 ** ex * 64), -128, 127)
        e[i] = ex + EXP_BIAS
    return q, e


def mx_quantize4(x: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Quantise each row to MXINT4: INT4 elements in [-8, 7] with 2 fraction
    bits and one shared E8M0 exponent per row. Returns the elements as the
    INT8 datapath sees them (x4 << 4), so the INT8 golden model applies as is,
    together with the exponents."""
    x = np.asarray(x, dtype=np.float64)
    q4 = np.zeros(x.shape, dtype=np.int64)
    e = np.zeros(x.shape[0], dtype=np.int64)
    for i, row in enumerate(x):
        amax = float(np.max(np.abs(row)))
        ex = 0 if amax == 0 else int(np.ceil(np.log2(amax * 4 / 7)))
        ex = min(max(ex, -127), 128)
        q4[i] = np.clip(np.round(row / 2.0 ** ex * 4), -8, 7)
        e[i] = ex + EXP_BIAS
    return q4 * 16, e


def pack4(row8: Sequence[int]) -> List[int]:
    """Pack one MXINT4 row (given as x4 << 4) into 8 bytes: element 2b in the
    low nibble of byte b, element 2b+1 in the high nibble."""
    n = [(int(x) >> 4) & 0xF for x in row8]
    return [n[2 * b] | (n[2 * b + 1] << 4) for b in range(len(n) // 2)]


# MXFP4 (OCP MX, E2M1 elements): nibble s e1 e0 m. Twice the value is an
# integer, 2|x| in {0, 1, 2, 3, 4, 6, 8, 12}, so the chip widens a nibble to
# INT8 as +-(2|x|) << 3 = 16 x, the same scale as an MXINT4 element x4 << 4.
FP4_MAG2 = (0, 1, 2, 3, 4, 6, 8, 12)          # 2|x| for codes 0..7
FP4_GRID = np.array(sorted({s * m / 2 for m in FP4_MAG2 for s in (1, -1)}))


def fp4_code8(nib: int) -> int:
    """INT8 element the datapath sees for one E2M1 nibble (16 x)."""
    mag = FP4_MAG2[nib & 7] << 3
    return -mag if nib & 8 else mag


def mx_quantize_fp4(x: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Quantise each row to MXFP4: E2M1 elements (|x| <= 6) and one shared E8M0
    exponent per row. The host tries the exponent at which the row maximum just
    fits (|x| <= 6) and the one below it (the maximum then clips to 6, the
    other elements get a finer grid) and keeps the one with the lower squared
    error; the chip only sees the exponent, so this choice is free.
    Elements are returned as the INT8 datapath sees them (16 x), with the
    exponents; a value is then 16 x / 64 * 2^(e - bias), as for MXINT4."""
    x = np.asarray(x, dtype=np.float64)
    q = np.zeros(x.shape, dtype=np.int64)
    e = np.zeros(x.shape[0], dtype=np.int64)
    for i, row in enumerate(x):
        amax = float(np.max(np.abs(row)))
        ex0 = 0 if amax == 0 else int(np.ceil(np.log2(amax * 4 / 6)))
        best = None
        for ex in (ex0, ex0 - 1):            # host picks the lower-error exponent
            ex = min(max(ex, -127), 128)
            y = row / 2.0 ** ex * 4
            g = FP4_GRID[np.abs(y[:, None] - FP4_GRID[None, :]).argmin(axis=1)]
            err = float(np.sum((g / 4 * 2.0 ** ex - row) ** 2))
            if best is None or err < best[0]:
                best = (err, g, ex)
        q[i] = np.round(best[1] * 16).astype(np.int64)
        e[i] = best[2] + EXP_BIAS
    return q, e


def fp4_nibble(c8: int) -> int:
    """E2M1 nibble of an INT8 element 16 x (inverse of fp4_code8)."""
    mag = FP4_MAG2.index(abs(int(c8)) >> 3)
    return mag | (8 if c8 < 0 else 0)


def pack_fp4(row8: Sequence[int]) -> List[int]:
    """Pack one MXFP4 row (given as 16 x) into 8 bytes, as pack4."""
    n = [fp4_nibble(x) for x in row8]
    return [n[2 * b] | (n[2 * b + 1] << 4) for b in range(len(n) // 2)]


def prepare_kvfp4(qf, kf, vf):
    """Like prepare(), but K and V in MXFP4 (returned as 16 x)."""
    d = qf.shape[1]
    q, eq = mx_quantize(qf * (np.log2(np.e) / np.sqrt(d)))
    k, ek = mx_quantize_fp4(kf)
    v, ev = mx_quantize_fp4(vf)
    return q, eq, k, ek, v, ev


def prepare_kv4(qf, kf, vf):
    """Like prepare(), but K and V in MXINT4 (returned as x4 << 4)."""
    d = qf.shape[1]
    q, eq = mx_quantize(qf * (np.log2(np.e) / np.sqrt(d)))
    k, ek = mx_quantize4(kf)
    v, ev = mx_quantize4(vf)
    return q, eq, k, ek, v, ev


def mx_dequantize(q: np.ndarray, e: np.ndarray) -> np.ndarray:
    q = np.asarray(q, dtype=np.float64)
    e = np.asarray(e, dtype=np.float64)
    return q / 64.0 * 2.0 ** (e[:, None] - EXP_BIAS)


def prepare(qf, kf, vf):
    """Fold log2(e)/sqrt(d) into Q and quantise all three to MXINT8."""
    d = qf.shape[1]
    q, eq = mx_quantize(qf * (np.log2(np.e) / np.sqrt(d)))
    k, ek = mx_quantize(kf)
    v, ev = mx_quantize(vf)
    return q, eq, k, ek, v, ev


def reference(qf, kf, vf):
    s = qf @ kf.T / np.sqrt(qf.shape[1])
    s = s - s.max(axis=1, keepdims=True)
    p = np.exp(s)
    p /= p.sum(axis=1, keepdims=True)
    return p @ vf


def reference_causal(qf, kf, vf):
    """FP32 causal attention: row i sees keys 0..i (what decode computes)."""
    n = qf.shape[0]
    s = qf @ kf.T / np.sqrt(qf.shape[1])
    s = np.where(np.tril(np.ones((n, n), dtype=bool)), s, -np.inf)
    s = s - s.max(axis=1, keepdims=True)
    p = np.exp(s)
    p /= p.sum(axis=1, keepdims=True)
    return p @ vf


# ---------------------------------------------------------------------------
# RoPE on write (feature 9)
# ---------------------------------------------------------------------------
# Pair i = elements (2i, 2i+1) turns by pos * theta_i, theta_i = 10000^(-2i/d).
# Phase is in turns, Q0.20: ROPE_W[i] = round(2^20 * theta_i / (2 pi)).
ROPE_W = [166886, 52774, 16689, 5277, 1669, 528, 167, 53]
# quarter-wave sine, Q1.8: ROPE_SIN[k] = round(256 * sin(2 pi k / 1024)), k = 0..256
ROPE_SIN = [int(round(256 * np.sin(2 * np.pi * k / 1024))) for k in range(257)]


def rope_cs(pos: int, i: int) -> Tuple[int, int]:
    """(cos, sin) in Q1.8 for pair i at position pos, as the RTL computes it."""
    ph = (pos * ROPE_W[i]) & 0xFFFFF
    a = ((ph + (1 << 9)) >> 10) & 1023          # 10-bit phase, rounded
    q, k = a >> 8, a & 255
    s0, c0 = ROPE_SIN[k], ROPE_SIN[256 - k]
    return [(c0, s0), (-s0, c0), (-c0, -s0), (s0, -c0)][q]


def rope_row(row: Sequence[int], pos: int) -> List[int]:
    """Rotate one INT8 row as the write-path rotator does: round, then
    saturate to INT8. The row keeps its exponent."""
    out = []
    for i in range(len(row) // 2):
        x, y = int(row[2 * i]), int(row[2 * i + 1])
        c, s = rope_cs(pos, i)
        out += [_sat((x * c - y * s + 128) >> 8, 8), _sat((x * s + y * c + 128) >> 8, 8)]
    return out


def rope_float(x: np.ndarray, pos: Sequence[int]) -> np.ndarray:
    """Exact (float) RoPE, interleaved pairs, for the FP32 reference."""
    x = np.array(x, dtype=np.float64)
    d = x.shape[1]
    th = 10000.0 ** (-2 * np.arange(d // 2) / d)
    for r, p in enumerate(pos):
        a = p * th
        c, s = np.cos(a), np.sin(a)
        x0, x1 = x[r, 0::2].copy(), x[r, 1::2].copy()
        x[r, 0::2] = x0 * c - x1 * s
        x[r, 1::2] = x0 * s + x1 * c
    return x


def rope_quantized(m: np.ndarray, e: np.ndarray, pos: Sequence[int]):
    """What the chip stores for rows written with RoPE on: rotated elements,
    same exponents."""
    out = np.array([rope_row(m[r], pos[r]) for r in range(m.shape[0])], dtype=np.int64)
    return out, np.asarray(e, dtype=np.int64)


def random_case(rng: np.random.Generator, n=16, d=16, scale=1.0, outliers=False):
    qf = rng.standard_normal((n, d)) * scale
    kf = rng.standard_normal((n, d)) * scale
    vf = rng.standard_normal((n, d))
    if outliers:
        kf[rng.integers(0, n, 2)] *= 8
        vf[rng.integers(0, n, 2)] *= 16
    return qf, kf, vf


if __name__ == "__main__":
    rng = np.random.default_rng(1)
    for fused in (False, True):
        for mode in (MODE_EXACT, MODE_MF1, MODE_MF2):
            cos = []
            for _ in range(20):
                qf, kf, vf = random_case(rng, scale=1.5)
                args = prepare(qf, kf, vf)
                r = attention(*args, fused=fused, mode=mode)
                got = mx_dequantize(r.o, r.eo)
                ref = reference(qf, kf, vf)
                cos.append(np.sum(got * ref) / np.linalg.norm(got) / np.linalg.norm(ref))
            print(f"fused={fused!s:5} mode={mode} mean cos={np.mean(cos):.5f} min={np.min(cos):.5f}")
    _check_decode_matches_fused(rng)
    print("decode with length N matches the fused pass row by row")


Writing model/raijin_model.py


In [16]:
%%writefile model/raijin_vec.py
"""Vectorised, bit-exact twin of the decode path of raijin_model.

`decode_causal` computes, for every query i of a sequence, the decode step
attention_decode(qrow=i, length=i+1): the online-softmax pass over keys 0..i
that Raijin runs per generated token. It is vectorised over any leading
dimensions (batch, head) and over queries, looping only over keys, so it can
stand in for the attention of a small language model. `check()` compares it
with raijin_model.attention_decode on random tensors, bit for bit.
"""
from __future__ import annotations

import numpy as np

import raijin_model as rm

_LUT = np.array(rm.EXP_LUT, dtype=np.int64)
_S1 = {rm.MODE_MF1: np.array(rm.MF1_S1), rm.MODE_MF2: np.array(rm.MF2_S1)}
_S2 = {rm.MODE_MF1: np.full(16, rm.NONE_CODE), rm.MODE_MF2: np.array(rm.MF2_S2)}


def _sat(x, bits):
    return np.clip(x, -(1 << (bits - 1)), (1 << (bits - 1)) - 1)


def _term(sh):
    return np.where(sh > 8, 0, 256 >> np.minimum(sh, 8))


def scores(q, eq, k, ek, offset=0):
    """Requantised Q7.4 scores, shape (..., Tq, Tk)."""
    dot = np.einsum("...id,...jd->...ij", q.astype(np.int64), k.astype(np.int64))
    sh = eq[..., :, None].astype(np.int64) + ek[..., None, :] - 262 + offset
    left = _sat(dot << np.clip(sh, 0, 15), 12)
    right = _sat(dot >> np.clip(-sh, 0, 31), 12)
    return np.where(sh >= 0, left, right)


def exp2(s, m, mode, thr=0):
    t = m * 16 - s
    kk, f = t >> 4, t & 15
    if mode == rm.MODE_EXACT:
        p = np.where(kk >= 9, 0, _LUT[f] >> np.minimum(kk, 9))
    else:
        s1, s2 = _S1[mode][f], _S2[mode][f]
        sa = np.minimum(kk + s1, rm.NONE_CODE)
        sb = np.where(s2 == rm.NONE_CODE, rm.NONE_CODE, np.minimum(kk + s2, rm.NONE_CODE))
        p = _term(sa) + _term(sb)
    if thr:
        p = np.where(kk >= thr, 0, p)
    return p


def _bitlen(y):
    mag = np.where(y < 0, ~y, y)
    nb = np.zeros(y.shape, dtype=np.int64)
    for b in range(48):
        nb = np.where((mag >> b) != 0, b + 1, nb)
    return nb


def decode_causal(q, eq, k, ek, v, ev, mode=rm.MODE_EXACT, thr=0, offset=0, stats=None, final_max=False):
    """q, k, v: int arrays (..., T, D); eq, ek, ev: (..., T). Returns (o, eo).
    final_max: start every row's running maximum at its final value (a score
    pass that finds the maximum before any weight is formed), so SKIPTHR is
    measured against the true maximum."""
    q, k, v = (np.asarray(a, dtype=np.int64) for a in (q, k, v))
    eq, ek, ev = (np.asarray(a, dtype=np.int64) for a in (eq, ek, ev))
    T, D = q.shape[-2], q.shape[-1]
    S = scores(q, eq, k, ek, offset)                       # (..., T, T)
    emax = np.maximum.accumulate(ev, axis=-1)              # e_vmax over keys 0..i
    lead = q.shape[:-2]
    m = np.full(lead + (T,), rm.M_INIT, dtype=np.int64)
    if final_max:
        c = np.where(np.tril(np.ones((T, T), dtype=bool)), (S + 15) >> 4, rm.M_INIT)
        m = np.maximum(c.max(axis=-1), rm.M_INIT)
    l = np.zeros(lead + (T,), dtype=np.int64)
    acc = np.zeros(lead + (T, D), dtype=np.int64)
    qi = np.arange(T)
    zeros = 0
    lwrap = 0
    for j in range(T):
        live = qi >= j                                     # queries that see key j
        s = S[..., :, j]
        mn = np.maximum(m, (s + 15) >> 4)
        delta = mn - m
        m = np.where(live, mn, m)
        p = exp2(s, m, mode, thr)
        use = live & (p != 0)
        zeros += int(np.sum(live & (p == 0)))
        dsh = np.minimum(delta, 31)
        lraw = (l >> dsh) + p
        lwrap += int(np.sum(use & (lraw >= (1 << 14))))  # the 14-bit l register would wrap
        l = np.where(use, lraw & 0x3FFF, l)
        dv = np.minimum(emax - ev[..., j:j + 1], 31)       # (..., T)
        term = (p[..., None] * v[..., None, j, :]) >> dv[..., None]
        acc = np.where(use[..., None], (acc >> dsh[..., None]) + term, acc)
    if stats is not None:
        stats["zero_p"] = stats.get("zero_p", 0) + zeros
        stats["pairs"] = stats.get("pairs", 0) + int(np.prod(lead)) * T * (T + 1) // 2
        stats["l_overflow"] = stats.get("l_overflow", 0) + lwrap
        stats["acc_overflow"] = stats.get("acc_overflow", 0) + int(np.sum((acc < -(1 << 23)) | (acc >= (1 << 23))))
    r = (1 << 19) // np.maximum(l, 1)
    y = acc * r[..., None]
    sh = np.maximum(0, _bitlen(y).max(axis=-1) - 7)        # (..., T)
    rnd = np.where(sh[..., None] > 0, (y + (1 << np.maximum(sh - 1, 0))[..., None]) >> sh[..., None], y)
    o = _sat(rnd, 8)
    eo = np.clip(emax + sh - 19, 0, 255)
    return o, eo


def check(trials=20, seed=0):
    rng = np.random.default_rng(seed)
    for t in range(trials):
        T = int(rng.integers(1, 33))
        qf, kf, vf = rm.random_case(rng, n=T, d=16, scale=float(rng.uniform(0.3, 3)), outliers=bool(t % 2))
        prep = rm.prepare_kv4 if t % 3 == 2 else rm.prepare
        q, eq, k, ek, v, ev = prep(qf, kf, vf)
        for mode in (rm.MODE_EXACT, rm.MODE_MF1, rm.MODE_MF2):
            thr = [0, 4, 6][t % 3]
            o, eo = decode_causal(q, eq, k, ek, v, ev, mode=mode, thr=thr)
            for i in range(T):
                r = rm.attention_decode(q, eq, k, ek, v, ev, qrow=i, length=i + 1, mode=mode, thr=thr)
                assert np.array_equal(o[i], r.o[0]) and eo[i] == r.eo[0], (t, mode, i)
    return True


if __name__ == "__main__":
    check()
    print("raijin_vec matches raijin_model.attention_decode")


Writing model/raijin_vec.py


Self-check of the golden model against FP32 attention (cosine similarity per mode):

In [17]:
!python3 model/raijin_model.py

fused=False mode=0 mean cos=0.99976 min=0.99956
fused=False mode=1 mean cos=0.99255 min=0.98423


fused=False mode=2 mean cos=0.99921 min=0.99880


fused=True  mode=0 mean cos=0.99982 min=0.99972


fused=True  mode=1 mean cos=0.99390 min=0.98972


fused=True  mode=2 mean cos=0.99911 min=0.99856
decode with length N matches the fused pass row by row


## 5. Testbench (`tb/`)

cocotb tests driven through Icarus Verilog:
- `test_blocks.py` — each arithmetic block alone, exhaustively or with random vectors, against the model;
- `test_core.py` — 19 tests of the core over its byte bus: all dataflows and exp modes, decode with KV cache, causal prefill, multi-query, streaming sinks, INT4/FP4 caches, max-first decode, the 32-key overflow test for the 14-bit sum, key skip, RoPE, paging, token window, mixed precision, a real TinyStories-8M attention head, and the heavy-hitter tracker;
- `test_chip.py` — the chip top through its SPI/QSPI pins;
- `run_tests.py` — builds and runs each suite.

In [18]:
%%writefile tb/test_blocks.py
"""Block-level cocotb tests. The toplevel is chosen by the runner (TOPLEVEL env);
each test skips itself unless it matches the current toplevel."""
import os
import random
import sys

import cocotb
from cocotb.clock import Clock
from cocotb.triggers import ClockCycles, RisingEdge, Timer

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "model"))
import raijin_model as rm  # noqa: E402

TOP = os.environ.get("COCOTB_TOPLEVEL", os.environ.get("TOPLEVEL", ""))


def s8(x):
    return x & 0xFF


def pack(vals, w):
    out = 0
    for i, v in enumerate(vals):
        out |= (v & ((1 << w) - 1)) << (i * w)
    return out


def signed(v, w):
    v = int(v) & ((1 << w) - 1)
    return v - (1 << w) if v >> (w - 1) else v


@cocotb.test(skip=TOP != "raijin_exp2")
async def test_exp2_sweep(dut):
    count = 0
    for mode in (0, 1, 2):
        for m in range(-128, 129, 3):
            for s in range(max(-2048, m * 16 - 2600), min(2047, m * 16) + 1, 7):
                dut.s.value = s & 0xFFF
                dut.m.value = m & 0x1FF
                dut.mode.value = mode
                await Timer(1, unit="ns")
                p, sa, sb = rm.exp2(s, m, mode)
                assert int(dut.p.value) == p, (mode, s, m)
                if mode:
                    assert int(dut.sa.value) == sa and int(dut.sb.value) == sb, (mode, s, m)
                count += 1
    dut._log.info(f"exp2: {count} points match")


@cocotb.test(skip=TOP != "raijin_requant")
async def test_requant_sweep(dut):
    rnd = random.Random(3)
    for _ in range(4000):
        dot = rnd.randint(-(1 << 19), (1 << 19) - 1) >> rnd.randint(0, 19)
        es = rnd.randint(200, 320)
        off = rnd.randint(-128, 127) if rnd.random() < 0.3 else rnd.randint(-4, 4)
        dut.dot.value = dot & 0xFFFFF
        dut.esum.value = es
        dut.offset.value = off & 0xFF
        await Timer(1, unit="ns")
        assert signed(dut.s.value, 12) == rm.requant(dot, es, off), (dot, es, off)


@cocotb.test(skip=TOP != "raijin_dpu")
async def test_dpu_random(dut):
    cocotb.start_soon(Clock(dut.clk, 20, unit="ns").start())
    dut.rst_n.value = 0
    dut.in_valid.value = 0
    await ClockCycles(dut.clk, 2)
    dut.rst_n.value = 1
    rnd = random.Random(5)
    vecs = []
    for i in range(200):
        a = [rnd.randint(-128, 127) for _ in range(16)]
        b = [rnd.randint(-128, 127) for _ in range(16)]
        if i < 2:
            a = [-128] * 16
            b = [-128 if i == 0 else 127] * 16
        vecs.append((a, b, rnd.randint(0, 255), rnd.randint(0, 255)))
    got = []
    for idx in range(len(vecs) + 3):
        if idx < len(vecs):
            a, b, ea, eb = vecs[idx]
            dut.a.value = pack(a, 8)
            dut.b.value = pack(b, 8)
            dut.ea.value = ea
            dut.eb.value = eb
            dut.in_valid.value = 1
            dut.in_tag.value = idx & 0xFF
        else:
            dut.in_valid.value = 0
        await RisingEdge(dut.clk)
        await Timer(1, unit="ns")
        if int(dut.out_valid.value):
            got.append((int(dut.out_tag.value), signed(dut.dot.value, 20), int(dut.esum.value)))
    assert len(got) == len(vecs)
    for i, (tag, dot, es) in enumerate(got):
        a, b, ea, eb = vecs[i]
        assert tag == i & 0xFF
        assert dot == rm.dpu(a, b), i
        assert es == ea + eb


@cocotb.test(skip=TOP != "raijin_out")
async def test_out_random(dut):
    cocotb.start_soon(Clock(dut.clk, 20, unit="ns").start())
    dut.rst_n.value = 0
    dut.in_valid.value = 0
    await ClockCycles(dut.clk, 2)
    dut.rst_n.value = 1
    rnd = random.Random(11)
    vecs = []
    for _ in range(150):
        mag = rnd.randint(1, 23)
        acc = [rnd.randint(-(1 << mag), (1 << mag) - 1) for _ in range(16)]
        if rnd.random() < 0.1:
            acc = [0] * 16
        r = rnd.randint(128, 4096)
        emax = rnd.randint(0, 255)
        vecs.append((acc, r, emax))
    got = []
    for idx in range(len(vecs) + 4):
        if idx < len(vecs):
            acc, r, emax = vecs[idx]
            dut.acc.value = pack(acc, 24)
            dut.r.value = pack([r] * 4, 13)   # RG = 4 identical copies of 1/l
            dut.emax.value = emax
            dut.in_tag.value = idx & 0xFF
            dut.in_valid.value = 1
        else:
            dut.in_valid.value = 0
        await RisingEdge(dut.clk)
        await Timer(1, unit="ns")
        if int(dut.out_valid.value):
            o = int(dut.o.value)
            got.append(([signed(o >> (8 * c), 8) for c in range(16)], int(dut.eo.value)))
    assert len(got) == len(vecs)
    for i, (o, eo) in enumerate(got):
        exp_o, exp_e = rm.out_row(*vecs[i])
        assert o == exp_o and eo == exp_e, (i, o, exp_o, eo, exp_e)


@cocotb.test(skip=TOP != "raijin_softmax_lane")
async def test_lane_online_and_recip(dut):
    """Drive one lane in online mode and compare m, l and 1/l with the model."""
    cocotb.start_soon(Clock(dut.clk, 20, unit="ns").start())
    dut.rst_n.value = 0
    for sig in (dut.clear, dut.s_valid, dut.m_update, dut.emit, dut.recip_start, dut.first, dut.last, dut.mask, dut.thr):
        sig.value = 0
    dut.mode.value = 0
    dut.tag.value = 0
    dut.s.value = 0
    await ClockCycles(dut.clk, 2)
    dut.rst_n.value = 1
    rnd = random.Random(21)
    for trial in range(40):
        mode = trial % 3
        n = 16
        scores = [rnd.randint(-2048, 2047) >> rnd.randint(0, 6) for _ in range(n)]
        dut.mode.value = mode
        dut.clear.value = 1
        await RisingEdge(dut.clk)
        dut.clear.value = 0
        dut.m_update.value = 1
        dut.emit.value = 1
        for s in scores:
            dut.s.value = s & 0xFFF
            dut.s_valid.value = 1
            await RisingEdge(dut.clk)
        dut.s_valid.value = 0
        await ClockCycles(dut.clk, 3)
        # model
        m, lsum = rm.M_INIT, 0
        for s in scores:
            mn = max(m, rm.ceil_int(s))
            dl = mn - m
            m = mn
            p, _, _ = rm.exp2(s, m, mode)
            lsum = (lsum >> min(dl, 31)) + p
        assert signed(dut.m.value, 9) == m
        assert int(dut.l.value) == lsum, (trial, int(dut.l.value), lsum)
        dut.recip_start.value = 1
        await RisingEdge(dut.clk)
        dut.recip_start.value = 0
        for _ in range(20):
            await RisingEdge(dut.clk)
            await Timer(1, unit="ns")
            if int(dut.r_valid.value):
                break
        assert int(dut.r.value) == rm.recip(lsum)


@cocotb.test(skip=TOP != "raijin_rope")
async def test_rope_rotator(dut):
    """The RoPE rotator against rm.rope_row: every position 0..255, every pair,
    random element values plus the extremes."""
    cocotb.start_soon(Clock(dut.clk, 10, unit="ns").start())
    dut.we_even.value = 0
    await ClockCycles(dut.clk, 2)
    rng = random.Random(5)
    n = 0
    for pos in range(256):
        for pair in range(8):
            xs = [(-128, -128), (127, -128), (127, 127)] if pos % 64 == 0 else []
            xs += [(rng.randint(-128, 127), rng.randint(-128, 127)) for _ in range(2)]
            for x, y in xs:
                dut.pos.value = pos
                dut.pair.value = pair
                dut.we_even.value = 1
                dut.wdata.value = s8(x)
                await RisingEdge(dut.clk)
                dut.we_even.value = 0
                dut.wdata.value = s8(y)
                await Timer(1, unit="ns")
                c, s = rm.rope_cs(pos, pair)
                ex = rm._sat((x * c - y * s + 128) >> 8, 8)
                ey = rm._sat((x * s + y * c + 128) >> 8, 8)
                assert (signed(dut.xo.value, 8), signed(dut.yo.value, 8)) == (ex, ey), (pos, pair, x, y)
                n += 1
    dut._log.info(f"rope rotator: {n} pairs match")


Writing tb/test_blocks.py


In [19]:
%%writefile tb/test_core.py
"""Full-pass cocotb tests: the RTL core must match the golden model bit for bit."""
import os
import sys

import cocotb
import numpy as np
from cocotb.clock import Clock
from cocotb.triggers import ClockCycles, RisingEdge, ReadOnly

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "model"))
import raijin_model as rm  # noqa: E402

N = int(os.environ.get("RAIJIN_N", 16))
P = int(os.environ.get("RAIJIN_P", 4))
D = int(os.environ.get("RAIJIN_D", 16))
RD_LATENCY = 3   # rdata is valid after the third rising edge following re


class Bus:
    def __init__(self, dut):
        self.dut = dut

    async def write(self, addr, data):
        d = self.dut
        d.addr.value = addr
        d.wdata.value = data & 0xFF
        d.we.value = 1
        await RisingEdge(d.clk)
        d.we.value = 0

    async def read(self, addr):
        d = self.dut
        d.addr.value = addr
        d.re.value = 1
        await RisingEdge(d.clk)
        d.re.value = 0
        await ClockCycles(d.clk, RD_LATENCY - 1)
        await ReadOnly()
        val = int(d.rdata.value)
        await RisingEdge(d.clk)
        return val

    async def read32(self, addr):
        b = [await self.read(addr + i) for i in range(4)]
        return b[0] | b[1] << 8 | b[2] << 16 | b[3] << 24


async def setup(dut):
    cocotb.start_soon(Clock(dut.clk, 20, unit="ns").start())
    dut.rst_n.value = 0
    dut.we.value = 0
    dut.re.value = 0
    dut.addr.value = 0
    dut.wdata.value = 0
    await ClockCycles(dut.clk, 3)
    dut.rst_n.value = 1
    await ClockCycles(dut.clk, 2)
    return Bus(dut)


async def load(bus, q, eq, k, ek, v, ev):
    for base, mat in ((0x0000, q), (0x1000, k), (0x2000, v)):
        for r in range(N):
            for c in range(D):
                await bus.write(base + r * D + c, int(mat[r][c]))
    for base, ex in ((0x4000, eq), (0x4100, ek), (0x4200, ev)):
        for r in range(N):
            await bus.write(base + r, int(ex[r]))


async def run_pass(bus, fused, mode, offset=0, causal=False):
    dut = bus.dut
    await bus.write(0x5001, offset)
    await bus.write(0x5000, 1 | (int(fused) << 1) | (mode << 2) | (int(causal) << 5))
    await wait_done(dut)
    return await read_out(bus)


async def wait_done(dut):
    # the start write lands two edges after it is presented; let done drop first
    await ClockCycles(dut.clk, 3)
    for _ in range(5000):
        await RisingEdge(dut.clk)
        if int(dut.done.value):
            break
    else:
        raise AssertionError("pass did not finish")


async def read_out(bus, rows=None):
    rows = range(N) if rows is None else rows
    o = np.zeros((N, D), dtype=np.int64)
    eo = np.zeros(N, dtype=np.int64)
    for r in rows:
        for c in range(D):
            b = await bus.read(0x3000 + r * D + c)
            o[r][c] = b - 256 if b & 0x80 else b
        eo[r] = await bus.read(0x4300 + r)
    cycles = await bus.read32(0x5004)
    skips = await bus.read32(0x5008)
    return o, eo, cycles, skips


CASES = [
    ("random", dict(scale=1.0)),
    ("sharp", dict(scale=2.5)),
    ("outliers", dict(scale=1.5, outliers=True)),
]


@cocotb.test()
async def test_ids(dut):
    bus = await setup(dut)
    assert await bus.read(0x500C) == N
    assert await bus.read(0x500D) == D
    assert await bus.read(0x500F) == 2
    assert await bus.read(0x5003) == RD_LATENCY


@cocotb.test()
async def test_all_modes(dut):
    """Every case, both dataflows, all three exponent modes, bit-exact."""
    bus = await setup(dut)
    rng = np.random.default_rng(int(os.environ.get("RAIJIN_SEED", 7)))
    summary = []
    for name, kw in CASES:
        qf, kf, vf = rm.random_case(rng, n=N, d=D, **kw)
        args = rm.prepare(qf, kf, vf)
        await load(bus, *args)
        ref = rm.reference(qf, kf, vf)
        for fused in (False, True):
            for mode in (rm.MODE_EXACT, rm.MODE_MF1, rm.MODE_MF2):
                exp = rm.attention(*args, fused=fused, mode=mode)
                o, eo, cycles, skips = await run_pass(bus, fused, mode)
                tag = f"{name} fused={int(fused)} mode={mode}"
                assert np.array_equal(o, exp.o), f"{tag}: output mismatch\nrtl=\n{o}\nmodel=\n{exp.o}"
                assert np.array_equal(eo, exp.eo), f"{tag}: exponent mismatch rtl={eo} model={exp.eo}"
                assert skips == exp.skips, f"{tag}: skips rtl={skips} model={exp.skips}"
                got = rm.mx_dequantize(o, eo)
                cos = float(np.sum(got * ref) / np.linalg.norm(got) / np.linalg.norm(ref))
                summary.append(f"{tag:34s} cycles={cycles:4d} skips={skips:3d} cos={cos:.5f}")
                dut._log.info(summary[-1])
    with open(os.path.join(os.path.dirname(__file__), "..", "results", f"core_N{N}_D{D}.txt"), "w") as f:
        f.write("\n".join(summary) + "\n")


@cocotb.test()
async def test_offset_and_reuse(dut):
    """A non-zero requant offset, and back-to-back passes without reloading."""
    bus = await setup(dut)
    rng = np.random.default_rng(99)
    qf, kf, vf = rm.random_case(rng, n=N, d=D, scale=0.7)
    args = rm.prepare(qf, kf, vf)
    await load(bus, *args)
    for off in (2, -3):
        for fused in (True, False):
            exp = rm.attention(*args, fused=fused, mode=rm.MODE_MF2, offset=off)
            o, eo, _, skips = await run_pass(bus, fused, rm.MODE_MF2, offset=off)
            assert np.array_equal(o, exp.o), f"off={off} fused={fused}\n{o}\n{exp.o}"
            assert np.array_equal(eo, exp.eo), f"off={off} fused={fused} {eo} {exp.eo}"
            assert skips == exp.skips


@cocotb.test()
async def test_random_regression(dut):
    """Many random tensors with random exponents spread, fused and unfused, exact and MF2."""
    bus = await setup(dut)
    rng = np.random.default_rng(1234)
    iters = int(os.environ.get("RAIJIN_ITERS", 6))
    for it in range(iters):
        scale = float(rng.uniform(0.3, 4.0))
        qf, kf, vf = rm.random_case(rng, n=N, d=D, scale=scale, outliers=bool(it % 2))
        args = rm.prepare(qf, kf, vf)
        await load(bus, *args)
        for fused in (False, True):
            mode = int(rng.integers(0, 3))
            exp = rm.attention(*args, fused=fused, mode=mode)
            o, eo, _, skips = await run_pass(bus, fused, mode)
            assert np.array_equal(o, exp.o), f"iter {it} fused={fused} mode={mode}"
            assert np.array_equal(eo, exp.eo)
            assert skips == exp.skips


@cocotb.test()
async def test_decode_kv_cache(dut):
    """Autoregressive decode with a KV cache: each step writes only the new
    token's Q, K and V rows, then one query attends to every cached key."""
    bus = await setup(dut)
    rng = np.random.default_rng(int(os.environ.get("RAIJIN_SEED", 7)) + 5)
    summary = []
    for name, kw in CASES:
        qf, kf, vf = rm.random_case(rng, n=N, d=D, **kw)
        q, eq, k, ek, v, ev = rm.prepare(qf, kf, vf)
        ref = rm.reference_causal(qf, kf, vf)
        mode = rm.MODE_EXACT if name != "sharp" else rm.MODE_MF2
        for t in range(N):
            for base, mat in ((0x0000, q), (0x1000, k), (0x2000, v)):
                for c in range(D):
                    await bus.write(base + t * D + c, int(mat[t][c]))
            for base, ex in ((0x4000, eq), (0x4100, ek), (0x4200, ev)):
                await bus.write(base + t, int(ex[t]))
            await bus.write(0x5010, t + 1)
            await bus.write(0x5011, t)
            await bus.write(0x5000, 1 | (mode << 2) | (1 << 4))
            await wait_done(dut)
            o, eo, cycles, skips = await read_out(bus, rows=[t])
            exp = rm.attention_decode(q, eq, k, ek, v, ev, qrow=t, length=t + 1, mode=mode)
            assert np.array_equal(o[t], exp.o[0]), f"{name} t={t}: rtl={o[t]} model={exp.o[0]}"
            assert eo[t] == exp.eo[0], f"{name} t={t}: e rtl={eo[t]} model={exp.eo[0]}"
            assert skips == exp.skips, f"{name} t={t}: skips rtl={skips} model={exp.skips}"
            got = rm.mx_dequantize(o[t:t + 1], eo[t:t + 1])[0]
            cos = float(np.dot(got, ref[t]) / np.linalg.norm(got) / np.linalg.norm(ref[t]))
            if t in (0, N // 2 - 1, N - 1):
                summary.append(f"decode {name:8s} mode={mode} len={t + 1:2d} cycles={cycles:4d} cos={cos:.5f}")
                dut._log.info(summary[-1])
    with open(os.path.join(os.path.dirname(__file__), "..", "results", f"decode_N{N}_D{D}.txt"), "w") as f:
        f.write(chr(10).join(summary) + chr(10))


@cocotb.test()
async def test_causal_prefill(dut):
    """Causal prefill in one fused pass: row i attends to keys 0..i. Bit-exact
    with the model, compared with FP32 causal attention, and timed against
    building the same outputs with N decode steps."""
    bus = await setup(dut)
    rng = np.random.default_rng(int(os.environ.get("RAIJIN_SEED", 7)) + 11)
    summary = []
    for name, kw in CASES:
        qf, kf, vf = rm.random_case(rng, n=N, d=D, **kw)
        args = rm.prepare(qf, kf, vf)
        await load(bus, *args)
        ref = rm.reference_causal(qf, kf, vf)
        for mode in (rm.MODE_EXACT, rm.MODE_MF1, rm.MODE_MF2):
            for off in (0, 2):
                exp = rm.attention(*args, fused=True, mode=mode, offset=off, causal=True)
                o, eo, cycles, skips = await run_pass(bus, True, mode, offset=off, causal=True)
                tag = f"causal {name} mode={mode} off={off}"
                assert np.array_equal(o, exp.o), f"{tag}: output mismatch\nrtl=\n{o}\nmodel=\n{exp.o}"
                assert np.array_equal(eo, exp.eo), f"{tag}: exponent mismatch rtl={eo} model={exp.eo}"
                assert skips == exp.skips, f"{tag}: skips rtl={skips} model={exp.skips}"
                if off == 0:
                    got = rm.mx_dequantize(o, eo)
                    cos = float(np.mean([np.dot(got[i], ref[i]) / np.linalg.norm(got[i]) /
                                         np.linalg.norm(ref[i]) for i in range(N)]))
                    summary.append(f"{tag:30s} cycles={cycles:4d} skips={skips:3d} mean_row_cos={cos:.5f}")
                    dut._log.info(summary[-1])
        # a plain fused pass right after a causal one must be unaffected
        exp = rm.attention(*args, fused=True, mode=rm.MODE_EXACT)
        o, eo, _, _ = await run_pass(bus, True, rm.MODE_EXACT)
        assert np.array_equal(o, exp.o) and np.array_equal(eo, exp.eo), f"{name}: non-causal after causal"
    with open(os.path.join(os.path.dirname(__file__), "..", "results", f"causal_N{N}_D{D}.txt"), "w") as f:
        f.write(chr(10).join(summary) + chr(10))


@cocotb.test()
async def test_multi_query_decode(dut):
    """Multi-query decode: P query rows share one K/V stream in a single step.
    GQA (every row sees all LEN keys) and speculative verify (row r sees keys
    0..LEN-P+r), each bit-exact with the model, timed against P single-row steps."""
    bus = await setup(dut)
    rng = np.random.default_rng(int(os.environ.get("RAIJIN_SEED", 7)) + 23)
    summary = []
    for name, kw in CASES:
        qf, kf, vf = rm.random_case(rng, n=N, d=D, **kw)
        args = rm.prepare(qf, kf, vf)
        await load(bus, *args)
        q, eq, k, ek, v, ev = args
        mode = rm.MODE_EXACT if name != "sharp" else rm.MODE_MF2
        for spec in (False, True):
            for length in sorted({P, N // 2, N}):
                for group in (0, N // P - 1):
                    await bus.write(0x5010, length)
                    await bus.write(0x5011, group * P)
                    ctrl = 1 | (mode << 2) | (1 << 4) | (1 << 6) | (int(spec) << 7)
                    await bus.write(0x5000, ctrl)
                    await wait_done(dut)
                    rows = list(range(group * P, group * P + P))
                    o, eo, cycles, skips = await read_out(bus, rows=rows)
                    exp = rm.attention_decode_multi(q, eq, k, ek, v, ev, group=group, p=P,
                                                    length=length, spec=spec, mode=mode)
                    tag = f"{name} spec={int(spec)} len={length} group={group}"
                    assert np.array_equal(o[rows], exp.o), f"{tag}: rtl=\n{o[rows]}\nmodel=\n{exp.o}"
                    assert np.array_equal(eo[rows], exp.eo), f"{tag}: e rtl={eo[rows]} model={exp.eo}"
                    assert skips == exp.skips, f"{tag}: skips rtl={skips} model={exp.skips}"
                    if group == 0:
                        # the same P outputs one row per step, for the speed comparison
                        single = 0
                        for r in rows:
                            await bus.write(0x5010, length - P + 1 + (r - rows[0]) if spec else length)
                            await bus.write(0x5011, r)
                            await bus.write(0x5000, 1 | (mode << 2) | (1 << 4))
                            await wait_done(dut)
                            single += await bus.read32(0x5004)
                        summary.append(f"{tag:36s} multi={cycles:3d} cycles  {P} single steps={single:4d} cycles")
                        dut._log.info(summary[-1])
    with open(os.path.join(os.path.dirname(__file__), "..", "results", f"multi_decode_N{N}_D{D}.txt"), "w") as f:
        f.write(chr(10).join(summary) + chr(10))


@cocotb.test()
async def test_streaming_kv_cache(dut):
    """StreamingLLM-style generation past the cache size: 4 attention-sink rows
    stay pinned and the other N - 4 rows are a ring of the most recent tokens.
    Attention is order-free over keys, so the decode step needs no change; the
    host only picks the row to overwrite. 3N tokens, each step bit-exact."""
    bus = await setup(dut)
    rng = np.random.default_rng(int(os.environ.get("RAIJIN_SEED", 7)) + 31)
    sinks, total = 4, 3 * N
    qf, kf, vf = rm.random_case(rng, n=total, d=D, scale=1.5)
    q, eq, k, ek, v, ev = rm.prepare(qf, kf, vf)
    ref = rm.reference_streaming(qf, kf, vf, N, sinks)
    # mirror of the on-chip cache contents, for the bit-exact model
    mq, mk, mv = (np.zeros((N, D), dtype=np.int64) for _ in range(3))
    meq, mek, mev = (np.zeros(N, dtype=np.int64) for _ in range(3))
    coss = []
    for t in range(total):
        slot = rm.streaming_slot(t, N, sinks)
        for base, mat, mir in ((0x0000, q, mq), (0x1000, k, mk), (0x2000, v, mv)):
            mir[slot] = mat[t]
            for c in range(D):
                await bus.write(base + slot * D + c, int(mat[t][c]))
        for base, ex, mir in ((0x4000, eq, meq), (0x4100, ek, mek), (0x4200, ev, mev)):
            mir[slot] = ex[t]
            await bus.write(base + slot, int(ex[t]))
        length = min(t + 1, N)
        await bus.write(0x5010, length)
        await bus.write(0x5011, slot)
        await bus.write(0x5000, 1 | (1 << 4))
        await wait_done(dut)
        o, eo, cycles, skips = await read_out(bus, rows=[slot])
        exp = rm.attention_decode(mq, meq, mk, mek, mv, mev, qrow=slot, length=length)
        assert np.array_equal(o[slot], exp.o[0]), f"t={t} slot={slot}: rtl={o[slot]} model={exp.o[0]}"
        assert eo[slot] == exp.eo[0], f"t={t}: e rtl={eo[slot]} model={exp.eo[0]}"
        got = rm.mx_dequantize(o[slot:slot + 1], eo[slot:slot + 1])[0]
        coss.append(float(np.dot(got, ref[t]) / np.linalg.norm(got) / np.linalg.norm(ref[t])))
    line = (f"streaming N={N} sinks={sinks} tokens={total}: mean cos={np.mean(coss):.5f} "
            f"min cos={np.min(coss):.5f} (tokens past the cache: mean {np.mean(coss[N:]):.5f})")
    dut._log.info(line)
    with open(os.path.join(os.path.dirname(__file__), "..", "results", f"streaming_N{N}_D{D}.txt"), "w") as f:
        f.write(line + chr(10))


async def load_kv4(bus, q, eq, k, ek, v, ev):
    """Q as MXINT8 rows; K and V packed MXINT4, token t at byte 8t of its region."""
    for r in range(N):
        for c in range(D):
            await bus.write(0x0000 + r * D + c, int(q[r][c]))
        await bus.write(0x4000 + r, int(eq[r]))
    for base, mat in ((0x1000, k), (0x2000, v)):
        for t in range(mat.shape[0]):
            for b, byte in enumerate(rm.pack4(mat[t])):
                await bus.write(base + 8 * t + b, byte)
    for base, ex in ((0x4100, ek), (0x4200, ev)):
        for t in range(len(ex)):
            await bus.write(base + t, int(ex[t]))


@cocotb.test()
async def test_int4_kv_cache(dut):
    """MXINT4 K/V (KVFMT = 1): full passes on N tokens, then a 2N-token KV cache
    in the same SRAM, every step bit-exact. Accuracy is compared with the same
    steps run with an MXINT8 cache (model only, it cannot hold 2N tokens)."""
    bus = await setup(dut)
    await bus.write(0x5012, 3)
    assert await bus.read(0x5012) == 3
    rng = np.random.default_rng(int(os.environ.get("RAIJIN_SEED", 7)) + 41)
    summary = []
    # full passes with an INT4 K/V
    qf, kf, vf = rm.random_case(rng, n=N, d=D, scale=1.5)
    args = rm.prepare_kv4(qf, kf, vf)
    await load_kv4(bus, *args)
    ref = rm.reference(qf, kf, vf)
    for fused, causal in ((False, False), (True, False), (True, True)):
        exp = rm.attention(*args, fused=fused, mode=rm.MODE_EXACT, causal=causal)
        o, eo, cycles, skips = await run_pass(bus, fused, rm.MODE_EXACT, causal=causal)
        tag = f"kv4 full fused={int(fused)} causal={int(causal)}"
        assert np.array_equal(o, exp.o) and np.array_equal(eo, exp.eo), f"{tag}: mismatch"
        assert skips == exp.skips
        if not causal:
            got = rm.mx_dequantize(o, eo)
            cos = float(np.sum(got * ref) / np.linalg.norm(got) / np.linalg.norm(ref))
            summary.append(f"{tag:34s} cycles={cycles:4d} cos={cos:.5f}")
            dut._log.info(summary[-1])
    # a 2N-token KV cache
    T = 2 * N
    for name, kw in CASES:
        qf, kf, vf = rm.random_case(rng, n=T, d=D, **kw)
        q, eq, k, ek, v, ev = rm.prepare_kv4(qf, kf, vf)
        q8, eq8, k8, ek8, v8, ev8 = rm.prepare(qf, kf, vf)
        ref = rm.reference_causal(qf, kf, vf)
        cos4, cos8 = [], []
        mq = np.zeros((N, D), dtype=np.int64)
        meq = np.zeros(N, dtype=np.int64)
        for t in range(T):
            row = t % N
            mq[row], meq[row] = q[t], eq[t]
            for c in range(D):
                await bus.write(0x0000 + row * D + c, int(q[t][c]))
            await bus.write(0x4000 + row, int(eq[t]))
            for base, mat in ((0x1000, k), (0x2000, v)):
                for b, byte in enumerate(rm.pack4(mat[t])):
                    await bus.write(base + 8 * t + b, byte)
            await bus.write(0x4100 + t, int(ek[t]))
            await bus.write(0x4200 + t, int(ev[t]))
            await bus.write(0x5010, t + 1)
            await bus.write(0x5011, row)
            await bus.write(0x5000, 1 | (1 << 4))
            await wait_done(dut)
            o, eo, cycles, _ = await read_out(bus, rows=[row])
            exp = rm.attention_decode(mq, meq, k, ek, v, ev, qrow=row, length=t + 1)
            assert np.array_equal(o[row], exp.o[0]), f"{name} t={t}: rtl={o[row]} model={exp.o[0]}"
            assert eo[row] == exp.eo[0]
            got = rm.mx_dequantize(o[row:row + 1], eo[row:row + 1])[0]
            cos4.append(float(np.dot(got, ref[t]) / np.linalg.norm(got) / np.linalg.norm(ref[t])))
            e8 = rm.attention_decode(q8, eq8, k8, ek8, v8, ev8, qrow=t, length=t + 1)
            g8 = rm.mx_dequantize(e8.o, e8.eo)[0]
            cos8.append(float(np.dot(g8, ref[t]) / np.linalg.norm(g8) / np.linalg.norm(ref[t])))
            if t == T - 1:
                last_cycles = cycles
        summary.append(f"kv4 decode {name:8s} tokens={T} cycles@len{T}={last_cycles:3d} "
                       f"mean cos INT4={np.mean(cos4):.5f} (INT8 cache, model={np.mean(cos8):.5f})")
        dut._log.info(summary[-1])
    # K in MXINT8, V in MXINT4 (KVFMT = 2): the accuracy-first setting
    qf, kf, vf = rm.random_case(rng, n=N, d=D, scale=2.5)
    q, eq, k, ek, _, _ = rm.prepare(qf, kf, vf)
    _, _, _, _, v, ev = rm.prepare_kv4(qf, kf, vf)
    await bus.write(0x5012, 2)
    for r in range(N):
        for c in range(D):
            await bus.write(0x0000 + r * D + c, int(q[r][c]))
            await bus.write(0x1000 + r * D + c, int(k[r][c]))
        for b, byte in enumerate(rm.pack4(v[r])):
            await bus.write(0x2000 + 8 * r + b, byte)
        for base, ex in ((0x4000, eq), (0x4100, ek), (0x4200, ev)):
            await bus.write(base + r, int(ex[r]))
    exp = rm.attention(q, eq, k, ek, v, ev, fused=True, mode=rm.MODE_EXACT, causal=True)
    o, eo, _, _ = await run_pass(bus, True, rm.MODE_EXACT, causal=True)
    assert np.array_equal(o, exp.o) and np.array_equal(eo, exp.eo), "K8/V4 causal pass mismatch"
    await bus.write(0x5012, 0)
    with open(os.path.join(os.path.dirname(__file__), "..", "results", f"kv4_N{N}_D{D}.txt"), "w") as f:
        f.write(chr(10).join(summary) + chr(10))


def _cos(a, b):
    return float(np.dot(a, b) / np.linalg.norm(a) / np.linalg.norm(b))


@cocotb.test()
async def test_fp4_kv_cache(dut):
    """MXFP4 K/V (KVFMT bits 2, 3): E2M1 nibbles in the same packed layout as
    MXINT4. Full passes and a 2N-token decode cache are bit-exact with the
    model; accuracy is compared with MXINT4 and MXINT8 caches run on the model.
    Then K8/V-FP4, and FP4 pages in mixed-precision paging."""
    bus = await setup(dut)
    await bus.write(0x5012, 0xF)
    assert await bus.read(0x5012) == 0xF
    rng = np.random.default_rng(int(os.environ.get("RAIJIN_SEED", 7)) + 97)
    summary = []

    async def load_kvf(q, eq, k, ek, v, ev):
        for r in range(N):
            for c in range(D):
                await bus.write(0x0000 + r * D + c, int(q[r][c]))
            await bus.write(0x4000 + r, int(eq[r]))
        for base, mat in ((0x1000, k), (0x2000, v)):
            for t in range(mat.shape[0]):
                for b, byte in enumerate(rm.pack_fp4(mat[t])):
                    await bus.write(base + 8 * t + b, byte)
        for base, ex in ((0x4100, ek), (0x4200, ev)):
            for t in range(len(ex)):
                await bus.write(base + t, int(ex[t]))

    qf, kf, vf = rm.random_case(rng, n=N, d=D, scale=1.5)
    args = rm.prepare_kvfp4(qf, kf, vf)
    a4 = rm.prepare_kv4(qf, kf, vf)
    await load_kvf(*args)
    ref = rm.reference(qf, kf, vf)
    for fused, causal in ((False, False), (True, False), (True, True)):
        exp = rm.attention(*args, fused=fused, mode=rm.MODE_EXACT, causal=causal)
        o, eo, cycles, skips = await run_pass(bus, fused, rm.MODE_EXACT, causal=causal)
        tag = f"fp4 full fused={int(fused)} causal={int(causal)}"
        assert np.array_equal(o, exp.o) and np.array_equal(eo, exp.eo), f"{tag}: mismatch"
        assert skips == exp.skips
        if not causal:
            got = rm.mx_dequantize(o, eo)
            e4 = rm.attention(*a4, fused=fused, mode=rm.MODE_EXACT)
            g4 = rm.mx_dequantize(e4.o, e4.eo)
            cos = float(np.sum(got * ref) / np.linalg.norm(got) / np.linalg.norm(ref))
            cos4 = float(np.sum(g4 * ref) / np.linalg.norm(g4) / np.linalg.norm(ref))
            summary.append(f"{tag:34s} cycles={cycles:4d} cos FP4={cos:.5f} (INT4, model={cos4:.5f})")
            dut._log.info(summary[-1])
    # a 2N-token KV cache
    T = 2 * N
    for name, kw in CASES:
        qf, kf, vf = rm.random_case(rng, n=T, d=D, **kw)
        q, eq, k, ek, v, ev = rm.prepare_kvfp4(qf, kf, vf)
        _, _, k4, ek4, v4, ev4 = rm.prepare_kv4(qf, kf, vf)
        q8, eq8, k8, ek8, v8, ev8 = rm.prepare(qf, kf, vf)
        ref = rm.reference_causal(qf, kf, vf)
        cf, c4, c8 = [], [], []
        mq = np.zeros((N, D), dtype=np.int64)
        meq = np.zeros(N, dtype=np.int64)
        for t in range(T):
            row = t % N
            mq[row], meq[row] = q[t], eq[t]
            for c in range(D):
                await bus.write(0x0000 + row * D + c, int(q[t][c]))
            await bus.write(0x4000 + row, int(eq[t]))
            for base, mat in ((0x1000, k), (0x2000, v)):
                for b, byte in enumerate(rm.pack_fp4(mat[t])):
                    await bus.write(base + 8 * t + b, byte)
            await bus.write(0x4100 + t, int(ek[t]))
            await bus.write(0x4200 + t, int(ev[t]))
            await bus.write(0x5010, t + 1)
            await bus.write(0x5011, row)
            await bus.write(0x5000, 1 | (1 << 4))
            await wait_done(dut)
            o, eo, cycles, _ = await read_out(bus, rows=[row])
            exp = rm.attention_decode(mq, meq, k, ek, v, ev, qrow=row, length=t + 1)
            assert np.array_equal(o[row], exp.o[0]), f"{name} t={t}: rtl={o[row]} model={exp.o[0]}"
            assert eo[row] == exp.eo[0]
            cf.append(_cos(rm.mx_dequantize(o[row:row + 1], eo[row:row + 1])[0], ref[t]))
            for kk, ekk, vv, evv, lst in ((k4, ek4, v4, ev4, c4), (k8, ek8, v8, ev8, c8)):
                e = rm.attention_decode(q8, eq8, kk, ekk, vv, evv, qrow=t, length=t + 1)
                lst.append(_cos(rm.mx_dequantize(e.o, e.eo)[0], ref[t]))
        summary.append(f"fp4 decode {name:8s} tokens={T} cycles@len{T}={cycles:3d} mean cos FP4={np.mean(cf):.5f} "
                       f"(model: INT4={np.mean(c4):.5f}, INT8={np.mean(c8):.5f}) min FP4={np.min(cf):.5f} INT4={np.min(c4):.5f}")
        dut._log.info(summary[-1])
    # K in MXINT8, V in MXFP4 (KVFMT = 0b1010)
    qf, kf, vf = rm.random_case(rng, n=N, d=D, scale=2.5)
    q, eq, k, ek, _, _ = rm.prepare(qf, kf, vf)
    _, _, _, _, v, ev = rm.prepare_kvfp4(qf, kf, vf)
    await bus.write(0x5012, 0xA)
    for r in range(N):
        for c in range(D):
            await bus.write(0x0000 + r * D + c, int(q[r][c]))
            await bus.write(0x1000 + r * D + c, int(k[r][c]))
        for b, byte in enumerate(rm.pack_fp4(v[r])):
            await bus.write(0x2000 + 8 * r + b, byte)
        for base, ex in ((0x4000, eq), (0x4100, ek), (0x4200, ev)):
            await bus.write(base + r, int(ex[r]))
    for mode in (rm.MODE_EXACT, rm.MODE_MF2):
        exp = rm.attention(q, eq, k, ek, v, ev, fused=True, mode=mode, causal=True)
        o, eo, _, _ = await run_pass(bus, True, mode, causal=True)
        assert np.array_equal(o, exp.o) and np.array_equal(eo, exp.eo), f"K8/V-FP4 mode {mode} mismatch"
    # mixed paging: INT8 sink page and last page, the rest MXFP4 (KVFMT bits 2, 3)
    await bus.write(0x5012, 0xC)
    T, NP, int8 = 24, 6, {0, 5}
    qf, kf, vf = rm.random_case(rng, n=T, d=D, scale=1.5)
    q, eq, k8, ek8, v8, ev8 = rm.prepare(qf, kf, vf)
    _, _, kp, ekp, vp, evp = rm.prepare_kvfp4(qf, kf, vf)
    ptab, nxt = [0] * (2 * N // 4), 0
    for pg in sorted(int8):
        ptab[pg], nxt = nxt, nxt + 2
    for pg in range(NP):
        if pg not in int8:
            ptab[pg], nxt = nxt, nxt + 1
    await write_ptab(bus, ptab)
    await bus.write(0x501B, sum(1 << pg for pg in range(NP) if pg not in int8))
    await bus.write(0x501A, 3)
    pick = lambda a8, af: np.array([a8[t] if t // 4 in int8 else af[t] for t in range(T)])
    km, ekm, vm, evm = pick(k8, kp), pick(ek8, ekp), pick(v8, vp), pick(ev8, evp)
    for t in range(T):
        pg, i = t // 4, t % 4
        slot = ptab[pg] * 4 + (i if pg not in int8 else 2 * i)
        for base, mat in ((0x1000, km), (0x2000, vm)):
            data = rm.pack_fp4(mat[t]) if pg not in int8 else [int(x) for x in mat[t]]
            for b, byte in enumerate(data):
                await bus.write(base + 8 * slot + b, int(byte) & 0xFF)
        await bus.write(0x4100 + slot, int(ekm[t]))
        await bus.write(0x4200 + slot, int(evm[t]))
        qr = t % N
        for c in range(D):
            await bus.write(qr * D + c, int(q[t][c]))
        await bus.write(0x4000 + qr, int(eq[t]))
        await bus.write(0x5010, t + 1)
        await bus.write(0x5011, qr)
        await bus.write(0x5000, 1 | (1 << 4))
        await wait_done(dut)
        o, eo, _, _ = await read_out(bus, rows=[qr])
        exp = rm.attention_decode(q[:t + 1], eq[:t + 1], km[:t + 1], ekm[:t + 1], vm[:t + 1], evm[:t + 1],
                                  qrow=t, length=t + 1)
        assert np.array_equal(o[qr], exp.o[0]) and eo[qr] == exp.eo[0], f"mixed FP4 t={t} mismatch"
    summary.append(f"fp4 mixed paging (INT8 pages 0 and 5, FP4 pages 1-4): {T} decode steps bit-exact")
    await bus.write(0x501A, 0)
    await bus.write(0x501B, 0)
    await write_ptab(bus, list(range(2 * N // 4)))
    await bus.write(0x5012, 0)
    with open(os.path.join(os.path.dirname(__file__), "..", "results", f"fp4_kv_N{N}_D{D}.txt"), "w") as f:
        f.write(chr(10).join(summary) + chr(10))


@cocotb.test()
async def test_max_first_decode(dut):
    """Max-first decode (SKIPTHR bit 7): pass A scores every cached key and
    finds the final maximum, pass B visits only keys with a nonzero weight
    against it. Bit-exact with the model's final-max decode, SKIPS and VSKIPS
    count the keys pass B never visits, and the heavy-hitter tracker sees the
    same scores as in a normal step. Cycles and V reads are compared with the
    normal (running-max) decode."""
    bus = await setup(dut)
    rng = np.random.default_rng(int(os.environ.get("RAIJIN_SEED", 7)) + 101)
    summary = []

    async def step(row, length, mf, thr):
        await bus.write(0x5013, thr | (0x80 if mf else 0))
        await bus.write(0x5010, length)
        await bus.write(0x5011, row)
        await bus.write(0x5000, 1 | (1 << 4))
        await wait_done(dut)
        o, eo, cycles, skips = await read_out(bus, rows=[row])
        return o[row], eo[row], cycles, skips, await bus.read32(0x5014)

    await bus.write(0x5013, 0x86)
    assert await bus.read(0x5013) == 0x86
    for name, kw in CASES:
        qf, kf, vf = rm.random_case(rng, n=N, d=D, **kw)
        args = rm.prepare(qf, kf, vf)
        await load(bus, *args)
        ref = rm.reference_causal(qf, kf, vf)
        for mode in (rm.MODE_EXACT, rm.MODE_MF2):
            for thr in (0, 6):
                stats = {False: [0, 0, 0, []], True: [0, 0, 0, []]}   # cycles, V reads, steps, cos
                for t in range(N):
                    for mf in (False, True):
                        await bus.write(0x5013, thr | (0x80 if mf else 0))
                        await bus.write(0x5010, t + 1)
                        await bus.write(0x5011, t)
                        await bus.write(0x5000, 1 | (1 << 4) | (mode << 2))
                        await wait_done(dut)
                        o, eo, cycles, skips = await read_out(bus, rows=[t])
                        vsk = await bus.read32(0x5014)
                        exp = rm.attention_decode(*args, qrow=t, length=t + 1, mode=mode, thr=thr, final_max=mf)
                        tag = f"{name} mode={mode} thr={thr} mf={int(mf)} len={t + 1}"
                        assert np.array_equal(o[t], exp.o[0]) and eo[t] == exp.eo[0], \
                            f"{tag}: rtl={o[t]} e={eo[t]} model={exp.o[0]} e={exp.eo[0]}"
                        assert skips == exp.skips and vsk == exp.skips, \
                            f"{tag}: skips={skips} vsk={vsk} model={exp.skips}"
                        st = stats[mf]
                        st[0] += cycles
                        st[1] += t + 1 - vsk
                        st[2] += t + 1
                        st[3].append(_cos(rm.mx_dequantize(o[t:t + 1], eo[t:t + 1])[0], ref[t]))
                        if t == N - 1:
                            st.append(cycles)
                n0, n1 = stats[False], stats[True]
                summary.append(f"{name:8s} mode={mode} thr={thr}: V reads {n0[1]:3d} -> {n1[1]:3d} of {n0[2]} "
                               f"({100 * (1 - n1[1] / n0[1]):.0f}% fewer), cycles at LEN={N} {n0[4]} -> {n1[4]}, "
                               f"total cycles {n0[0]} -> {n1[0]}, mean cos {np.mean(n0[3]):.5f} -> {np.mean(n1[3]):.5f}")
                dut._log.info(summary[-1])
    # 2N-token MXINT4 cache, LEN = 2N
    await bus.write(0x5012, 3)
    T = 2 * N
    qf, kf, vf = rm.random_case(rng, n=T, d=D, scale=1.5)
    q, eq, k, ek, v, ev = rm.prepare_kv4(qf, kf, vf)
    for base, mat in ((0x1000, k), (0x2000, v)):
        for t in range(T):
            for b, byte in enumerate(rm.pack4(mat[t])):
                await bus.write(base + 8 * t + b, byte)
    for t in range(T):
        await bus.write(0x4100 + t, int(ek[t]))
        await bus.write(0x4200 + t, int(ev[t]))
    row = 5
    for c in range(D):
        await bus.write(row * D + c, int(q[T - 1][c]))
    await bus.write(0x4000 + row, int(eq[T - 1]))
    mq = np.zeros((N, D), dtype=np.int64)
    meq = np.zeros(N, dtype=np.int64)
    mq[row], meq[row] = q[T - 1], eq[T - 1]
    res = {}
    for mf in (False, True):
        o, eo, cycles, skips, vsk = await step(row, T, mf, 0)
        exp = rm.attention_decode(mq, meq, k, ek, v, ev, qrow=row, length=T, final_max=mf)
        assert np.array_equal(o, exp.o[0]) and eo == exp.eo[0], f"INT4 LEN={T} mf={mf} mismatch"
        assert skips == exp.skips and vsk == exp.skips
        res[mf] = (cycles, T - vsk)
    summary.append(f"MXINT4 cache LEN={T}: cycles {res[False][0]} -> {res[True][0]}, "
                   f"V reads {res[False][1]} -> {res[True][1]}")
    dut._log.info(summary[-1])
    await bus.write(0x5012, 0)
    # the tracker accumulates the same weights in either kind of step
    qf, kf, vf = rm.random_case(rng, n=N, d=D, scale=2.5)
    args = rm.prepare(qf, kf, vf)
    await load(bus, *args)
    accs = {}
    for mf in (False, True):
        await bus.write(0x7080, 0x3)
        await step(N - 1, N, mf, 0)
        accs[mf] = [await bus.read(0x7000 + i) for i in range(2 * N)] + [await bus.read(0x7081)]
    assert accs[False] == accs[True], (accs[False], accs[True])
    await bus.write(0x7080, 0x2)
    await bus.write(0x5013, 0)
    with open(os.path.join(os.path.dirname(__file__), "..", "results", f"max_first_N{N}_D{D}.txt"), "w") as f:
        f.write(chr(10).join(summary) + chr(10))


@cocotb.test()
async def test_flat_32_keys(dut):
    """The softmax sum l is 14 bits (Q6.8): a 32-token INT4 cache with flat
    attention sums to 32.0, which the old 13-bit register wrapped. A zero
    query gives every key weight exactly 1.0; the output must be the mean of V, bit-exact
    with the model, in normal and max-first decode."""
    bus = await setup(dut)
    rng = np.random.default_rng(int(os.environ.get("RAIJIN_SEED", 7)) + 113)
    await bus.write(0x5012, 3)
    T = 2 * N
    qf, kf, vf = rm.random_case(rng, n=T, d=D, scale=1.0)
    qf[T - 1] = 0.0                                # every score 0: p = 1.0 each, l = 32 * 256
    q, eq, k, ek, v, ev = rm.prepare_kv4(qf, kf, vf)
    for base, mat in ((0x1000, k), (0x2000, v)):
        for t in range(T):
            for b, byte in enumerate(rm.pack4(mat[t])):
                await bus.write(base + 8 * t + b, byte)
    for t in range(T):
        await bus.write(0x4100 + t, int(ek[t]))
        await bus.write(0x4200 + t, int(ev[t]))
    row = 3
    for c in range(D):
        await bus.write(row * D + c, int(q[T - 1][c]))
    await bus.write(0x4000 + row, int(eq[T - 1]))
    mq = np.zeros((N, D), dtype=np.int64)
    meq = np.zeros(N, dtype=np.int64)
    mq[row], meq[row] = q[T - 1], eq[T - 1]
    vmean = rm.mx_dequantize(v, ev).mean(axis=0)
    lines = []
    for mf in (False, True):
        await bus.write(0x5013, 0x80 if mf else 0)
        await bus.write(0x5010, T)
        await bus.write(0x5011, row)
        await bus.write(0x5000, 1 | (1 << 4))
        await wait_done(dut)
        o, eo, cycles, _ = await read_out(bus, rows=[row])
        exp = rm.attention_decode(mq, meq, k, ek, v, ev, qrow=row, length=T, final_max=mf)
        assert exp.l[0] == T * 256, exp.l
        assert np.array_equal(o[row], exp.o[0]) and eo[row] == exp.eo[0], f"mf={mf}: rtl={o[row]} model={exp.o[0]}"
        got = rm.mx_dequantize(o[row:row + 1], eo[row:row + 1])[0]
        lines.append(f"flat LEN={T} max_first={int(mf)}: l={exp.l[0]} cos vs mean(V)={_cos(got, vmean):.5f} cycles={cycles}")
        assert _cos(got, vmean) > 0.999, lines[-1]
        dut._log.info(lines[-1])
    await bus.write(0x5013, 0)
    await bus.write(0x5012, 0)
    with open(os.path.join(os.path.dirname(__file__), "..", "results", f"flat32_N{N}_D{D}.txt"), "w") as f:
        f.write(chr(10).join(lines) + chr(10))


@cocotb.test()
async def test_key_skip(dut):
    """Hardware key skip (SKIPTHR, 0x5013): keys scoring SKIPTHR or more powers
    of two below the running max get p = 0. When no row of a P-row group uses a
    key, its V row is never read (VSKIPS, 0x5014). Bit-exact with the model for
    fused, unfused, causal and decode passes; accuracy vs FP32 is recorded."""
    bus = await setup(dut)
    rng = np.random.default_rng(int(os.environ.get("RAIJIN_SEED", 7)) + 23)
    summary = []
    for name, kw in CASES:
        qf, kf, vf = rm.random_case(rng, n=N, d=D, **kw)
        args = rm.prepare(qf, kf, vf)
        await load(bus, *args)
        ref = rm.reference(qf, kf, vf)
        refc = rm.reference_causal(qf, kf, vf)
        for thr in (0, 6, 4, 2):
            await bus.write(0x5013, thr)
            assert await bus.read(0x5013) == thr
            for fused, causal in ((False, False), (True, False), (True, True)):
                for mode in (rm.MODE_EXACT, rm.MODE_MF2):
                    exp = rm.attention(*args, fused=fused, mode=mode, causal=causal, thr=thr)
                    o, eo, cycles, skips = await run_pass(bus, fused, mode, causal=causal)
                    vsk = await bus.read32(0x5014)
                    tag = f"{name} thr={thr} fused={int(fused)} causal={int(causal)} mode={mode}"
                    assert np.array_equal(o, exp.o), f"{tag}: output mismatch\nrtl=\n{o}\nmodel=\n{exp.o}"
                    assert np.array_equal(eo, exp.eo), f"{tag}: exponent mismatch"
                    assert skips == exp.skips, f"{tag}: skips rtl={skips} model={exp.skips}"
                    exp_v = rm.v_reads_saved(exp.used, P, causal)
                    assert vsk == exp_v, f"{tag}: vskips rtl={vsk} model={exp_v}"
                    r = refc if causal else ref
                    got = rm.mx_dequantize(o, eo)
                    cos = float(np.mean([np.dot(got[i], r[i]) / np.linalg.norm(got[i]) /
                                         np.linalg.norm(r[i]) for i in range(N)]))
                    total = N * N if not causal else N * (N + 1) // 2
                    beats = (N // P) * N if not causal else sum((g + 1) * P for g in range(N // P))
                    summary.append(f"{tag:46s} cycles={cycles:4d} p0_keys={skips:3d}/{total:3d} "
                                   f"v_saved={vsk:3d}/{beats:3d} mean_row_cos={cos:.5f}")
                    dut._log.info(summary[-1])
            # decode: one row, every zero-p key saves its V read
            for t in (N // 2 - 1, N - 1):
                q, eq, k, ek, v, ev = args
                await bus.write(0x5010, t + 1)
                await bus.write(0x5011, t)
                await bus.write(0x5000, 1 | (1 << 4))
                await wait_done(dut)
                o, eo, cycles, skips = await read_out(bus, rows=[t])
                vsk = await bus.read32(0x5014)
                exp = rm.attention_decode(*args, qrow=t, length=t + 1, thr=thr)
                tag = f"{name} thr={thr} decode len={t + 1}"
                assert np.array_equal(o[t], exp.o[0]) and eo[t] == exp.eo[0], f"{tag}: mismatch"
                assert skips == exp.skips and vsk == exp.skips, f"{tag}: skips={skips} vsk={vsk} model={exp.skips}"
    await bus.write(0x5013, 0)
    with open(os.path.join(os.path.dirname(__file__), "..", "results", f"keyskip_N{N}_D{D}.txt"), "w") as f:
        f.write(chr(10).join(summary) + chr(10))


@cocotb.test()
async def test_rope(dut):
    """RoPE on write (ROPE 0x5018, ROPEPOS 0x5019): the host writes raw Q and K
    rows and the chip rotates each pair on its way into SRAM. Passes are
    bit-exact with the model of the rotator, and are compared with FP32 RoPE
    attention. Decode uses absolute positions from ROPEPOS."""
    bus = await setup(dut)
    rng = np.random.default_rng(int(os.environ.get("RAIJIN_SEED", 7)) + 31)
    summary = []
    for name, kw in CASES:
        qf, kf, vf = rm.random_case(rng, n=N, d=D, **kw)
        q, eq, k, ek, v, ev = rm.prepare(qf, kf, vf)
        pos = list(range(N))
        qr, eqr = rm.rope_quantized(q, eq, pos)
        kr, ekr = rm.rope_quantized(k, ek, pos)
        await bus.write(0x5018, 1)
        assert await bus.read(0x5018) == 1
        await load(bus, q, eqr, k, ekr, v, ev)          # raw rows; the chip rotates
        await bus.write(0x5018, 0)
        qrf, krf = rm.rope_float(qf, pos), rm.rope_float(kf, pos)
        for fused, causal in ((False, False), (True, False), (True, True)):
            for mode in (rm.MODE_EXACT, rm.MODE_MF2):
                exp = rm.attention(qr, eqr, kr, ekr, v, ev, fused=fused, mode=mode, causal=causal)
                o, eo, cycles, skips = await run_pass(bus, fused, mode, causal=causal)
                tag = f"rope {name} fused={int(fused)} causal={int(causal)} mode={mode}"
                assert np.array_equal(o, exp.o), f"{tag}: output mismatch\nrtl=\n{o}\nmodel=\n{exp.o}"
                assert np.array_equal(eo, exp.eo), f"{tag}: exponent mismatch"
                assert skips == exp.skips, f"{tag}: skips"
                ref = (rm.reference_causal if causal else rm.reference)(qrf, krf, vf)
                got = rm.mx_dequantize(o, eo)
                cos = float(np.mean([np.dot(got[i], ref[i]) / np.linalg.norm(got[i]) /
                                     np.linalg.norm(ref[i]) for i in range(N)]))
                if mode == rm.MODE_EXACT:
                    summary.append(f"{tag:42s} cycles={cycles:4d} mean_row_cos={cos:.5f}")
                    dut._log.info(summary[-1])
        # decode at absolute positions 200 + t (ROPEPOS), one token at a time
        base = 200
        apos = [base + t for t in range(N)]
        qa, eqa = rm.rope_quantized(q, eq, apos)
        ka, eka = rm.rope_quantized(k, ek, apos)
        refa = rm.reference_causal(rm.rope_float(qf, apos), rm.rope_float(kf, apos), vf)
        coss = []
        for t in range(N):
            await bus.write(0x5018, 3)
            await bus.write(0x5019, base + t)
            for b, mat in ((0x0000, q), (0x1000, k)):
                for c in range(D):
                    await bus.write(b + t * D + c, int(mat[t][c]))
            await bus.write(0x5018, 0)
            for c in range(D):
                await bus.write(0x2000 + t * D + c, int(v[t][c]))
            for b, ex in ((0x4000, eqa), (0x4100, eka), (0x4200, ev)):
                await bus.write(b + t, int(ex[t]))
            await bus.write(0x5010, t + 1)
            await bus.write(0x5011, t)
            await bus.write(0x5000, 1 | (1 << 4))
            await wait_done(dut)
            o, eo, cycles, skips = await read_out(bus, rows=[t])
            exp = rm.attention_decode(qa, eqa, ka, eka, v, ev, qrow=t, length=t + 1)
            assert np.array_equal(o[t], exp.o[0]) and eo[t] == exp.eo[0], f"rope decode {name} t={t}"
            got = rm.mx_dequantize(o[t:t + 1], eo[t:t + 1])[0]
            coss.append(float(np.dot(got, refa[t]) / np.linalg.norm(got) / np.linalg.norm(refa[t])))
        summary.append(f"rope {name} decode pos={base}..{base + N - 1}{'':14s} cycles={cycles:4d} mean_cos={np.mean(coss):.5f}")
        dut._log.info(summary[-1])
    with open(os.path.join(os.path.dirname(__file__), "..", "results", f"rope_N{N}_D{D}.txt"), "w") as f:
        f.write(chr(10).join(summary) + chr(10))


async def write_ptab(bus, ptab):
    """PTAB (0x501C..F): 2N/4 entries, entry 2b in the low nibble of byte b."""
    ent = list(ptab) + [0] * (2 * N // 4 - len(ptab))
    for b in range(len(ent) // 2):
        await bus.write(0x501C + b, ent[2 * b] | (ent[2 * b + 1] << 4))


@cocotb.test()
async def test_paged_kv_cache(dut):
    """Paged KV cache (PAGED 0x501A, PTAB 0x501C): the decode step reads logical
    token j from physical token PTAB[j/4]*4 + j%4. Two sequences share one
    cache and one 4-token prompt page (prefix sharing), their pages are
    interleaved as a page allocator would hand them out, and an MXINT4 cache of
    2N tokens is scattered over a random page order. Every step is bit-exact
    with decode on the logical (gathered) sequence."""
    bus = await setup(dut)
    rng = np.random.default_rng(int(os.environ.get("RAIJIN_SEED", 7)) + 53)
    summary = []
    await bus.write(0x501A, 1)
    assert await bus.read(0x501A) == 1
    # ---- two sequences, shared prompt page, INT8 ----
    pf = 4
    qf, kf, vf = rm.random_case(rng, n=pf + 8 + 4, d=D, scale=1.5)
    q, eq, k, ek, v, ev = rm.prepare(qf, kf, vf)
    seqs = {"A": list(range(pf)) + list(range(pf, pf + 8)),          # prompt + 8 own tokens
            "B": list(range(pf)) + list(range(pf + 8, pf + 12))}     # prompt + 4 own tokens
    ptabs = {"A": [0, 2, 3], "B": [0, 1]}        # logical page -> physical page
    qrow_of = {"A": 12, "B": 13}

    async def put_token(seq, t):
        src = seqs[seq][t]
        phys = ptabs[seq][t // 4] * 4 + t % 4
        for base, mat in ((0x1000, k), (0x2000, v)):
            for c in range(D):
                await bus.write(base + phys * D + c, int(mat[src][c]))
        await bus.write(0x4100 + phys, int(ek[src]))
        await bus.write(0x4200 + phys, int(ev[src]))

    async def step(seq, t, fmt="int8", arrays=None):
        qa, eqa, ka, eka, va, eva = arrays
        qr = qrow_of[seq] if fmt == "int8" else t % N
        src = seqs[seq][t]
        for c in range(D):
            await bus.write(qr * D + c, int(qa[src][c]))
        await bus.write(0x4000 + qr, int(eqa[src]))
        await write_ptab(bus, ptabs[seq])
        await bus.write(0x5010, t + 1)
        await bus.write(0x5011, qr)
        await bus.write(0x5000, 1 | (1 << 4))
        await wait_done(dut)
        o, eo, cycles, _ = await read_out(bus, rows=[qr])
        idx = seqs[seq][:t + 1]
        exp = rm.attention_decode(qa[idx], eqa[idx], ka[idx], eka[idx], va[idx], eva[idx],
                                  qrow=t, length=t + 1)
        assert np.array_equal(o[qr], exp.o[0]) and eo[qr] == exp.eo[0], \
            f"paged {fmt} {seq} t={t}: rtl={o[qr]} e={eo[qr]} model={exp.o[0]} e={exp.eo[0]}"
        return rm.mx_dequantize(o[qr:qr + 1], eo[qr:qr + 1])[0], cycles

    for t in range(pf):                      # the shared prompt, written once
        await put_token("A", t)
    coss = {"A": [], "B": []}
    refs = {s: rm.reference_causal(qf[seqs[s]], kf[seqs[s]], vf[seqs[s]]) for s in seqs}
    order = []
    for i in range(8):                       # interleaved generation
        order.append(("A", pf + i))
        if i < 4:
            order.append(("B", pf + i))
    for seq, t in order:
        await put_token(seq, t)
        got, cycles = await step(seq, t, arrays=(q, eq, k, ek, v, ev))
        r = refs[seq][t]
        coss[seq].append(float(np.dot(got, r) / np.linalg.norm(got) / np.linalg.norm(r)))
    for s in seqs:
        summary.append(f"paged int8 seq {s}: {len(seqs[s])} tokens, pages {ptabs[s]}, "
                       f"shared prompt page 0, last step {cycles} cycles, mean cos={np.mean(coss[s]):.5f}")
        dut._log.info(summary[-1])
    # ---- MXINT4 cache of 2N tokens over a shuffled page order ----
    await bus.write(0x5012, 3)
    T = 2 * N
    qf, kf, vf = rm.random_case(rng, n=T, d=D, scale=1.5)
    arr4 = rm.prepare_kv4(qf, kf, vf)
    q4, eq4, k4, ek4, v4, ev4 = arr4
    perm = [int(x) for x in rng.permutation(T // 4)]
    seqs["C"], ptabs["C"] = list(range(T)), perm
    ref = rm.reference_causal(qf, kf, vf)
    cos4 = []
    for t in range(T):
        phys = perm[t // 4] * 4 + t % 4
        for base, mat in ((0x1000, k4), (0x2000, v4)):
            for b, byte in enumerate(rm.pack4(mat[t])):
                await bus.write(base + 8 * phys + b, byte)
        await bus.write(0x4100 + phys, int(ek4[t]))
        await bus.write(0x4200 + phys, int(ev4[t]))
        got, cycles = await step("C", t, fmt="int4", arrays=arr4)
        cos4.append(float(np.dot(got, ref[t]) / np.linalg.norm(got) / np.linalg.norm(ref[t])))
    summary.append(f"paged int4: {T} tokens, page order {perm}, LEN={T} step {cycles} cycles, "
                   f"mean cos={np.mean(cos4):.5f}")
    dut._log.info(summary[-1])
    await bus.write(0x5012, 0)
    await bus.write(0x501A, 0)
    await write_ptab(bus, list(range(2 * N // 4)))
    with open(os.path.join(os.path.dirname(__file__), "..", "results", f"paged_N{N}_D{D}.txt"), "w") as f:
        f.write(chr(10).join(summary) + chr(10))


@cocotb.test()
async def test_token_window(dut):
    """Token window (region 6): each decode token is one 52-byte write burst
    (Q, K, V rows, three exponents, CTRL with auto LEN + 1) and one 17-byte
    read burst (O row and e_o). Plain, then paged with a shuffled page order
    and RoPE at absolute positions, every step bit-exact."""
    bus = await setup(dut)
    rng = np.random.default_rng(int(os.environ.get("RAIJIN_SEED", 7)) + 71)
    summary = []
    for variant in ("plain", "paged+rope"):
        qf, kf, vf = rm.random_case(rng, n=N, d=D, scale=1.5)
        q, eq, k, ek, v, ev = rm.prepare(qf, kf, vf)
        paged = variant != "plain"
        perm = [int(x) for x in rng.permutation(N // 4)] if paged else list(range(2 * N // 4))
        base = 50
        if paged:
            await write_ptab(bus, perm)
            await bus.write(0x501A, 1)
            qm, eqm = rm.rope_quantized(q, eq, [base + t for t in range(N)])
            km, ekm = rm.rope_quantized(k, ek, [base + t for t in range(N)])
        else:
            qm, eqm, km, ekm = q, eq, k, ek
        await bus.write(0x5010, 0)
        coss = []
        for t in range(N):
            qr = t
            await bus.write(0x5011, qr)
            if paged:
                await bus.write(0x5018, 3)
                await bus.write(0x5019, base + t)
            burst = list(q[t]) + list(k[t]) + list(v[t]) + [eq[t], ek[t], ev[t], 1 | (1 << 4)]
            for b, val in enumerate(burst):
                await bus.write(0x6000 + b, int(val))
            await wait_done(dut)
            out = [await bus.read(0x6000 + b) for b in range(D + 1)]
            o = np.array([x - 256 if x & 0x80 else x for x in out[:D]])
            assert await bus.read(0x5010) == t + 1
            exp = rm.attention_decode(qm, eqm, km, ekm, v, ev, qrow=t, length=t + 1)
            assert np.array_equal(o, exp.o[0]) and out[D] == exp.eo[0], \
                f"{variant} t={t}: rtl={o} e={out[D]} model={exp.o[0]} e={exp.eo[0]}"
            ref = (rm.reference_causal(rm.rope_float(qf, [base + i for i in range(N)]),
                                       rm.rope_float(kf, [base + i for i in range(N)]), vf)
                   if paged else rm.reference_causal(qf, kf, vf))[t]
            got = rm.mx_dequantize(o[None, :], np.array([out[D]]))[0]
            coss.append(float(np.dot(got, ref) / np.linalg.norm(got) / np.linalg.norm(ref)))
        summary.append(f"token window {variant}: {N} tokens, 52 + 17 bytes per token, mean cos={np.mean(coss):.5f}")
        dut._log.info(summary[-1])
        await bus.write(0x5018, 0)
        await bus.write(0x501A, 0)
        await write_ptab(bus, list(range(2 * N // 4)))
    with open(os.path.join(os.path.dirname(__file__), "..", "results", f"token_window_N{N}_D{D}.txt"), "w") as f:
        f.write(chr(10).join(summary) + chr(10))


@cocotb.test()
async def test_mixed_precision_kv(dut):
    """Per-page KV precision (PAGED bit 1, PFMT 0x501B): each 4-token page is
    MXINT8 or MXINT4. In the 16-word cache, 2 INT8 pages + 4 INT4 pages hold
    24 tokens. Policies: all INT4, INT8 for the sink page and the most recent
    page, and INT8 for the sink page and the page with the largest INT4 K error.
    Every decode step is bit-exact; accuracy is compared with an all-INT8
    cache (model only, it cannot hold 24 tokens)."""
    bus = await setup(dut)
    rng = np.random.default_rng(int(os.environ.get("RAIJIN_SEED", 7)) + 83)
    T, NP = 24, 6
    # stale cache contents from an earlier sequence (never read for LEN keys)
    for base in (0x1000, 0x2000):
        for a in range(N * D):
            await bus.write(base + a, int(rng.integers(0, 256)))
    summary = []
    for name, kw in CASES:
        qf, kf, vf = rm.random_case(rng, n=T, d=D, **kw)
        q, eq, k8, ek8, v8, ev8 = rm.prepare(qf, kf, vf)
        _, _, k4, ek4, v4, ev4 = rm.prepare_kv4(qf, kf, vf)
        ref = rm.reference_causal(qf, kf, vf)
        kerr = [float(np.abs(rm.mx_dequantize(k4[p*4:p*4+4], ek4[p*4:p*4+4]) - kf[p*4:p*4+4]).max())
                for p in range(NP)]
        worst = max(range(1, NP), key=lambda p: kerr[p])
        policies = {
            "all INT4": set(),
            "INT8 sink + recent": {0, NP - 1},
            "INT8 sink + outlier": {0, worst},
        }
        # model-only all-INT8 reference
        c8 = []
        for t in range(T):
            e = rm.attention_decode(q, eq, k8, ek8, v8, ev8, qrow=t, length=t + 1)
            g = rm.mx_dequantize(e.o, e.eo)[0]
            c8.append(float(np.dot(g, ref[t]) / np.linalg.norm(g) / np.linalg.norm(ref[t])))
        summary.append(f"{name:8s} all INT8 (model, 1.5x the SRAM)     mean cos={np.mean(c8):.5f}")
        for pol, int8 in policies.items():
            # slot pages: INT8 pages take an even pair first, INT4 pages one each
            ptab, nxt = [0] * (2 * N // 4), 0
            for p in sorted(int8):
                ptab[p], nxt = nxt, nxt + 2
            for p in range(NP):
                if p not in int8:
                    ptab[p], nxt = nxt, nxt + 1
            assert nxt <= 2 * N // 4
            pfmt = sum(1 << p for p in range(NP) if p not in int8)
            await write_ptab(bus, ptab)
            await bus.write(0x501B, pfmt)
            await bus.write(0x501A, 3)
            r1a = await bus.read(0x501A)
            r1b = await bus.read(0x501B)
            assert r1a == 3 and r1b == pfmt, (r1a, r1b)
            km = np.array([k8[t] if t // 4 in int8 else k4[t] for t in range(T)])
            ekm = np.array([ek8[t] if t // 4 in int8 else ek4[t] for t in range(T)])
            vm = np.array([v8[t] if t // 4 in int8 else v4[t] for t in range(T)])
            evm = np.array([ev8[t] if t // 4 in int8 else ev4[t] for t in range(T)])
            coss = []
            for t in range(T):
                p, i = t // 4, t % 4
                slot = ptab[p] * 4 + (i if p not in int8 else 2 * i)
                for base, mat in ((0x1000, km), (0x2000, vm)):
                    data = rm.pack4(mat[t]) if p not in int8 else [int(x) for x in mat[t]]
                    for b, byte in enumerate(data):
                        await bus.write(base + 8 * slot + b, int(byte) & 0xFF)
                await bus.write(0x4100 + slot, int(ekm[t]))
                await bus.write(0x4200 + slot, int(evm[t]))
                qr = t % N
                for c in range(D):
                    await bus.write(qr * D + c, int(q[t][c]))
                await bus.write(0x4000 + qr, int(eq[t]))
                await bus.write(0x5010, t + 1)
                await bus.write(0x5011, qr)
                await bus.write(0x5000, 1 | (1 << 4))
                await wait_done(dut)
                o, eo, cycles, _ = await read_out(bus, rows=[qr])
                qa = q[:t + 1].copy()
                eqa = eq[:t + 1].copy()
                exp = rm.attention_decode(qa, eqa, km[:t + 1], ekm[:t + 1], vm[:t + 1], evm[:t + 1],
                                          qrow=t, length=t + 1)
                assert np.array_equal(o[qr], exp.o[0]) and eo[qr] == exp.eo[0], \
                    f"{name} {pol} t={t}: rtl={o[qr]} e={eo[qr]} model={exp.o[0]} e={exp.eo[0]}"
                g = rm.mx_dequantize(o[qr:qr + 1], eo[qr:qr + 1])[0]
                coss.append(float(np.dot(g, ref[t]) / np.linalg.norm(g) / np.linalg.norm(ref[t])))
            words = 4 * len(int8) + 2 * (NP - len(int8))
            summary.append(f"{name:8s} {pol:22s} {words:2d} words  mean cos={np.mean(coss):.5f} "
                           f"min cos={np.min(coss):.5f}")
            dut._log.info(summary[-1])
    await bus.write(0x501A, 0)
    await bus.write(0x501B, 0)
    await write_ptab(bus, list(range(2 * N // 4)))
    with open(os.path.join(os.path.dirname(__file__), "..", "results", f"mixed_kv_N{N}_D{D}.txt"), "w") as f:
        f.write(chr(10).join(summary) + chr(10))


LM_VECTORS = os.path.join(os.path.dirname(__file__), "..", "results", "lm_head_vectors.npz")


@cocotb.test(skip=not os.path.exists(LM_VECTORS))
async def test_lm_head(dut):
    """Replay one real attention head through the RTL: layer 0, head 0 of
    TinyStories-8M on the prompt line from model/lm_eval.py, as one causal
    prefill pass. The RTL must match the golden model's causal pass bit for bit."""
    vec = np.load(LM_VECTORS)
    t_len = len(vec["tokens"])
    assert t_len <= N
    pad = lambda a: np.concatenate([a, np.zeros((N - len(a),) + a.shape[1:], a.dtype)])  # noqa: E731
    q, k, v = pad(vec["q"]), pad(vec["k"]), pad(vec["v"])
    # padded rows get a neutral exponent; causal rows < t_len never see them
    eq, ek, ev = (np.concatenate([vec[e], np.full(N - t_len, 127)]) for e in ("eq", "ek", "ev"))
    bus = await setup(dut)
    await load(bus, q, eq, k, ek, v, ev)
    o, eo, cycles, skips = await run_pass(bus, True, rm.MODE_EXACT, causal=True)
    exp = rm.attention(q, eq, k, ek, v, ev, fused=True, mode=rm.MODE_EXACT, causal=True)
    # the benchmark ran each row as a decode step; compare the dequantised rows too
    got = rm.mx_dequantize(o[:t_len], eo[:t_len])
    dec = rm.mx_dequantize(vec["o"], vec["eo"])
    cos = float(np.mean([np.dot(got[i], dec[i]) / (np.linalg.norm(got[i]) * np.linalg.norm(dec[i]) + 1e-30)
                         for i in range(t_len)]))
    lines = [f"layer 0 head 0, {t_len} prompt tokens, causal prefill pass: {cycles} cycles, {skips} skips",
             f"mean row cosine vs the benchmark's decode-step outputs: {cos:.6f}"]
    for t in range(t_len):
        ok = np.array_equal(o[t], exp.o[t]) and eo[t] == exp.eo[t]
        lines.append(f"row {t:2d} token {int(vec['tokens'][t]):5d} e_o={int(eo[t]):3d} "
                     f"o={' '.join(f'{int(x):4d}' for x in o[t])} {'match' if ok else 'MISMATCH'}")
        assert ok, f"row {t}: rtl {o[t]} e={eo[t]} model {exp.o[t]} e={exp.eo[t]}"
    dut._log.info(lines[0])
    with open(os.path.join(os.path.dirname(__file__), "..", "results", "lm_head_rtl.txt"), "w") as f:
        f.write(chr(10).join(lines) + chr(10))


@cocotb.test(skip=os.environ.get("COCOTB_TEST_FILTER") != "test_gl_short")
async def test_gl_short(dut):
    """Short workload for gate-level power: one load, one fused pass, one
    decode step. Logs the simulated window of each phase in ns and cycles."""
    from cocotb.utils import get_sim_time
    bus = await setup(dut)
    rng = np.random.default_rng(5)
    qf, kf, vf = rm.random_case(rng, n=N, d=D, scale=1.5)
    args = rm.prepare(qf, kf, vf)
    t0 = get_sim_time("ns")
    await load(bus, *args)
    t1 = get_sim_time("ns")
    exp = rm.attention(*args, fused=True, mode=rm.MODE_EXACT)
    o, eo, cycles, _ = await run_pass(bus, True, rm.MODE_EXACT)
    t2 = get_sim_time("ns")
    assert np.array_equal(o, exp.o) and np.array_equal(eo, exp.eo), "fused pass mismatch"
    await bus.write(0x5010, N)
    await bus.write(0x5011, N - 1)
    await bus.write(0x5000, 1 | (1 << 4))
    await wait_done(dut)
    o2, eo2, dcycles, _ = await read_out(bus, rows=[N - 1])
    t3 = get_sim_time("ns")
    dec = rm.attention_decode(*args, qrow=N - 1, length=N, mode=rm.MODE_EXACT)
    assert np.array_equal(o2[N - 1], dec.o[0]) and eo2[N - 1] == dec.eo[0], "decode mismatch"
    msg = (f"GL window: load {t0:.0f}-{t1:.0f} ns, fused pass + readback {t1:.0f}-{t2:.0f} ns "
           f"({cycles} busy cycles), decode + readback {t2:.0f}-{t3:.0f} ns ({dcycles} busy cycles); "
           f"total {t3:.0f} ns = {t3 / 20:.0f} clock cycles")
    dut._log.info(msg)
    with open(os.path.join(os.path.dirname(__file__), "..", "results", "gl_window.txt"), "w") as f:
        f.write(msg + chr(10))


@cocotb.test()
async def test_heavy_hitter(dut):
    """Heavy-hitter tracker (feature 15, region 7): every single-row decode step
    adds each cached key's max-normalised attention weight to its slot's
    accumulator, and HHMIN names the lowest-scoring slot outside the sinks and
    the recent window. All
    accumulators and HHMIN are bit-exact with rm.hh_step; clear, halve, the
    per-slot clear and the token-window readback (byte 17) are checked too."""
    bus = await setup(dut)
    rng = np.random.default_rng(int(os.environ.get("RAIJIN_SEED", 7)) + 71)
    skip, recent = 2, 4
    summary = []

    async def read_acc():
        a = []
        for sl in range(rm.HH_NK):
            lo = await bus.read(0x7000 + 2 * sl)
            hi = await bus.read(0x7001 + 2 * sl)
            a.append(lo | hi << 8)
        return a

    await bus.write(0x7080, 0b011)          # enable, clear all
    await bus.write(0x7082, skip)
    await bus.write(0x7083, recent)
    assert await bus.read(0x7080) == 1 and await bus.read(0x7082) == skip and await bus.read(0x7083) == recent
    for name, kw in CASES[:2]:
        qf, kf, vf = rm.random_case(rng, n=N, d=D, **kw)
        q, eq, k, ek, v, ev = rm.prepare(qf, kf, vf)
        await load(bus, q, eq, k, ek, v, ev)
        await bus.write(0x7080, 0b011)
        acc = [0] * rm.HH_NK
        for t in range(N):
            await bus.write(0x5010, t + 1)
            await bus.write(0x5011, t)
            await bus.write(0x5000, 1 | (1 << 4))
            await wait_done(dut)
            o, eo, cycles, _ = await read_out(bus, rows=[t])
            exp = rm.attention_decode(q, eq, k, ek, v, ev, qrow=t, length=t + 1)
            assert np.array_equal(o[t], exp.o[0]) and eo[t] == exp.eo[0], f"{name} t={t}: decode output"
            sc = rm.decode_scores(q, eq, k, ek, qrow=t, length=t + 1)
            mexp = rm.hh_step(acc, sc, list(range(t + 1)), skip, recent)
            if t >= skip + recent:
                assert await bus.read(0x7081) == mexp, f"{name} t={t}: HHMIN"
        got = await read_acc()
        assert got == acc, f"{name}: accumulators\nrtl={got}\nmodel={acc}"
        summary.append(f"hh {name:8s} after {N} steps: acc={acc[:N]} HHMIN={mexp} decode cycles at LEN=N: {cycles}")
        dut._log.info(summary[-1])
    # decay, per-slot clear, token-window byte 17
    await bus.write(0x7080, 0b101)
    acc = [a >> 1 for a in acc]
    await bus.write(0x7000 + 2 * 5, 0)
    acc[5] = 0
    assert await read_acc() == acc, "halve / per-slot clear"
    assert await bus.read(0x6011) == mexp, "token window HHMIN"
    # disabled: accumulators stay put
    await bus.write(0x7080, 0)
    await bus.write(0x5000, 1 | (1 << 4))
    await wait_done(dut)
    assert await read_acc() == acc, "tracker off"
    with open(os.path.join(os.path.dirname(__file__), "..", "results", f"heavy_hitter_N{N}_D{D}.txt"), "w") as f:
        f.write(chr(10).join(summary) + chr(10))


Writing tb/test_core.py


In [20]:
%%writefile tb/test_chip.py
"""Chip-level cocotb tests: the host talks to raijin_chip over SPI only, with
idle clock gating on. Results must match the golden model bit for bit."""
import os
import sys

import cocotb
import numpy as np
from cocotb.clock import Clock
from cocotb.triggers import ClockCycles, RisingEdge, Timer

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "model"))
import raijin_model as rm  # noqa: E402

N = int(os.environ.get("RAIJIN_N", 16))
P = int(os.environ.get("RAIJIN_P", 4))
D = int(os.environ.get("RAIJIN_D", 16))
CLK_NS = 20                    # 50 MHz
SCK_HALF_NS = 4 * CLK_NS       # SCK = f_clk / 8 = 6.25 MHz


class Spi:
    """SPI / QSPI host on the chip pins. quad=True uses 0x32 / 0x6B, which move
    the data bytes on io[3:0] (command, address and dummy stay single lane)."""

    def __init__(self, dut, quad=False):
        self.dut = dut
        self.quad = quad
        self.bytes = 0
        self.ns = 0

    async def _clock(self, out_nib, width):
        d = self.dut
        d.io_in.value = out_nib
        await Timer(SCK_HALF_NS, unit="ns")
        d.sck.value = 1
        io = int(d.io_out.value)
        await Timer(SCK_HALF_NS, unit="ns")
        d.sck.value = 0
        return io & 0xF if width == 4 else (io >> 1) & 1

    async def _xfer(self, head, data=(), n_in=0, quad=False):
        d = self.dut
        t0 = cocotb.utils.get_sim_time("ns")
        got = []
        d.cs_n.value = 0
        await Timer(SCK_HALF_NS, unit="ns")
        for b in head:                                  # command, address, dummy
            for k in range(7, -1, -1):
                await self._clock((b >> k) & 1, 1)
        for b in data:
            if quad:
                await self._clock((b >> 4) & 0xF, 4)
                await self._clock(b & 0xF, 4)
            else:
                for k in range(7, -1, -1):
                    await self._clock((b >> k) & 1, 1)
        for _ in range(n_in):
            v = 0
            if quad:
                v = (await self._clock(0, 4)) << 4
                v |= await self._clock(0, 4)
            else:
                for _ in range(8):
                    v = (v << 1) | await self._clock(0, 1)
            got.append(v)
        await Timer(SCK_HALF_NS, unit="ns")
        d.cs_n.value = 1
        await Timer(2 * SCK_HALF_NS, unit="ns")
        self.bytes += len(head) + len(data) + n_in
        self.ns += cocotb.utils.get_sim_time("ns") - t0
        return got

    async def write(self, addr, data):
        data = [int(x) & 0xFF for x in (data if hasattr(data, "__len__") else [data])]
        cmd = 0x32 if self.quad else 0x02
        await self._xfer([cmd, addr >> 8, addr & 0xFF], data, quad=self.quad)

    async def read(self, addr, n=1):
        cmd = 0x6B if self.quad else 0x0B
        return await self._xfer([cmd, addr >> 8, addr & 0xFF, 0x00], n_in=n, quad=self.quad)

    async def read32(self, addr):
        b = await self.read(addr, 4)
        return b[0] | b[1] << 8 | b[2] << 16 | b[3] << 24


async def setup(dut):
    cocotb.start_soon(Clock(dut.clk, CLK_NS, unit="ns").start())
    dut.rst_n.value = 0
    dut.cs_n.value = 1
    dut.sck.value = 0
    dut.io_in.value = 0
    await ClockCycles(dut.clk, 3)
    dut.rst_n.value = 1
    await ClockCycles(dut.clk, 2)
    return Spi(dut)


async def load(spi, q, eq, k, ek, v, ev):
    for base, mat in ((0x0000, q), (0x1000, k), (0x2000, v)):
        await spi.write(base, np.asarray(mat).reshape(-1))      # one burst per tensor
    for base, ex in ((0x4000, eq), (0x4100, ek), (0x4200, ev)):
        await spi.write(base, ex)


async def wait_done(dut):
    for _ in range(20000):
        await RisingEdge(dut.clk)
        if int(dut.done.value):
            return
    raise AssertionError("pass did not finish")


async def read_out(spi, rows):
    o = np.zeros((N, D), dtype=np.int64)
    eo = np.zeros(N, dtype=np.int64)
    if rows == list(range(rows[0], rows[0] + len(rows))):
        flat = await spi.read(0x3000 + rows[0] * D, len(rows) * D)
        ex = await spi.read(0x4300 + rows[0], len(rows))
        for i, r in enumerate(rows):
            o[r] = [b - 256 if b & 0x80 else b for b in flat[i * D:(i + 1) * D]]
            eo[r] = ex[i]
    return o, eo


@cocotb.test()
async def test_spi_registers(dut):
    """ID registers and a register write/read round trip over SPI."""
    spi = await setup(dut)
    ids = await spi.read(0x500C, 4)
    assert ids[0] == N and ids[1] == D and ids[2] == P, ids
    await spi.write(0x5001, 0xFD)
    assert (await spi.read(0x5001))[0] == 0xFD
    await spi.write(0x5001, 0)
    # exponent memory: a burst write then a burst read back
    await spi.write(0x4100, list(range(100, 100 + N)))
    assert await spi.read(0x4100, N) == list(range(100, 100 + N))
    # the same over QSPI, and mixed: quad write, single read and back
    spi.quad = True
    vals = [(37 * i + 11) & 0xFF for i in range(N)]
    await spi.write(0x4200, vals)
    assert await spi.read(0x4200, N) == vals
    spi.quad = False
    assert await spi.read(0x4200, N) == vals
    await spi.write(0x4200, vals[::-1])
    spi.quad = True
    assert await spi.read(0x4200, N) == vals[::-1]


async def run_passes(dut, quad):
    """Fused, causal and unfused passes, and KV-cache decode, all over SPI with
    the clock gate active. Bit-exact; records the SPI bytes per pass, the
    gated-clock share of an idle-heavy decode loop, and host-limited rates."""
    spi = await setup(dut)
    spi.quad = quad
    lane = "qspi" if quad else "spi"
    rng = np.random.default_rng(int(os.environ.get("RAIJIN_SEED", 7)) + 61)
    qf, kf, vf = rm.random_case(rng, n=N, d=D, scale=1.5)
    args = rm.prepare(qf, kf, vf)
    spi.bytes, spi.ns = 0, 0
    await load(spi, *args)
    load_bytes, load_ns = spi.bytes, spi.ns
    summary = []
    for fused, causal in ((True, False), (True, True), (False, False)):
        exp = rm.attention(*args, fused=fused, mode=rm.MODE_EXACT, causal=causal)
        await spi.write(0x5000, 1 | (int(fused) << 1) | (int(causal) << 5))
        await wait_done(dut)
        o, eo = await read_out(spi, list(range(N)))
        cycles = await spi.read32(0x5004)
        tag = f"{lane} fused={int(fused)} causal={int(causal)}"
        assert np.array_equal(o, exp.o), f"{tag}: output mismatch\nrtl=\n{o}\nmodel=\n{exp.o}"
        assert np.array_equal(eo, exp.eo), f"{tag}: exponent mismatch"
        summary.append(f"{tag:28s} cycles={cycles:4d}")
        dut._log.info(summary[-1])
    # decode loop: per token, write one K/V/Q row + exponents, start, read one row
    q, eq, k, ek, v, ev = args
    g0, t0 = int(dut.gated_cycles.value), cocotb.utils.get_sim_time("ns")
    spi.bytes, spi.ns = 0, 0
    for t in range(N):
        for base, mat in ((0x0000, q), (0x1000, k), (0x2000, v)):
            await spi.write(base + t * D, mat[t])
        for base, ex in ((0x4000, eq), (0x4100, ek), (0x4200, ev)):
            await spi.write(base + t, int(ex[t]))
        await spi.write(0x5010, [t + 1, t])
        await spi.write(0x5000, 1 | (1 << 4))
        await wait_done(dut)
        o, eo = await read_out(spi, [t])
        exp = rm.attention_decode(q, eq, k, ek, v, ev, qrow=t, length=t + 1)
        assert np.array_equal(o[t], exp.o[0]) and eo[t] == exp.eo[0], f"{lane} decode t={t}"
        # an idle gap between tokens, as when the host runs the rest of the model
        await ClockCycles(dut.clk, 200)
    total = (cocotb.utils.get_sim_time("ns") - t0) / CLK_NS
    gated = int(dut.gated_cycles.value) - g0
    per_tok_us = spi.ns / N / 1000
    summary.append(f"{lane} load: {load_bytes} bytes for Q, K, V and exponents, {load_ns / 1000:.1f} us on the bus")
    summary.append(f"{lane} decode loop: {N} tokens, {spi.bytes} bytes, {per_tok_us:.1f} us of bus time per token "
                   f"(host-limited {1e6 / per_tok_us:,.0f} tokens/s), {total:.0f} cycles, "
                   f"core clock gated {gated} cycles ({100 * gated / total:.1f}%)")
    for line in summary[-2:]:
        dut._log.info(line)
    assert gated > 0
    return summary


@cocotb.test()
async def test_spi_passes(dut):
    """Fused, causal and unfused passes, and KV-cache decode, all over single-
    lane SPI with the clock gate active. Bit-exact; records the bus bytes and
    time per pass and per token, and the gated-clock share of a decode loop."""
    summary = await run_passes(dut, quad=False)
    with open(os.path.join(os.path.dirname(__file__), "..", "results", f"chip_spi_N{N}_D{D}.txt"), "w") as f:
        f.write(chr(10).join(summary) + chr(10))


@cocotb.test()
async def test_qspi_passes(dut):
    """The same passes and decode loop over QSPI (0x32 / 0x6B)."""
    summary = await run_passes(dut, quad=True)
    with open(os.path.join(os.path.dirname(__file__), "..", "results", f"chip_qspi_N{N}_D{D}.txt"), "w") as f:
        f.write(chr(10).join(summary) + chr(10))


@cocotb.test()
async def test_qspi_token_window(dut):
    """Decode over QSPI with the token window: per token one 55-byte write
    transaction (header + Q/K/V/exponents/CTRL) and one 21-byte read. Bit-exact;
    records bus time per token against the per-register loop."""
    spi = await setup(dut)
    rng = np.random.default_rng(int(os.environ.get("RAIJIN_SEED", 7)) + 61)
    qf, kf, vf = rm.random_case(rng, n=N, d=D, scale=1.5)
    q, eq, k, ek, v, ev = rm.prepare(qf, kf, vf)
    lines = []
    for quad in (False, True):
        spi.quad = quad
        await spi.write(0x5010, 0)
        spi.bytes, spi.ns = 0, 0
        for t in range(N):
            await spi.write(0x5011, t)
            await spi.write(0x6000, list(q[t]) + list(k[t]) + list(v[t]) + [eq[t], ek[t], ev[t], 1 | (1 << 4)])
            await wait_done(dut)
            out = await spi.read(0x6000, D + 1)
            o = np.array([x - 256 if x & 0x80 else x for x in out[:D]])
            exp = rm.attention_decode(q, eq, k, ek, v, ev, qrow=t, length=t + 1)
            assert np.array_equal(o, exp.o[0]) and out[D] == exp.eo[0], f"token window quad={quad} t={t}"
        per_tok_us = spi.ns / N / 1000
        lines.append(f"{'qspi' if quad else 'spi'} token window: {spi.bytes / N:.0f} bytes and {per_tok_us:.1f} us "
                     f"of bus time per token (host-limited {1e6 / per_tok_us:,.0f} tokens/s)")
        dut._log.info(lines[-1])
    with open(os.path.join(os.path.dirname(__file__), "..", "results", f"chip_token_window_N{N}_D{D}.txt"), "w") as f:
        f.write(chr(10).join(lines) + chr(10))


Writing tb/test_chip.py


In [21]:
%%writefile tb/run_tests.py
#!/usr/bin/env python3
"""Build and run every cocotb test with Icarus Verilog.

Usage: python3 tb/run_tests.py [blocks|core|uart|chip|all] [--sim icarus|verilator]
Run it inside the tools container: scripts/dock.sh python3 tb/run_tests.py
"""
import argparse
import os
import sys
from pathlib import Path

from cocotb_tools.runner import get_runner

ROOT = Path(__file__).resolve().parent.parent
RTL = ROOT / "rtl"
SOURCES = [RTL / f for f in (
    "raijin_dpu.v", "raijin_requant.v", "raijin_exp2.v", "raijin_softmax_lane.v",
    "raijin_acc_array.v", "raijin_dupreg.v", "raijin_rope.v", "raijin_hh.v", "raijin_out.v", "raijin_mem.v", "raijin_core.v",
    "raijin_spi.v", "raijin_chip.v")]

# RAIJIN_WAVES=1 dumps a VCD (for power annotation); pair with COCOTB_TEST_FILTER to pick one test
WAVES = bool(os.environ.get("RAIJIN_WAVES"))

BLOCKS = ["raijin_exp2", "raijin_requant", "raijin_dpu", "raijin_out", "raijin_softmax_lane", "raijin_rope"]


FPGA_SOURCES = [ROOT / "fpga" / "rtl" / f for f in ("uart_rx.v", "uart_tx.v", "raijin_bridge.v")] + \
    [ROOT / "tb" / "raijin_uart_tb_top.v"]


PDK_SRAM = Path(os.environ.get("PDK_ROOT", "/foss/pdks")) / "sky130A" / "libs.ref" /     "sky130_sram_macros" / "verilog" / "sky130_sram_1kbyte_1rw1r_32x256_8.v"


def macro_model(build_dir):
    """Copy of the OpenRAM behavioural model without its per-access log, which
    Icarus cannot elaborate (it indexes a memory inside $display)."""
    build_dir.mkdir(parents=True, exist_ok=True)
    dst = build_dir / "sky130_sram_1kbyte_1rw1r_32x256_8.v"
    lines = PDK_SRAM.read_text().splitlines()
    keep = [";" if ("$display" in ln and "mem[" in ln) else ln for ln in lines]
    dst.write_text(chr(10).join(keep) + chr(10))
    return dst


def run(top, module, sim, params=None, extra_env=None, macro=False):
    runner = get_runner(sim)
    sources = SOURCES + (FPGA_SOURCES if top == "raijin_uart_tb_top" else [])
    build_dir = ROOT / "sim_build" / f"{top}_{sim}{'_macro' if macro else ''}"
    build_args = ["-Wall"] if sim == "icarus" else ["-Wno-fatal", "-Wno-WIDTH"]
    defines = {"RAIJIN_TRACE": 1} if os.environ.get("RAIJIN_TRACE") else {}
    if macro:
        # simulate against the real OpenRAM macro model instead of rtl/raijin_mem.v
        sources = [s for s in sources if s.name != "raijin_mem.v"] + [macro_model(build_dir)]
        defines = dict(defines, RAIJIN_SKY130_SRAM=1)
    runner.build(sources=sources, hdl_toplevel=top, build_dir=build_dir, defines=defines,
                 parameters=params or {}, always=True, build_args=build_args, waves=WAVES,
                 timescale=("1ns", "1ps"))
    env = dict(extra_env or {})
    env["COCOTB_TOPLEVEL"] = top
    os.environ.update(env)
    res = runner.test(hdl_toplevel=top, test_module=module, test_dir=ROOT / "tb",
                      build_dir=build_dir, extra_env=env, waves=WAVES)
    return res


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("what", nargs="?", default="all", choices=["blocks", "core", "uart", "chip", "all"])
    ap.add_argument("--sim", default="icarus")
    ap.add_argument("--n", type=int, default=16)
    ap.add_argument("--p", type=int, default=4, help="DPUs / softmax lanes")
    ap.add_argument("--macro", action="store_true",
                    help="use the PDK's OpenRAM SRAM model (as synthesised for sky130)")
    args = ap.parse_args()
    (ROOT / "results").mkdir(exist_ok=True)
    if args.what in ("blocks", "all"):
        for top in BLOCKS:
            run(top, "test_blocks", args.sim)
    if args.what in ("core", "all"):
        run("raijin_core", "test_core", args.sim, params={"N": args.n, "D": 16, "P": args.p},
            extra_env={"RAIJIN_N": str(args.n), "RAIJIN_D": "16", "RAIJIN_P": str(args.p)}, macro=args.macro)
    if args.what in ("uart", "all"):
        run("raijin_uart_tb_top", "test_uart", args.sim)
    if args.what in ("chip", "all"):
        run("raijin_chip", "test_chip", args.sim, params={"N": args.n, "D": 16, "P": args.p},
            extra_env={"RAIJIN_N": str(args.n), "RAIJIN_D": "16", "RAIJIN_P": str(args.p)})


if __name__ == "__main__":
    sys.exit(main())


Writing tb/run_tests.py


Test vectors for `test_lm_head`: the quantised Q, K, V of layer 0, head 0 of TinyStories-8M on a real prompt (captured by `model/lm_eval.py`).

In [22]:
import base64
open('results/lm_head_vectors.npz', 'wb').write(base64.b64decode(
'UEsDBC0AAAAAAAAAIQAgbiaL//////////8FABQAcS5ucHkBABAAgAcAAAAAAACABwAAAAAAAJNOVU1QWQEAdgB7J2Rlc2NyJzogJzxpOCcsICdmb3J0cmFuX29yZGVyJzogRmFsc2UsICdzaGFwZSc6ICgxNCwgMTYpLCB9ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAK9f////////8TAAAAAAAAADIAAAAAAAAA3f///////////////////+b/////////3P/////////v/////////9b/////////BgAAAAAAAAAOAAAAAAAAAA0AAAAAAAAAIwAAAAAAAAAGAAAAAAAAAEkAAAAAAAAACAAAAAAAAADv/////////z4AAAAAAAAAGAAAAAAAAADa/////////wgAAAAAAAAA3P/////////W/////////+b/////////IAAAAAAAAAAiAAAAAAAAAPb/////////EgAAAAAAAAAGAAAAAAAAABcAAAAAAAAASgAAAAAAAAAaAAAAAAAAAPH/////////5/////////8hAAAAAAAAANf/////////JwAAAAAAAACp/////////9//////////HgAAAAAAAADi/////////y8AAAAAAAAAFQAAAAAAAAAbAAAAAAAAAAcAAAAAAAAA2v////////9EAAAAAAAAACgAAAAAAAAADAAAAAAAAAD6/////////ycAAAAAAAAA8P///////////////////+f/////////vf/////////r/////////+//////////QgAAAAAAAAD3/////////xEAAAAAAAAA2f/////////u/////////zsAAAAAAAAA5f/////////z/////////+3//////////f/////////Y/////////9H/////////+f/////////Y/////////wcAAAAAAAAACAAAAAAAAAAGAAAAAAAAAAcAAAAAAAAAHwAAAAAAAADu//////////3/////////QwAAAAAAAAABAAAAAAAAABgAAAAAAAAAEwAAAAAAAAA4AAAAAAAAAI7/////////qf////////8EAAAAAAAAAL7/////////DAAAAAAAAAAhAAAAAAAAAHoAAAAAAAAABAAAAAAAAAAIAAAAAAAAANX/////////8f////////9DAAAAAAAAACUAAAAAAAAACQAAAAAAAAD1/////////yAAAAAAAAAA2f/////////O//////////3/////////8f////////8EAAAAAAAAAP7/////////EwAAAAAAAAD6/////////wwAAAAAAAAABwAAAAAAAAAAAAAAAAAAAFgAAAAAAAAA8P/////////4/////////9f/////////AwAAAAAAAADG/////////yMAAAAAAAAAqP/////////j/////////zQAAAAAAAAA+P////////8nAAAAAAAAAB8AAAAAAAAAGgAAAAAAAAD//////////8b/////////PwAAAAAAAAAhAAAAAAAAANv/////////DAAAAAAAAAAaAAAAAAAAAO3/////////EAAAAAAAAADi//////////r/////////9/////////8ZAAAAAAAAAOb/////////RAAAAAAAAADi/////////8f/////////8v////////9UAAAAAAAAAOL/////////7f////////8DAAAAAAAAABsAAAAAAAAA+v////////8HAAAAAAAAAN//////////GgAAAAAAAADs//////////z/////////9f////////9NAAAAAAAAAAAAAAAAAAAA1v////////8YAAAAAAAAAG8AAAAAAAAA9//////////g/////////wMAAAAAAAAAIQAAAAAAAADN/////////zcAAAAAAAAA4f/////////g//////////7/////////8v////////9SAAAAAAAAABYAAAAAAAAABQAAAAAAAADD/////////zwAAAAAAAAAdwAAAAAAAADW/////////+b/////////NQAAAAAAAAAIAAAAAAAAAMX////////////////////x/////////+H/////////DAAAAAAAAADG/////////yMAAAAAAAAAJQAAAAAAAAD///////////v/////////AwAAAAAAAABRAAAAAAAAACIAAAAAAAAABAAAAAAAAAD7/////////wQAAAAAAAAAv//////////L//////////D/////////9P////////8IAAAAAAAAAAUAAAAAAAAA//////////8HAAAAAAAAAAYAAAAAAAAAAQAAAAAAAAAIAAAAAAAAAFAAAAAAAAAA6v////////8tAAAAAAAAAPb/////////7//////////F/////////8X/////////LAAAAAAAAAD5//////////3/////////8P////////8BAAAAAAAAABkAAAAAAAAAIAAAAAAAAADz/////////83/////////TQAAAAAAAADk/////////1BLAwQtAAAAAAAAACEA6FZ4////////////BgAUAGVxLm5weQEAEADwAAAAAAAAAPAAAAAAAAAAk05VTVBZAQB2AHsnZGVzY3InOiAnPGk4JywgJ2ZvcnRyYW5fb3JkZXInOiBGYWxzZSwgJ3NoYXBlJzogKDE0LCksIH0gICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIAqAAAAAAAAAAIAAAAAAAAAAfwAAAAAAAACAAAAAAAAAAIAAAAAAAAAAfwAAAAAAAACAAAAAAAAAAH8AAAAAAAAAgAAAAAAAAACAAAAAAAAAAH8AAAAAAAAAgAAAAAAAAACAAAAAAAAAAIAAAAAAAAAAUEsDBC0AAAAAAAAAIQA18zpj//////////8FABQAay5ucHkBABAAgAcAAAAAAACABwAAAAAAAJNOVU1QWQEAdgB7J2Rlc2NyJzogJzxpOCcsICdmb3J0cmFuX29yZGVyJzogRmFsc2UsICdzaGFwZSc6ICgxNCwgMTYpLCB9ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAK3f////////9KAAAAAAAAAGAAAAAAAAAADgAAAAAAAAAWAAAAAAAAAMb/////////KQAAAAAAAAC0/////////8f/////////8f////////8HAAAAAAAAALH/////////XQAAAAAAAAAgAAAAAAAAAN3/////////+P/////////y/////////zsAAAAAAAAAVAAAAAAAAAATAAAAAAAAABEAAAAAAAAAy/////////8iAAAAAAAAALL/////////xv/////////n/////////w8AAAAAAAAAsf////////9JAAAAAAAAADAAAAAAAAAA4//////////0/////////+H/////////QQAAAAAAAABhAAAAAAAAACUAAAAAAAAAKAAAAAAAAADL/////////zoAAAAAAAAAlf/////////U/////////wIAAAAAAAAA8//////////G/////////z4AAAAAAAAAOwAAAAAAAADf/////////wYAAAAAAAAA5/////////9SAAAAAAAAAGYAAAAAAAAAFQAAAAAAAAAfAAAAAAAAAMP/////////MgAAAAAAAAC+/////////9b/////////4P/////////q/////////7D/////////PwAAAAAAAAArAAAAAAAAAL7/////////9P/////////c/////////0wAAAAAAAAAdQAAAAAAAAAyAAAAAAAAABQAAAAAAAAAzP////////8rAAAAAAAAAJn/////////wP/////////7/////////9//////////vP////////8iAAAAAAAAADIAAAAAAAAA3P////////8FAAAAAAAAAPv/////////PgAAAAAAAABsAAAAAAAAACEAAAAAAAAAMAAAAAAAAADG/////////xwAAAAAAAAAn//////////W/////////+3/////////9v////////+t/////////zsAAAAAAAAAPAAAAAAAAADV/////////xcAAAAAAAAA8/////////9NAAAAAAAAAGcAAAAAAAAAGwAAAAAAAAAcAAAAAAAAAN//////////PgAAAAAAAACx/////////9T/////////9//////////a/////////8D/////////QwAAAAAAAAAeAAAAAAAAAMv/////////CgAAAAAAAADh/////////0EAAAAAAAAAXwAAAAAAAAAmAAAAAAAAACgAAAAAAAAAzv////////82AAAAAAAAAJj/////////2P////////8CAAAAAAAAAPH/////////yP////////86AAAAAAAAADoAAAAAAAAA4P////////8HAAAAAAAAAO7/////////RwAAAAAAAABOAAAAAAAAACUAAAAAAAAAJQAAAAAAAADH/////////zoAAAAAAAAAq//////////c/////////8f//////////v////////+0/////////zcAAAAAAAAAJwAAAAAAAAC+/////////wgAAAAAAAAA5P////////9aAAAAAAAAAG4AAAAAAAAAKgAAAAAAAAAtAAAAAAAAAMP/////////TwAAAAAAAAC2/////////+3/////////vP////////8MAAAAAAAAAKv/////////KQAAAAAAAAA6AAAAAAAAAML/////////8f/////////4/////////0QAAAAAAAAAbAAAAAAAAAAeAAAAAAAAAA0AAAAAAAAA2/////////9RAAAAAAAAALD/////////yf////////8AAAAAAAAAAA0AAAAAAAAAxv////////9BAAAAAAAAAD0AAAAAAAAA4f/////////u/////////+//////////bAAAAAAAAABTAAAAAAAAACYAAAAAAAAAMgAAAAAAAADV/////////zUAAAAAAAAAu/////////+4/////////+H/////////9P////////+4/////////1AAAAAAAAAAUAAAAAAAAADD/////////wQAAAAAAAAA5P////////9AAAAAAAAAAG8AAAAAAAAAIAAAAAAAAAAkAAAAAAAAAM//////////JgAAAAAAAACY/////////8b////////////////////p/////////8H/////////JQAAAAAAAAAsAAAAAAAAANz/////////BAAAAAAAAADV/////////zoAAAAAAAAAawAAAAAAAAATAAAAAAAAADEAAAAAAAAA0/////////8lAAAAAAAAALX/////////yv/////////v//////////r/////////xv////////88AAAAAAAAADQAAAAAAAAA0v////////8NAAAAAAAAAFBLAwQtAAAAAAAAACEAcm3T+///////////BgAUAGVrLm5weQEAEADwAAAAAAAAAPAAAAAAAAAAk05VTVBZAQB2AHsnZGVzY3InOiAnPGk4JywgJ2ZvcnRyYW5fb3JkZXInOiBGYWxzZSwgJ3NoYXBlJzogKDE0LCksIH0gICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIAqAAAAAAAAAAIAAAAAAAAAAgAAAAAAAAACAAAAAAAAAAIAAAAAAAAAAgAAAAAAAAACAAAAAAAAAAIAAAAAAAAAAgAAAAAAAAACAAAAAAAAAAIAAAAAAAAAAgAAAAAAAAACAAAAAAAAAAIAAAAAAAAAAUEsDBC0AAAAAAAAAIQB0IeWj//////////8FABQAdi5ucHkBABAAgAcAAAAAAACABwAAAAAAAJNOVU1QWQEAdgB7J2Rlc2NyJzogJzxpOCcsICdmb3J0cmFuX29yZGVyJzogRmFsc2UsICdzaGFwZSc6ICgxNCwgMTYpLCB9ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAKawAAAAAAAACM/////////6v/////////8P////////9JAAAAAAAAAA8AAAAAAAAA/v/////////5/////////7f/////////t/////////8hAAAAAAAAABAAAAAAAAAASwAAAAAAAAAfAAAAAAAAAL//////////EAAAAAAAAABXAAAAAAAAAN3/////////+//////////B/////////wEAAAAAAAAA8f/////////c/////////yAAAAAAAAAAwv/////////i/////////9z/////////MgAAAAAAAACd/////////2MAAAAAAAAA1/////////9fAAAAAAAAAGMAAAAAAAAA9v////////8uAAAAAAAAAEwAAAAAAAAARwAAAAAAAAAfAAAAAAAAAL7/////////GAAAAAAAAAAWAAAAAAAAAAYAAAAAAAAAyf////////8eAAAAAAAAAPr/////////LwAAAAAAAAD5/////////1gAAAAAAAAAAwAAAAAAAADK//////////D/////////2f/////////w/////////wkAAAAAAAAA6P////////9AAAAAAAAAALX/////////MwAAAAAAAABTAAAAAAAAAAQAAAAAAAAASgAAAAAAAAAIAAAAAAAAABEAAAAAAAAA5P////////9KAAAAAAAAAAEAAAAAAAAAAQAAAAAAAACU/////////xMAAAAAAAAA0//////////l/////////zMAAAAAAAAAAQAAAAAAAADl/////////xgAAAAAAAAAEAAAAAAAAAAiAAAAAAAAAKD/////////IQAAAAAAAAAYAAAAAAAAABoAAAAAAAAAKwAAAAAAAAD0//////////b/////////JAAAAAAAAADy/////////+//////////CgAAAAAAAAAjAAAAAAAAAAMAAAAAAAAAIQAAAAAAAAAGAAAAAAAAANv/////////BwAAAAAAAAAOAAAAAAAAAEIAAAAAAAAABgAAAAAAAAAsAAAAAAAAAAAAAAAAAAAADQAAAAAAAAD4//////////L/////////FwAAAAAAAADr////////////////////7v////////8SAAAAAAAAADcAAAAAAAAARgAAAAAAAABVAAAAAAAAAOL/////////2P////////9ZAAAAAAAAAAIAAAAAAAAAOAAAAAAAAABIAAAAAAAAAEYAAAAAAAAAGQAAAAAAAAC6/////////xYAAAAAAAAAJQAAAAAAAAD7/////////7r/////////HwAAAAAAAADx/////////zMAAAAAAAAA6/////////9HAAAAAAAAACAAAAAAAAAAqP/////////P/////////3cAAAAAAAAA7v////////9mAAAAAAAAAAEAAAAAAAAAQwAAAAAAAABqAAAAAAAAAKT/////////DAAAAAAAAAAsAAAAAAAAAPv/////////8//////////q/////////ycAAAAAAAAAn//////////F/////////7T/////////zv////////9kAAAAAAAAADkAAAAAAAAA5/////////9KAAAAAAAAABEAAAAAAAAA8v////////8eAAAAAAAAAN7/////////OAAAAAAAAAABAAAAAAAAANr/////////4f////////8kAAAAAAAAAAgAAAAAAAAACwAAAAAAAADP/////////9H/////////JQAAAAAAAADg/////////1AAAAAAAAAAEQAAAAAAAAD//////////8b/////////EQAAAAAAAADq/////////+v/////////7f////////9KAAAAAAAAAMn/////////kv////////8IAAAAAAAAANP/////////KQAAAAAAAACp/////////0gAAAAAAAAAOgAAAAAAAACn/////////ywAAAAAAAAA9v////////8TAAAAAAAAAOj/////////DQAAAAAAAAC1/////////ycAAAAAAAAAKAAAAAAAAADg/////////yYAAAAAAAAAqP////////8jAAAAAAAAAAEAAAAAAAAAKwAAAAAAAAAIAAAAAAAAAAwAAAAAAAAA3P////////8bAAAAAAAAAFsAAAAAAAAAMAAAAAAAAAC3/////////+b/////////LwAAAAAAAADw/////////woAAAAAAAAALAAAAAAAAAALAAAAAAAAANL/////////IAAAAAAAAAAjAAAAAAAAAA8AAAAAAAAAtv/////////E/////////x8AAAAAAAAARAAAAAAAAAD9//////////D//////////P////////8SAAAAAAAAAFBLAwQtAAAAAAAAACEAd46J0///////////BgAUAGV2Lm5weQEAEADwAAAAAAAAAPAAAAAAAAAAk05VTVBZAQB2AHsnZGVzY3InOiAnPGk4JywgJ2ZvcnRyYW5fb3JkZXInOiBGYWxzZSwgJ3NoYXBlJzogKDE0LCksIH0gICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIAp8AAAAAAAAAHwAAAAAAAAAewAAAAAAAAB9AAAAAAAAAHsAAAAAAAAAfQAAAAAAAAB8AAAAAAAAAHsAAAAAAAAAfQAAAAAAAAB9AAAAAAAAAH0AAAAAAAAAfAAAAAAAAAB7AAAAAAAAAHwAAAAAAAAAUEsDBC0AAAAAAAAAIQA8l01B//////////8FABQAby5ucHkBABAAgAcAAAAAAACABwAAAAAAAJNOVU1QWQEAdgB7J2Rlc2NyJzogJzxpOCcsICdmb3J0cmFuX29yZGVyJzogRmFsc2UsICdzaGFwZSc6ICgxNCwgMTYpLCB9ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAKawAAAAAAAACM/////////6v/////////8P////////9JAAAAAAAAAA8AAAAAAAAA/v/////////5/////////7f/////////t/////////8hAAAAAAAAABAAAAAAAAAASwAAAAAAAAAfAAAAAAAAAL//////////EAAAAAAAAABmAAAAAAAAAKL/////////wf/////////j/////////zYAAAAAAAAABwAAAAAAAAD1/////////wMAAAAAAAAAuv/////////D/////////w4AAAAAAAAAGQAAAAAAAAAcAAAAAAAAADEAAAAAAAAAxf////////8lAAAAAAAAAFcAAAAAAAAAvf/////////b//////////H/////////LQAAAAAAAAAHAAAAAAAAAO3/////////CAAAAAAAAADP/////////9b/////////AAAAAAAAAAAZAAAAAAAAAAoAAAAAAAAALwAAAAAAAADV/////////ywAAAAAAAAARwAAAAAAAADW//////////X/////////+P////////8fAAAAAAAAAAcAAAAAAAAA5P////////8SAAAAAAAAAN3/////////7f/////////3/////////xkAAAAAAAAA+P////////8vAAAAAAAAAOb/////////MQAAAAAAAABOAAAAAAAAAND/////////6//////////Y/////////xYAAAAAAAAA+//////////n/////////xYAAAAAAAAAzv/////////h//////////r/////////HwAAAAAAAADp/////////zMAAAAAAAAA4P////////82AAAAAAAAAH8AAAAAAAAAzv/////////U/////////8L/////////QgAAAAAAAADz/////////9P/////////LQAAAAAAAADR/////////9z/////////JwAAAAAAAAAmAAAAAAAAAP3/////////JgAAAAAAAADo/////////18AAAAAAAAASQAAAAAAAADX/////////+f/////////3/////////8cAAAAAAAAAPv/////////7f////////8RAAAAAAAAANf/////////4f////////8EAAAAAAAAABwAAAAAAAAA+f////////8sAAAAAAAAAOL/////////LgAAAAAAAABEAAAAAAAAANr/////////5f/////////s/////////yQAAAAAAAAAAAAAAAAAAADr/////////xIAAAAAAAAA3v/////////q/////////xIAAAAAAAAAFwAAAAAAAAAIAAAAAAAAACcAAAAAAAAA6P////////8qAAAAAAAAAEcAAAAAAAAA1P/////////o//////////T/////////FwAAAAAAAAALAAAAAAAAAOj/////////HwAAAAAAAADu/////////97/////////BQAAAAAAAAAfAAAAAAAAAO3/////////JwAAAAAAAADn/////////zkAAAAAAAAA9f/////////P/////////9H/////////3f////////9NAAAAAAAAACIAAAAAAAAA3v////////86AAAAAAAAAAIAAAAAAAAA6v////////8JAAAAAAAAAP//////////DwAAAAAAAAAaAAAAAAAAAN7/////////FgAAAAAAAABnAAAAAAAAAPT/////////AgAAAAAAAADb/////////yEAAAAAAAAAGAAAAAAAAADA/////////1AAAAAAAAAACgAAAAAAAADu/////////+D/////////JgAAAAAAAADs/////////xQAAAAAAAAA6P////////9yAAAAAAAAAEgAAAAAAAAAxf/////////e/////////+L/////////IwAAAAAAAAABAAAAAAAAAPP/////////GAAAAAAAAADV/////////97/////////AQAAAAAAAAAXAAAAAAAAAAgAAAAAAAAAHgAAAAAAAADX/////////y8AAAAAAAAATAAAAAAAAADY/////////+//////////0P////////8FAAAAAAAAAAwAAAAAAAAA4/////////8uAAAAAAAAAN7/////////5P/////////p/////////yEAAAAAAAAA6P////////8mAAAAAAAAANr/////////RwAAAAAAAABLAAAAAAAAAOr//////////f/////////D/////////+n/////////FgAAAAAAAADc/////////0UAAAAAAAAA6//////////p/////////8//////////JwAAAAAAAADW/////////x4AAAAAAAAA2f////////9cAAAAAAAAAFBLAwQtAAAAAAAAACEAhqpJhP//////////BgAUAGVvLm5weQEAEADwAAAAAAAAAPAAAAAAAAAAk05VTVBZAQB2AHsnZGVzY3InOiAnPGk4JywgJ2ZvcnRyYW5fb3JkZXInOiBGYWxzZSwgJ3NoYXBlJzogKDE0LCksIH0gICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIAp8AAAAAAAAAHwAAAAAAAAAfAAAAAAAAAB8AAAAAAAAAHwAAAAAAAAAewAAAAAAAAB8AAAAAAAAAHwAAAAAAAAAfAAAAAAAAAB8AAAAAAAAAHsAAAAAAAAAfAAAAAAAAAB8AAAAAAAAAHwAAAAAAAAAUEsDBC0AAAAAAAAAIQBuUNOr//////////8KABQAdG9rZW5zLm5weQEAEADwAAAAAAAAAPAAAAAAAAAAk05VTVBZAQB2AHsnZGVzY3InOiAnPGk4JywgJ2ZvcnRyYW5fb3JkZXInOiBGYWxzZSwgJ3NoYXBlJzogKDE0LCksIH0gICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIAoeHQAAAAAAAGIJAAAAAAAAAQEAAAAAAACAAgAAAAAAAAsAAAAAAAAAZAIAAAAAAAB1AQAAAAAAAAEBAAAAAAAAHgUAAAAAAAAQCgAAAAAAAHoOAAAAAAAARU4AAAAAAAANAAAAAAAAAF8FAAAAAAAAUEsBAi0DLQAAAAAAAAAhACBuJouABwAAgAcAAAUAAAAAAAAAAAAAAIABAAAAAHEubnB5UEsBAi0DLQAAAAAAAAAhAOhWeP/wAAAA8AAAAAYAAAAAAAAAAAAAAIABtwcAAGVxLm5weVBLAQItAy0AAAAAAAAAIQA18zpjgAcAAIAHAAAFAAAAAAAAAAAAAACAAd8IAABrLm5weVBLAQItAy0AAAAAAAAAIQBybdP78AAAAPAAAAAGAAAAAAAAAAAAAACAAZYQAABlay5ucHlQSwECLQMtAAAAAAAAACEAdCHlo4AHAACABwAABQAAAAAAAAAAAAAAgAG+EQAAdi5ucHlQSwECLQMtAAAAAAAAACEAd46J0/AAAADwAAAABgAAAAAAAAAAAAAAgAF1GQAAZXYubnB5UEsBAi0DLQAAAAAAAAAhADyXTUGABwAAgAcAAAUAAAAAAAAAAAAAAIABnRoAAG8ubnB5UEsBAi0DLQAAAAAAAAAhAIaqSYTwAAAA8AAAAAYAAAAAAAAAAAAAAIABVCIAAGVvLm5weVBLAQItAy0AAAAAAAAAIQBuUNOr8AAAAPAAAAAKAAAAAAAAAAAAAACAAXwjAAB0b2tlbnMubnB5UEsFBgAAAAAJAAkA1AEAAKgkAAAAAA=='))

9874

## 6. Running the tests

Block-level tests (about 10 s):

In [23]:
!python3 tb/run_tests.py blocks 2>&1 | grep -E 'PASS|FAIL|TESTS='

                                                        ** test_blocks.test_exp2_sweep              PASS       65238.00           1.18      55071.13  **
                                                        ** TESTS=6 PASS=1 FAIL=0 SKIP=5                        65238.00           1.19      55034.97  **


                                                        ** test_blocks.test_requant_sweep           PASS        4000.00           0.07      54023.59  **
                                                        ** TESTS=6 PASS=1 FAIL=0 SKIP=5                         4000.00           0.07      53486.84  **


                                                        ** test_blocks.test_dpu_random              PASS        4081.00           0.02     229183.59  **
                                                        ** TESTS=6 PASS=1 FAIL=0 SKIP=5                         4081.00           0.02     222358.22  **


                                                        ** test_blocks.test_out_random              PASS        3101.00           0.03      99122.96  **
                                                        ** TESTS=6 PASS=1 FAIL=0 SKIP=5                         3101.00           0.03      96922.72  **


                                                        ** test_blocks.test_lane_online_and_recip   PASS       27221.00           0.05     565058.97  **
                                                        ** TESTS=6 PASS=1 FAIL=0 SKIP=5                        27221.00           0.05     555382.06  **


                                                        ** test_blocks.test_rope_rotator            PASS       41931.00           0.27     157372.03  **
                                                        ** TESTS=6 PASS=1 FAIL=0 SKIP=5                        41931.00           0.27     157061.46  **


Core tests at P = 4 lanes, N = d = 16 (about 2.5 minutes):

In [24]:
!python3 tb/run_tests.py core 2>&1 | grep -E 'PASS|FAIL|TESTS='

                                                        ** test_core.test_ids                  PASS         400.00           0.01      60058.63  **
                                                        ** test_core.test_all_modes            PASS      547660.00           7.74      70784.16  **
                                                        ** test_core.test_offset_and_reuse     PASS      127220.00           1.56      81390.16  **
                                                        ** test_core.test_random_regression    PASS      430420.00           5.09      84487.38  **
                                                        ** test_core.test_decode_kv_cache      PASS      190660.00           2.40      79538.52  **
                                                        ** test_core.test_causal_prefill       PASS      562720.00           6.39      88092.96  **
                                                        ** test_core.test_multi_query_decode   PASS      396460.

Chip-level tests through the SPI and QSPI pins:

In [25]:
!python3 tb/run_tests.py chip 2>&1 | grep -E 'PASS|FAIL|TESTS='

                                                        ** test_chip.test_spi_registers       PASS      159760.00           0.36     443021.59  **
                                                        ** test_chip.test_spi_passes          PASS     4440760.00           8.34     532548.14  **
                                                        ** test_chip.test_qspi_passes         PASS     1769100.00           6.07     291684.10  **
                                                        ** test_chip.test_qspi_token_window   PASS     2262500.00           5.29     427785.52  **
                                                        ** TESTS=4 PASS=4 FAIL=0 SKIP=0                8632120.00          20.05     430427.96  **


## 7. Selected results written by the tests

Full passes: cycles, skipped keys and cosine similarity to FP32 attention for unfused (`fused=0`) and fused (`fused=1`) dataflows in the three exp modes:

In [26]:
print(open('results/core_N16_D16.txt').read())

random fused=0 mode=0              cycles= 411 skips=  0 cos=0.99979
random fused=0 mode=1              cycles= 411 skips=  0 cos=0.98123
random fused=0 mode=2              cycles= 411 skips=  0 cos=0.99756
random fused=1 mode=0              cycles= 111 skips=  0 cos=0.99983
random fused=1 mode=1              cycles= 111 skips=  0 cos=0.98123
random fused=1 mode=2              cycles= 111 skips=  0 cos=0.99756
sharp fused=0 mode=0               cycles= 411 skips=182 cos=0.99975
sharp fused=0 mode=1               cycles= 411 skips=179 cos=0.99516
sharp fused=0 mode=2               cycles= 411 skips=181 cos=0.99959
sharp fused=1 mode=0               cycles= 111 skips=138 cos=0.99976
sharp fused=1 mode=1               cycles= 111 skips=134 cos=0.99518
sharp fused=1 mode=2               cycles= 111 skips=137 cos=0.99959
outliers fused=0 mode=0            cycles= 411 skips=118 cos=0.99987
outliers fused=0 mode=1            cycles= 411 skips=112 cos=0.99965
outliers fused=0 mode=2           

A real language-model head (TinyStories-8M, layer 0, head 0) through the RTL: every output row matches the golden model.

In [27]:
print(open('results/lm_head_rtl.txt').read())

layer 0 head 0, 14 prompt tokens, causal prefill pass: 97 cycles, 2 skips
mean row cosine vs the benchmark's decode-step outputs: 0.999993
row  0 token  7454 e_o=124 o= 107 -116  -85  -16   73   15   -2   -7  -73  -73   33   16   75   31  -65   16 match
row  1 token  2402 e_o=124 o= 102  -94  -63  -29   54    7  -11    3  -70  -61   14   25   28   49  -59   37 match
row  2 token   257 e_o=124 o=  87  -67  -37  -15   45    7  -19    8  -49  -42    0   25    9   47  -43   44 match
row  3 token   640 e_o=124 o=  71  -42  -11   -8   31    7  -28   18  -35  -19   -9   25   -8   47  -26   49 match
row  4 token    11 e_o=124 o=  78  -48  -21  -40   22   -5  -25   22  -50  -31   -6   31  -23   51  -32   54 match
row  5 token   612 e_o=123 o= 127  -50  -44  -62   66  -13  -46   45  -47  -36   39   38   -3   38  -24   95 match
row  6 token   373 e_o=124 o=  73  -41  -25  -33   28   -5  -20   17  -41  -31    4   28   -7   44  -30   46 match
row  7 token   257 e_o=124 o=  68  -38  -27  -20   36   

Max-first decode and the heavy-hitter tracker:

In [28]:
print(open('results/max_first_N16_D16.txt').read()[:1500])
print(open('results/heavy_hitter_N16_D16.txt').read())

random   mode=0 thr=0: V reads 136 -> 136 of 136 (0% fewer), cycles at LEN=16 50 -> 71, total cycles 680 -> 896, mean cos 0.99987 -> 0.99986
random   mode=0 thr=6: V reads 130 -> 127 of 136 (2% fewer), cycles at LEN=16 50 -> 68, total cycles 680 -> 887, mean cos 0.99986 -> 0.99983
random   mode=2 thr=0: V reads 136 -> 136 of 136 (0% fewer), cycles at LEN=16 50 -> 71, total cycles 680 -> 896, mean cos 0.99902 -> 0.99901
random   mode=2 thr=6: V reads 130 -> 127 of 136 (2% fewer), cycles at LEN=16 50 -> 68, total cycles 680 -> 887, mean cos 0.99897 -> 0.99896
sharp    mode=0 thr=0: V reads  69 ->  41 of 136 (41% fewer), cycles at LEN=16 50 -> 64, total cycles 680 -> 809, mean cos 0.99988 -> 0.99988
sharp    mode=0 thr=6: V reads  59 ->  31 of 136 (47% fewer), cycles at LEN=16 50 -> 59, total cycles 680 -> 791, mean cos 0.99985 -> 0.99981
sharp    mode=2 thr=0: V reads  70 ->  42 of 136 (40% fewer), cycles at LEN=16 50 -> 64, total cycles 680 -> 809, mean cos 0.99976 -> 0.99975
sharp    m